# MultiBridge Fault-Aware Selective SHM — V1.2

**ICAM3E — un seul notebook pour Jupyter web, sans Bash.**

Version issue de la V1.1 réellement exécutée : lecteurs, panneaux physiques, étiquettes, exclusions et plis externes conservés. La variante livrée est **`quality_crossfit`** : le modèle structurel Fault-Aware est entraîné avec des scores qualité hors groupe. Les modèles V1.1 compatibles servent de contrôles en lecture seule ; sinon leurs règles sont réexécutées et ce réapprentissage est déclaré.

**Nouveautés :** décision instrumentale nullable (`CHECK`, `NO_CHECK`, `UNDETERMINED`), détection de l’absence de seuil utile, comparaison Route345 `max` / quantile temporel 0,90 fixé, traçabilité des scores extrêmes et analyse nominal/corrompu exclusivement diagnostique.

Les données test de V1.0/V1.1 ont été consultées : il s’agit d’une **analyse révisée**, pas d’une nouvelle confirmation indépendante. Aucun gain de performance n’est garanti. Les défauts instrumentaux sont simulés ; les données brutes ne sont jamais modifiées.


## 00 — Configuration Jupyter web
Vérifiez uniquement les chemins et le nom d’étude ci-dessous, puis **Restart Kernel and Run All Cells**. Réglages livrés : `full`, `quality_crossfit`, `auto` (GPU CUDA accessible et testé, sinon CPU), reprise activée. Les anciens dossiers `runs_v10` et `runs_v11` ne sont jamais écrasés.

Les fenêtres sont reprises du protocole V1.1 vérifié ; en l’absence de ce fichier, **20 secondes** est la durée fixe préspécifiée, sans nouvelle recherche. Activez `require_reference_protocol=True` pour bloquer plutôt que recourir à cette valeur.


In [1]:
from __future__ import annotations
# ===== Seule cellule de configuration à vérifier dans Jupyter =====
import os, sys, re, csv, io, json, math, time, gc, hashlib, random, platform, warnings, unicodedata, copy, tempfile, traceback, inspect, contextlib
from pathlib import Path
from dataclasses import dataclass, field, asdict, replace
from datetime import datetime, timezone
from collections import Counter, defaultdict
from fractions import Fraction

VERSION = '1.2.0'
SOURCE_RESEARCH_DATE = '2026-09-11'
REFERENCE_VERSION = '1.1.0'
REFERENCE_IMPLEMENTATION_HASH = 'f18fc5151aad1bfa32892b302252151061503cefe3f91e6894585e765310ba03'
IMPLEMENTATION_HASH = 'd1bdf0edaca80030db4e397f4be56e2bbd02fba1cb538ebd417b785897ed8505'

@dataclass
class Config:
    # Pas de variable d'environnement pour le mode : full est le réglage livré.
    mode: str = 'full'                     # full / audit / smoke / selftest
    study_name: str = 'V1_2_MAIN'           # garder pour reprendre ; changer pour un nouveau protocole
    resume: bool = True
    # V1.2 : qualité hors groupe ; comparateurs V1.1 réutilisés si vérifiés.
    # Le contrôle FaultAware_Control conserve les scores qualité in-sample de V1.1.
    variant: str = 'quality_crossfit'    # decision_corrected / quality_crossfit
    reference_policy: str = 'prefer_compatible' # prefer_compatible / require_compatible / never
    reference_study_name: str = 'V1_1_MAIN'
    reference_run_root: str = ''              # vide -> project_root/runs_v11/reference_study_name
    reference_cache_root: str = ''            # vide -> project_root/cache_v11
    trust_own_local_reference_models: bool = True # jamais charger des poids inconnus téléchargés
    reuse_verified_native_cache: bool = True
    frozen_window_seconds: float = 20.0    # no new search; fallback documented when parent metadata is absent
    require_reference_protocol: bool = False
    run_route345_aggregation_study: bool = True
    route345_temporal_quantile: float = 0.90 # fixed inverse empirical CDF, NOT test-selected
    quality_diagnostic_top_windows: int = 3
    quality_alarm_budget: float = 0.05     # cible EMPIRIQUE par acquisition nominale de calibration
    min_quality_cal_groups: int = 20       # avertissement uniquement, pas une garantie
    min_calibration_groups_per_class: int = 5
    crossfit_quality_folds: int = 3        # borné par les acquisitions saines d'entraînement
    reference_quantile: float = 0.95
    test_was_previously_inspected: bool = True
    project_root: str = '/home/a.riyahi/ICARC2027_MultiBridge_FaultAware_SHM'
    z24_root: str = '/home/a.riyahi/EvoBridge_SHM/data/raw/z24'
    route345_file: str = '/home/a.riyahi/EvoBridge_SHM/data/raw/route345/Rt345Bridge.h5'
    openlab_2025_root: str = '/home/a.riyahi/EvoBridge_SHM/data/raw/openlab/openlab_damage'
    openlab_2024_root: str = '/home/a.riyahi/EvoBridge_SHM/data/raw/openlab/openlab_reference'
    conference: str = 'ICAM3E'
    conference_url: str = 'https://icam3e.sciencesconf.org/index'
    device_preference: str = 'auto'         # auto = CUDA fonctionnel sinon CPU ; ou cpu
    cpu_threads: int = 4                   # bornés à l'affinité / allocation détectée
    allow_local_compute: bool = True       # Jupyter web : pas de SLURM_JOB_ID obligatoire
    progress_every: int = 5
    show_figures: bool = True
    seed: int = 2027
    required_tasks: tuple = ('Z24','Route345','openLAB2025')
    run_reference_2024: bool = True
    enable_quality_transfer: bool = True
    enable_minirocket: bool = False         # facultatif ; coûts mémoire nettement supérieurs
    seed_diagnostics: bool = False          # l'ensemble 3 graines est toujours évalué
    full_file_hashes: bool = True
    route345_v2_md5: str = '5c3e2aff20e1984fce15bd18f57f62eb'
    overrides_json: str = ''
    z24_reference_channels: tuple = ('R1V','R2L','R2T','R2V','R3V')
    z24_exclude_installation_state2: bool = True
    route345_axes: tuple = ('y','z')
    route345_excluded_sensors: tuple = ('A14',)
    openlab_excitation_primary: tuple = ('random',)
    openlab_vehicle_primary: tuple = ('achse_a40',)
    openlab_allow_postload_as_change: bool = False
    min_physical_sensors: int = 2
    train_channel_coverage: float = 0.90
    window_seconds: tuple = (10.0,20.0,40.0)
    reference_window_seconds: float = 20.0
    train_overlap: float = 0.5
    eval_overlap: float = 0.0
    band_hz: tuple = (0.5,25.0)
    psd_bins: int = 64
    welch_segment_seconds: float = 4.0
    max_missing_fraction: float = 0.02
    max_interpolation_gap_seconds: float = 0.05
    timestamp_relative_tolerance: float = 0.01
    outer_folds: int = 3
    split_fractions: tuple = (0.55,0.15,0.15,0.15)  # compatibilité des tests V0.3 ; splitter externe V1 ci-dessous
    fault_types: tuple = ('missing','freeze','gain','clipping','noise','drift')
    heldout_fault: str = 'drift'
    fault_severities: tuple = (0.25,0.60,1.0)
    train_fault_repeats: int = 2
    eval_fault_repeats: int = 1
    max_faulty_physical_sensors: int = 1
    seeds: tuple = (2027,2039,2053)
    max_epochs: int = 100
    patience: int = 12
    batch_size: int = 128
    lr: float = 1e-3
    weight_decay: float = 1e-4
    latent_dim: int = 32
    hidden_dim: int = 64
    dropout: float = 0.10
    tree_estimators: int = 400
    quality_tree_estimators: int = 300
    target_coverages: tuple = (0.60,0.80,0.90,1.0)
    primary_coverage: float = 0.80
    min_calibration_groups_for_affine: int = 12
    bootstrap_replicates: int = 1000
    smoke_windows_per_acquisition: int = 3
    max_feature_store_gb: float = 8.0
    transfer_rows_per_acquisition: int = 2000

CFG=Config()
if CFG.mode not in {'full','audit','smoke','selftest'}: raise ValueError('Mode non reconnu')
if CFG.device_preference not in {'auto','cpu'}: raise ValueError('device_preference doit être auto ou cpu')
assert CFG.variant in {'decision_corrected','quality_crossfit'}
assert CFG.reference_policy in {'prefer_compatible','require_compatible','never'}
assert 0 <= CFG.quality_alarm_budget < 1 and 0 < CFG.reference_quantile < 1
assert CFG.crossfit_quality_folds>=2
assert CFG.frozen_window_seconds>0 and CFG.route345_temporal_quantile==0.90
assert CFG.quality_diagnostic_top_windows>=1
assert CFG.eval_fault_repeats==1, 'Une répétition externe par type/sévérité dans le protocole comparatif livré.'
assert CFG.outer_folds==3, 'Le protocole livré et testé utilise trois plis externes.'
assert CFG.heldout_fault in CFG.fault_types and CFG.primary_coverage in CFG.target_coverages
assert CFG.max_faulty_physical_sensors==1 and 0<=CFG.train_overlap<1 and CFG.eval_overlap==0
assert re.fullmatch(r'[A-Za-z0-9_.-]+',CFG.study_name), 'study_name ne doit pas contenir de chemin.'
try: affinity=len(os.sched_getaffinity(0))
except (AttributeError,OSError): affinity=os.cpu_count() or 1
try: allocated=int(os.environ.get('SLURM_CPUS_PER_TASK',str(affinity)))
except ValueError: allocated=affinity
CFG.cpu_threads=max(1,min(int(CFG.cpu_threads),affinity,allocated))
# Définir les limites avant l'import de NumPy et sklearn, sans appel threadpoolctl fragile.
for variable in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[variable]=str(CFG.cpu_threads)
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')

import numpy as np
import pandas as pd
import scipy
from scipy import signal, stats, optimize
from scipy.io import loadmat, whosmat
from scipy.special import expit, logit
import h5py
import sklearn
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, balanced_accuracy_score, log_loss, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import joblib
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display, Markdown, FileLink, HTML
import matplotlib.pyplot as plt

ROOT=Path(CFG.project_root).expanduser()
RUN_ID=CFG.study_name if CFG.mode=='full' else f'{CFG.study_name}_{CFG.mode}'
RUN=ROOT/'runs_v12'/RUN_ID
BASE_RUN=RUN
CACHE=ROOT/'cache_v12'
REFERENCE_RUN=Path(CFG.reference_run_root).expanduser() if CFG.reference_run_root else ROOT/'runs_v11'/CFG.reference_study_name
REFERENCE_CACHE=Path(CFG.reference_cache_root).expanduser() if CFG.reference_cache_root else ROOT/'cache_v11'
for sub in ('audit','models','predictions','tables','figures','logs','features','checkpoints','folds'):
    (RUN/sub).mkdir(parents=True,exist_ok=True)
CACHE.mkdir(parents=True,exist_ok=True)
OVERRIDES=json.loads(Path(CFG.overrides_json).read_text()) if CFG.overrides_json else {}
print(f'V{VERSION} | mode={CFG.mode} | étude={RUN_ID}',flush=True)
print('Dossier de résultats :',RUN)
print('Python',sys.version.split()[0], '| NumPy',np.__version__, '| SciPy',scipy.__version__, '| PyTorch',torch.__version__)
print('Ressources CPU :',CFG.cpu_threads, 'threads ; aucun shell requis.')
print('Variante :',CFG.variant,'| politique V1.1 :',CFG.reference_policy,'| dossier source en lecture seule :',REFERENCE_RUN)
if CFG.variant=='quality_crossfit':print('FaultAware est réentraîné avec qualité hors groupe ; les comparateurs V1.1 compatibles restent figés.')

V1.2.0 | mode=full | étude=V1_2_MAIN
Dossier de résultats : /home/a.riyahi/ICARC2027_MultiBridge_FaultAware_SHM/runs_v12/V1_2_MAIN
Python 3.11.7 | NumPy 1.23.5 | SciPy 1.11.4 | PyTorch 2.7.1+cu118
Ressources CPU : 4 threads ; aucun shell requis.
Variante : quality_crossfit | politique V1.1 : prefer_compatible | dossier source en lecture seule : /home/a.riyahi/ICARC2027_MultiBridge_FaultAware_SHM/runs_v11/V1_1_MAIN
FaultAware est réentraîné avec qualité hors groupe ; les comparateurs V1.1 compatibles restent figés.


## 01 — Ressources, journal, reprise et isolation des versions

GPU testé réellement dans le noyau courant ; sinon CPU. Aucun GPU distant n'est réservé et aucune soumission Slurm n'est créée. Le noyau web doit être autorisé pour cette charge. Les threads sont bornés ; ne pas modifier les paquets du cluster automatiquement.

V1.1 écrit dans `runs_v11/` et `cache_v11/`. Les caches natifs V1.0 peuvent être lus après vérification de la source et de l'empreinte de l'onde ; ils ne sont jamais modifiés.

In [2]:
def json_safe(x):
    if isinstance(x,dict): return {str(k):json_safe(v) for k,v in x.items()}
    if isinstance(x,(list,tuple,set)): return [json_safe(v) for v in x]
    if isinstance(x,np.ndarray): return json_safe(x.tolist())
    if isinstance(x,np.integer): return int(x)
    if isinstance(x,np.bool_): return bool(x)
    if isinstance(x,(float,np.floating)): return float(x) if np.isfinite(x) else None
    if isinstance(x,(Path,datetime)): return str(x)
    if isinstance(x,bytes): return x.decode('utf-8','replace')
    return x

def write_json(obj,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+f'.{os.getpid()}.tmp')
    tmp.write_text(json.dumps(json_safe(obj),indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
    tmp.replace(path)

def stable_hash(obj):
    return hashlib.sha256(json.dumps(json_safe(obj),sort_keys=True,ensure_ascii=False,allow_nan=False).encode()).hexdigest()

def file_hash(path,algo='sha256'):
    h=hashlib.new(algo)
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8*1024**2),b''):h.update(b)
    return h.hexdigest()

def table(rows,name):
    df=rows if isinstance(rows,pd.DataFrame) else pd.DataFrame(rows)
    p=RUN/name;p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+f'.{os.getpid()}.tmp');df.to_csv(tmp,index=False);tmp.replace(p)
    return df

GATES=[];DIAGNOSTICS=[];EXPERIMENT_STATUS={};METRICS=[];PAIRWISE=[]
DEVICE_EVENTS=[];FOLD_SUMMARIES={};FOLD_PAYLOADS={};FOLD_METRICS=[];FOLD_PAIRWISE=[]
ALL_FOLD_DECISIONS={};RUNTIME_STARTED=time.perf_counter()

def note(kind,source,message,**details):
    row={'kind':kind,'source':str(source),'message':str(message),'utc':datetime.now(timezone.utc).isoformat(),**json_safe(details)}
    DIAGNOSTICS.append(row)
    with (BASE_RUN/'logs/events.jsonl').open('a',encoding='utf-8') as f:
        f.write(json.dumps(json_safe(row),ensure_ascii=False,allow_nan=False)+'\n')

def gate(name,condition,details,mandatory=True):
    GATES.append({'gate':name,'passed':bool(condition),'mandatory':mandatory,'details':json_safe(details)})
    write_json(GATES,BASE_RUN/'audit/gates.json')
    print(f"[{'PASS' if condition else 'BLOCKED'}] {name}: {str(details)[:220]}",flush=True)
    return bool(condition)

def flush_reports():
    table(DIAGNOSTICS,'audit/DIAGNOSTICS.csv');write_json(GATES,BASE_RUN/'audit/gates.json')
    write_json(EXPERIMENT_STATUS,BASE_RUN/'audit/EXPERIMENT_STATUS.json')
    write_json(DEVICE_EVENTS,BASE_RUN/'audit/DEVICE_EVENTS.json')

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if str(globals().get('DEVICE','cpu')).startswith('cuda'):
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True,warn_only=True)
    if hasattr(torch.backends,'cudnn'):torch.backends.cudnn.benchmark=False

def gpu_error(e):
    # Numerical/data errors are not hidden by switching hardware.
    text=str(e).lower()
    return isinstance(e,RuntimeError) and any(s in text for s in ('cuda','cublas','cudnn','out of memory','device-side'))

def use_cpu(reason):
    global DEVICE
    old=str(globals().get('DEVICE','cpu'));DEVICE=torch.device('cpu')
    DEVICE_EVENTS.append({'event':'CPU_FALLBACK','from':old,'reason':str(reason),'utc':datetime.now(timezone.utc).isoformat()})
    try:torch.cuda.empty_cache()
    except Exception:pass
    gc.collect();write_json(DEVICE_EVENTS,BASE_RUN/'audit/DEVICE_EVENTS.json')
    print('Calcul sur CPU :',reason,flush=True)

def choose_device():
    if CFG.device_preference=='cpu':return torch.device('cpu')
    try:
        if not torch.cuda.is_available():return torch.device('cpu')
        visible=[]
        for i in range(torch.cuda.device_count()):
            try:
                free,total=torch.cuda.mem_get_info(i)
                visible.append((int(free),i,int(total)))
            except Exception:visible.append((0,i,0))
        for free,i,total in sorted(visible,reverse=True):
            try:
                device=torch.device(f'cuda:{i}')
                torch.cuda.set_device(i)
                x=torch.ones((64,64),device=device);result=(x@x).sum();torch.cuda.synchronize(i)
                if not torch.isfinite(result):raise RuntimeError('CUDA test nonfinite')
                del x,result
                DEVICE_EVENTS.append({'event':'CUDA_TEST_PASS','device':str(device),'name':torch.cuda.get_device_name(i),'free_bytes_at_probe':free,'total_bytes':total})
                return device
            except Exception as e:DEVICE_EVENTS.append({'event':'CUDA_TEST_FAIL','index':i,'error':repr(e)})
    except Exception as e:DEVICE_EVENTS.append({'event':'CUDA_DETECTION_FAIL','error':repr(e)})
    return torch.device('cpu')

DEVICE=choose_device();torch.set_default_dtype(torch.float32);torch.set_num_threads(CFG.cpu_threads)
try:torch.set_num_interop_threads(max(1,min(2,CFG.cpu_threads)))
except RuntimeError:pass
seed_all(CFG.seed)
print('Périphérique sélectionné :',DEVICE, '| CUDA visible :',torch.cuda.device_count() if str(DEVICE).startswith('cuda') else 'non utilisé',flush=True)
if re.search(r'(^|[.-])(login|frontend|head)',platform.node().lower()):
    print('AVERTISSEMENT : hôte nommé comme un serveur de connexion. Vérifier que ce noyau Jupyter est autorisé pour le calcul ; aucun contournement ni réservation n’est effectué.',flush=True)
    note('host_policy_notice',platform.node(),'Web kernel without scheduler enforcement; CPU threads bounded, administrator policy still applies.')

def require_compute():
    # The user explicitly runs within an already-open web kernel. No shell/job submission required.
    print('Exécution dans le noyau Jupyter courant :',platform.node(),str(DEVICE),flush=True)

ENV={'version':VERSION,'config':asdict(CFG),'host':platform.node(),'python':sys.version,
 'packages':{'numpy':np.__version__,'pandas':pd.__version__,'scipy':scipy.__version__,'sklearn':sklearn.__version__,'torch':torch.__version__,'h5py':h5py.__version__},
 'device_initial':str(DEVICE),'slurm_job':os.environ.get('SLURM_JOB_ID'),'interface':'Jupyter web','utc':datetime.now(timezone.utc).isoformat()}


# Existing environment is preserved. We deliberately do not call the old threadpoolctl scanner.
try:
    from importlib.metadata import version as package_version
    ENV['packages']['threadpoolctl']=package_version('threadpoolctl')
except Exception:pass


def array_fingerprint(a):
    a=np.ascontiguousarray(a)
    return hashlib.sha256(memoryview(a).cast('B')).hexdigest()

NONSCIENTIFIC_FIELDS={'project_root','study_name','resume','cpu_threads','device_preference','allow_local_compute','progress_every','show_figures'}
def scientific_signature():
    return stable_hash({'implementation':IMPLEMENTATION_HASH,'config':{k:v for k,v in asdict(CFG).items() if k not in NONSCIENTIFIC_FIELDS}})

SIGNATURE=scientific_signature()
lock=BASE_RUN/'STUDY_LOCK.json'
if lock.exists():
    prior=json.loads(lock.read_text())
    if prior['signature']!=SIGNATURE:
        raise ValueError('Configuration scientifique/code différents dans ce dossier. Donner un nouveau study_name, sans écraser les résultats antérieurs.')
    if not CFG.resume:raise FileExistsError('Étude existante ; activer resume ou choisir un autre study_name.')
else:write_json({'signature':SIGNATURE,'implementation':IMPLEMENTATION_HASH,'config':asdict(CFG)},lock)
# Metadata is written only after a compatible study lock is established.
write_json(ENV,BASE_RUN/'environment.json')
write_json(DEVICE_EVENTS,BASE_RUN/'audit/DEVICE_EVENTS.json')
write_json(ENV,BASE_RUN/'sessions'/('environment_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%f')+'_'+str(os.getpid())+'.json'))

@contextlib.contextmanager
def output_context(folder):
    global RUN
    previous=RUN;RUN=Path(folder)
    for sub in ('audit','models','predictions','tables','figures','logs','features','checkpoints'):(RUN/sub).mkdir(parents=True,exist_ok=True)
    try:yield
    finally:RUN=previous

def atomic_joblib(obj,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');joblib.dump(obj,tmp,compress=3);tmp.replace(path)

def atomic_torch(obj,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');torch.save(obj,tmp);tmp.replace(path)


PREDICTOR_ORIGINS={}
QUALITY_DIAGNOSTICS={}

Périphérique sélectionné : cpu | CUDA visible : non utilisé
AVERTISSEMENT : hôte nommé comme un serveur de connexion. Vérifier que ce noyau Jupyter est autorisé pour le calcul ; aucun contournement ni réservation n’est effectué.


## 02 — Source registry and exact endpoint definitions

**Evidence hierarchy:** native metadata / campaign log, official dataset documentation, then explicitly cited prior MARWAN audit. Filename numbers alone never override group names. A supported pattern is not cryptographic release verification.

Z24 state 02 (installation of settlement equipment) is excluded by default; states 01 and 08 are reference regimes, not identical physical baselines. openLAB post-load and exposed-tendon stages are retained in the audit, but not silently labeled permanent damage. Default openLAB primary scope is random excitation at Achse A40, reference versus documented tendon cuts; all random fields are retained because a single field has too few independent reference runs for four partitions. Sweep, fixed-frequency and moving-vehicle trials are not mixed into that primary experiment.

In [3]:
SOURCES={
 'Z24':{'url':'https://bwk.kuleuven.be/bwm/z24','evidence':'KU Leuven PDT description and chronological table; five reference channels, native 100 Hz. No redistribution of raw Z24.'},
 'Route345':{'url':'https://zenodo.org/records/15497673','evidence':'V2 checksum; acceleration/strain separate. Native sample_rate=128 versus internal_sample_rate=512 documented in prior MARWAN audit, 26 Aug 2026.'},
 'Route345_panel_QC':{'source':'AGENTS.md, 26 Aug 2026; ProTwin_B_Three_Bridge_Reference_Calibrated_Experiment_V0_2_1.pdf, 30 Jul 2026', 'evidence_status':'REPORTED_REFERENCE_PROJECT_QC', 'evidence':'A14/z near-constant at damage_6/test_1..test_4 in the prior executed report. V1.0 removes the complete A14 sensor symmetrically from every acquisition before splitting. This does not claim a new real-data QC execution here; y-axis removal is a conservative physical-sensor policy.'},
 'openLAB2025':{'url':'https://zenodo.org/records/18546379','log':'2025_05_05_Übersicht_Messungen.txt','evidence':'BAM campaign log; WZ_V47/WZ_V86 velocities; SH1/SH2 shaker acceleration; laser displacement.'},
 'openLAB2024':{'url':'https://opara.zih.tu-dresden.de/items/6653124a-8659-40b8-817e-51250639c95b','evidence':'Separate reference monitoring; six G_ACCZ channels, documented 500 Hz; event-triggered records, not continuous exposure.'},
 'Welch':{'url':'https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.welch.html','evidence':'Explicit Hann, density scaling, nfft=nperseg. Bin interpolation does not improve physical resolution.'},
 'calibration':{'url':'https://proceedings.mlr.press/v70/guo17a.html','evidence':'Post-hoc calibration background. V1.0 uses monotone affine-logit calibration on separate groups; no distribution-free guarantee.'},
 'selective':{'url':'https://proceedings.mlr.press/v97/geifman19a.html','evidence':'Risk/coverage terminology. This implementation is not claimed to reproduce SelectiveNet.'},
}
SOURCES['CUDA']={'url':'https://docs.pytorch.org/docs/stable/generated/torch.cuda.is_available.html',
                   'evidence':'Availability is local to this Python kernel; no Slurm allocation is requested.'}
SOURCES['grouped_evaluation']={'url':'https://scikit-learn.org/stable/modules/cross_validation.html',
 'evidence':'Group-disjoint development/evaluation principle. Our three-way outer rotation is specified below, not a claimed dataset requirement.'}
write_json(SOURCES,RUN/'audit/SOURCES.json')
Z24_STATES={1:'initial_reference',2:'equipment_installation',3:'pier_lowering_20mm',4:'pier_lowering_40mm',5:'pier_lowering_80mm',6:'pier_lowering_95mm',7:'pier_lifting_foundation_tilt',8:'new_reference',9:'concrete_spalling_12m2',10:'concrete_spalling_24m2',11:'landslide_abutment',12:'concrete_hinge_failure',13:'two_anchor_heads',14:'four_anchor_heads',15:'two_tendons',16:'four_tendons',17:'six_tendons'}

def norm(s):
    s=unicodedata.normalize('NFKD',str(s)).replace('²','2')
    s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'_+','_',s.strip().lower())

def canonical_channel(s):
    return re.sub(r'_+', '_', re.sub(r'\s+','',re.sub(r'\[.*?\]','',str(s))).strip().upper())

def classify_rt_state(top):
    s=norm(top)
    if re.fullmatch(r'baseline[_ -]?[12]',s): return 0,s,'native_named_reference'
    if re.fullmatch(r'damage[_ -]?[1-6]',s): return 1,s,'native_named_damage'
    return None,s,'unresolved'

def classify_ol_state(token):
    s=norm(token)
    if 'ungeschadigt' in s: return 0,'reference','documented_reference'
    if 'getrennt' in s: return 1,s,'documented_cut'
    if re.search(r'laststufe_0?[1-7]_entlastet',s):
        return (1 if CFG.openlab_allow_postload_as_change else None),s,'postload_change_not_damage_proof'
    if 'freigelegt' in s: return None,s,'exposure_not_cut'
    return None,s,'unresolved'

## 03 — Records and metadata helpers

One record denotes a physical acquisition, not a sensor or frequency bin. A waveform fingerprint prevents exact duplicate recordings from landing in different partitions. Long records are stored once as memory-mapped arrays; window dictionaries never retain copies of raw waveforms.

In [4]:
@dataclass
class Record:
    task: str
    path: str
    native_key: str
    acquisition_id: str
    group_id: str
    state: str
    y: object
    channels: list
    sensor_ids: list
    axes: list
    units: list
    fs: float
    fs_evidence: str
    modality: str
    setup: int = 0
    excitation: str = ''
    vehicle: str = ''
    eligible: bool = True
    reason: str = ''
    metadata: dict = field(default_factory=dict)
    n_samples: int = 0
    sha256: str = ''
    split: str = ''
    cache_path: str = ''
    waveform_hash: str = ''

RECORDS=[]; RAW_INVENTORY=[]; HASHES={}; LOG_ROWS=[]

def register_file(path,task):
    p=Path(path).resolve();st=p.stat();key=str(p)
    if key not in HASHES:
        sha=file_hash(p) if CFG.full_file_hashes else stable_hash([str(p),st.st_size,st.st_mtime_ns])
        HASHES[key]=sha
        RAW_INVENTORY.append({'task':task,'path':key,'bytes':st.st_size,'sha256':sha if CFG.full_file_hashes else None,'stat_fingerprint':None if CFG.full_file_hashes else sha})
    return HASHES[key]

def labels_from_mat(x):
    a=np.asarray(x)
    if a.dtype.kind=='O':
        out=[]
        for v in a.ravel(): out.extend(labels_from_mat(v))
        return out
    if a.dtype.kind not in 'US': raise ValueError('MAT labels must be strings/chars, not numeric indices')
    if a.ndim==2 and a.dtype.itemsize//(4 if a.dtype.kind=='U' else 1)==1:
        return [''.join(row.astype(str)).strip() for row in a]
    return [(v.decode('utf-8') if isinstance(v,bytes) else str(v)).strip() for v in a.ravel()]

def mat_data(path):
    m=loadmat(path,variable_names=['data','labelshulp','fs','Fs','sample_rate','sampling_rate'])
    if 'data' not in m or 'labelshulp' not in m: raise ValueError('Z24 requires data AND labelshulp; numeric pseudo-sensors are refused')
    labels=[canonical_channel(v) for v in labels_from_mat(m['labelshulp'])]
    a=np.asarray(m['data'])
    if a.ndim!=2: raise ValueError(f'MAT data shape {a.shape}')
    if a.shape[1]==len(labels): pass
    elif a.shape[0]==len(labels): a=a.T
    else: raise ValueError('MAT label count does not match a signal axis')
    if len(set(labels))!=len(labels): raise ValueError('Duplicate Z24 sensor labels after normalization')
    return a,labels,m

def scalar_number(v):
    try:
        a=np.asarray(v).squeeze()
        if a.size==1: return float(a)
    except (ValueError,TypeError): return None
    return None

def rate_from_metadata(attrs,n=None,time_vector=None):
    """No generic 'frequency' match; internal_sample_rate is NEVER an effective rate."""
    effective=[];internal=[]
    exact={'sample_rate','sampling_rate','sampling_frequency','sampling_frequency_hz','sample_rate_hz','fs','effective_sample_rate','effective_sampling_rate','output_sample_rate'}
    for key,value in attrs.items():
        k=norm(str(key).split(':')[-1]).replace(' ','_')
        v=scalar_number(value)
        if v is None or not 0<v<1e6: continue
        if 'internal' in k and ('rate' in k or 'frequency' in k): internal.append(v)
        elif k in exact: effective.append((v,str(key)))
    if effective:
        vals=np.array([p[0] for p in effective])
        if not np.allclose(vals,vals[0],rtol=1e-4): raise ValueError(f'Conflicting effective rates: {effective}')
        fs=float(vals[0]);src='native_attr:'+effective[0][1]
    elif time_vector is not None:
        t=np.asarray(time_vector,dtype=float).squeeze()
        if t.ndim!=1 or len(t)<3 or (n and len(t)!=n): raise ValueError('Invalid native elapsed-time vector')
        dt=np.diff(t)
        if np.any(dt<=0): raise ValueError('Native time vector is nonmonotonic')
        fs=float(1/np.median(dt));src='native_elapsed_time_vector'
    else: fs=None;src='unresolved'
    return fs,src,sorted(set(internal))

## 04 — Z24 native PDT/AVT adapter

Uses the five documented reference channels instead of pretending that roving channels are at identical locations in every setup. Setup files are never concatenated as simultaneous samples. Native lengths are preserved (including shorter files); no padding to 65,536, no deletion of `03setup01.mat` without observed QA evidence. Source units may remain native/unknown: no unsupported conversion to SI is invented.

In [5]:
def discover_z24(root):
    root=Path(root)
    if not root.is_dir(): note('missing','Z24',root);return []
    recs=[]
    for p in sorted(root.rglob('*.mat')):
        if norm(p.parent.name)!='avt' or not re.fullmatch(r'\d{1,2}',p.parent.parent.name): continue
        try:
            sid=int(p.parent.parent.name)
            if sid not in Z24_STATES: raise ValueError('Unknown PDT state')
            a,labels,m=mat_data(p)
            selected=[c for c in CFG.z24_reference_channels if c in labels]
            if len(selected)<CFG.min_physical_sensors: raise ValueError(f'Insufficient verified reference channels: {selected}')
            fs,src,_=rate_from_metadata(m)
            if fs is None: fs=100.0;src='KU_Leuven_PDT_documentation_not_embedded'
            if abs(fs-100)>0.01: raise ValueError(f'Unexpected Z24 fs={fs}')
            setup_match=re.search(r'setup(\d+)',p.stem,re.I)
            if not setup_match: raise ValueError('Cannot identify acquisition setup')
            setup=int(setup_match.group(1))
            y=0 if sid in {1,8} else 1
            eligible=not(sid==2 and CFG.z24_exclude_installation_state2)
            key=f'Z24/{sid:02d}/setup{setup:02d}'
            recs.append(Record('Z24',str(p.resolve()),'',key,key,Z24_STATES[sid],y,selected,
                [re.sub('[VLT]$','',c) for c in selected],[c[-1] for c in selected],
                ['native_acceleration_unit_unverified']*len(selected),fs,src,'acceleration',setup=setup,
                eligible=eligible,reason='' if eligible else 'equipment_installation_excluded',n_samples=a.shape[0],
                metadata={'state_id':sid,'all_labels':labels,'source_preprocessing':'native_PDT_acquisition_antialias_30Hz','units_note':'relative/shape comparisons only; no asserted absolute SI amplitude'},sha256=register_file(p,'Z24')))
        except Exception as e: note('loader_error',p,e,task='Z24',trace=traceback.format_exc(limit=2))
    return recs

## 05 — Route345 native HDF5 adapter

Exact top-level `baseline_1`, `baseline_2`, `damage_1`...`damage_6` labels; `sample_rate` takes precedence over `internal_sample_rate`. A date string in the `time` attribute is **not** interpreted as elapsed duration. All selected y/z channels of a test must have the same length and effective rate. Static strain is inventoried but excluded. Both axes share a physical sensor identity, so a simulated failure affects the selected physical sensor, not an invented independent axis.

The prior executed project report (ProTwin_B_Three_Bridge_Reference_Calibrated_Experiment_V0_2_1, 30 July 2026) and AGENTS.md (26 August 2026) recorded near-constant A14/z in the four DAMAGE_6 passages. V1.0 therefore excludes the entire A14 sensor from **all** acquisitions as a predeclared symmetric panel policy. This is reported legacy project evidence, not a newly measured real-data result. The original A14 leaves remain inventoried; no raw data are deleted. This new study uses bridge-specific learning, unlike the earlier ProTwin external-transfer design.

In [6]:
def h5_attrs_lineage(h5,path):
    attrs={};obj=h5[path]
    while True:
        for k,v in obj.attrs.items(): attrs[f'{obj.name}:{k}']=v
        if obj.name=='/': break
        obj=obj.parent
    return attrs

def discover_rt(path):
    p=Path(path);recs=[]
    if not p.is_file(): note('missing','Route345',p);return recs
    md5=file_hash(p,'md5');matched=md5==CFG.route345_v2_md5
    gate('Route345_release_V2',matched,{'md5':md5,'expected':CFG.route345_v2_md5,
         'synthetic_fixture_not_original_release': CFG.mode=='selftest'},mandatory=CFG.mode!='selftest')
    sha=register_file(p,'Route345');tree=[]
    with h5py.File(p,'r') as h5:
        def visitor(name,obj):
            aa={}
            for k,v in obj.attrs.items():
                ar=np.asarray(v)
                aa[k]=json_safe(v) if ar.size<25 else {'shape':list(ar.shape),'first_values':json_safe(ar.ravel()[:5])}
            tree.append({'path':name,'kind':type(obj).__name__,'shape':str(getattr(obj,'shape','')),'attrs':json.dumps(json_safe(aa),ensure_ascii=False)})
        h5.visititems(visitor)
        for top in sorted(h5.keys()):
            y,state,label_src=classify_rt_state(top)
            if y is None: note('unresolved_state',top,'Unknown Route345 root group');continue
            if 'acceleration' not in h5[top]: continue
            for test_name,test in h5[top+'/acceleration'].items():
                if not isinstance(test,h5py.Group): continue
                try:
                    leaves=[];chs=[];physical=[];axes=[];lengths=[];rates=[];internal=[];rate_sources=[];sensor_metadata={}
                    for sensor_name,sensor in sorted(test.items()):
                        if not isinstance(sensor,h5py.Group) or not re.fullmatch(r'A\d+',sensor_name,re.I): continue
                        sn='A'+str(int(sensor_name[1:])).zfill(2)
                        sensor_metadata[sn]=json_safe(dict(sensor.attrs))
                        if sn in CFG.route345_excluded_sensors: continue
                        for axis in CFG.route345_axes:
                            if axis not in sensor: continue
                            ds=sensor[axis];shape=ds.shape
                            if sum(int(v>1) for v in shape)!=1: raise ValueError(f'Non-vector leaf: {ds.name} {shape}')
                            n=max(shape);attrs=h5_attrs_lineage(h5,ds.name)
                            fs,src,ir=rate_from_metadata(attrs,n=n)
                            if fs is None:
                                # Only verified V2 gets this documented fallback, and it is reported as such.
                                if not matched: raise ValueError('No effective sample_rate and unverified release')
                                fs=128.0;src='verified_V2_documented_128Hz_fallback'
                            if abs(fs-128)>0.1: raise ValueError(f'Effective rate {fs} conflicts with V2, internal={ir}')
                            leaves.append(ds.name);chs.append(f'{sn}:{axis}');physical.append(sn);axes.append(axis)
                            lengths.append(n);rates.append(fs);internal.extend(ir);rate_sources.append(src)
                    if len(set(physical))<CFG.min_physical_sensors: raise ValueError('Insufficient physical sensors')
                    if len(set(lengths))!=1: raise ValueError(f'Unequal native channel lengths, no silent truncation: {sorted(set(lengths))}')
                    if not np.allclose(rates,rates[0]): raise ValueError('Effective rates differ across synchronous channels')
                    native=test.name;key='Route345'+native
                    recs.append(Record('Route345',str(p.resolve()),native,key,key,state,y,chs,physical,axes,
                        ['m/s^2']*len(chs),rates[0],';'.join(sorted(set(rate_sources))),'acceleration',n_samples=lengths[0],
                        eligible=matched or CFG.mode=='selftest',reason=('SYNTHETIC_FIXTURE' if CFG.mode=='selftest' else '' if matched else 'unverified_release'),sha256=sha,
                        metadata={'paths':leaves,'root_attrs':json_safe(dict(h5[top].attrs)),'test_attrs':json_safe(dict(test.attrs)),
                            'internal_rates':sorted(set(internal)),'sensor_attrs':sensor_metadata,'label_evidence':label_src,'release_v2_verified':matched,
                            'source_preprocessing':'source_calibrated_linear_detrended_decimated','excluded_panel_sensors':list(CFG.route345_excluded_sensors)}))
                except Exception as e: note('loader_error',test.name,e,task='Route345',trace=traceback.format_exc(limit=2))
    table(tree,'audit/ROUTE345_HDF5_TREE.csv')
    write_json({'configured_excluded_sensors':list(CFG.route345_excluded_sensors),
                'scope':'complete physical sensor, all acquisitions and all selected axes',
                'source_evidence':SOURCES['Route345_panel_QC'],
                'applied_to_all_extracted_records':all(not set(r.sensor_ids).intersection(CFG.route345_excluded_sensors) for r in recs)},
               RUN/'audit/ROUTE345_SYMMETRIC_PANEL_QC.json')
    return recs

## 06 — openLAB: campaign log, CSV schema, exact sensor roles

The earlier MARWAN audit contains the native headers:
`21_1k21_WZ_V47 [m/s]`, `33_1k33_WZ_V86 [m/s]`,
`49_2k17_Accel_UR_SH1 [m/s²]`, `50_2k18_Accel_BAM_SH2 [m/s²]`,
`52_2k19_Laser_Distanz [m]`.

Only the first two are structural responses in the 2025 primary experiment. SH1/SH2 are inputs and the laser is displacement, **not force**. Aliases differing by case or doubled underscores are normalized, but unknown numeric columns never become structural sensors. The log is joined on the acquisition timestamp plus run suffix, not on the first integer in the filename.

In [7]:
STAMP_RE=re.compile(r'\d{4}-\d{2}-\d{2}[ T]\d{2}[-:]\d{2}[-:]\d{2}\s*\(\d+\)')
OL2025_ROLES={
 '21_1K21_WZ_V47':('response','velocity','m/s','OL_V47'),
 '33_1K33_WZ_V86':('response','velocity','m/s','OL_V86'),
 '49_2K17_ACCEL_UR_SH1':('excitation','acceleration','m/s^2','SH1'),
 '50_2K18_ACCEL_BAM_SH2':('excitation','acceleration','m/s^2','SH2'),
 '52_2K19_LASER_DISTANZ':('auxiliary','displacement','m','LASER'),
}

def stamp_key(s):
    m=STAMP_RE.search(str(s))
    if not m: return None
    return re.sub(r'\s+',' ',m.group(0).replace('T',' ').replace(':','-')).strip()

def read_text_file(path):
    raw=Path(path).read_bytes()
    for enc in (('utf-16',) if raw.startswith((b'\xff\xfe',b'\xfe\xff')) else ('utf-8-sig','cp1252','latin1')):
        try: return raw.decode(enc),enc
        except UnicodeError: continue
    raise UnicodeError(f'Undecodable text: {path}')

def load_openlab_log(root):
    index={};rows=[]
    files=[p for p in Path(root).rglob('*.txt') if 'ubersicht' in norm(p.name) or 'messungen' in norm(p.name)]
    for path in files:
        text,_=read_text_file(path);sha=register_file(path,'openLAB2025_log')
        for line_num,line in enumerate(text.splitlines(),1):
            if '|' not in line: continue
            fields=[v.strip() for v in line.strip().strip('|').split('|')]
            if len(fields)<4: continue
            stamp=stamp_key(fields[0])
            if not stamp: continue
            entry={'stamp':stamp,'excitation_raw':fields[1],'vehicle_raw':fields[2], 'state_raw':fields[3],
                   'remarks':fields[4] if len(fields)>4 else '', 'log_path':str(path),'line':line_num,'log_sha256':sha}
            if stamp in index and any(index[stamp][k]!=entry[k] for k in ('excitation_raw','vehicle_raw','state_raw')):
                raise ValueError(f'Conflicting log entries for {stamp}')
            index[stamp]=entry;rows.append(entry)
    table(rows,'audit/OPENLAB_CAMPAIGN_LOG.csv')
    return index

def time_header(name):
    s=norm(name)
    return re.fullmatch(r'(time|timestamp|datetime|zeit|t)(\s*\[.*?\])?',s) is not None

def column_role(name,task):
    c=canonical_channel(name)
    if time_header(name): return ('time','time','s',c)
    if task=='openLAB2025' and c in OL2025_ROLES: return OL2025_ROLES[c]
    if task=='openLAB2024' and re.fullmatch(r'G_ACCZ_PE[12][123]_CB0750_0',c):
        return ('response','acceleration','m/s^2',c)
    if task=='openLAB2025' and ('SH1' in c or 'SH2' in c or 'SHAKER' in c):
        return ('excitation','unknown','unknown',c)
    return ('unresolved','unknown','unknown',c)

def csv_spec(path,task):
    """Inspect bounded bytes, locate actual header, lock delimiter/encoding explicitly."""
    with Path(path).open('rb') as f: raw=f.read(128*1024)
    text=None;encoding=None
    for enc in (('utf-16',) if raw.startswith((b'\xff\xfe',b'\xfe\xff')) else ('utf-8-sig','cp1252','latin1')):
        try: text=raw.decode(enc);encoding=enc;break
        except UnicodeError: continue
    if text is None: raise ValueError('Cannot decode CSV header')
    lines=text.splitlines()
    candidates=[]
    for i,line in enumerate(lines[:80]):
        for sep in (';',',','\t'):
            cols=next(csv.reader([line],delimiter=sep))
            if len(cols)<2: continue
            cols=[c.strip() for c in cols]
            nt=sum(time_header(c) for c in cols)
            known=sum(column_role(c,task)[0]=='response' for c in cols)
            if nt==1 and known:
                candidates.append((known,len(cols),-i,sep,i,cols))
    if not candidates: raise ValueError('No supported time+response CSV schema (not a structural recording or unknown header)')
    _,_,_,sep,line,cols=max(candidates)
    canonical=[canonical_channel(c) for c in cols]
    if len(set(canonical))!=len(canonical): raise ValueError('Duplicate CSV columns after alias normalization')
    return {'encoding':encoding,'sep':sep,'header_line':line,'columns':cols,
            'decimal':',' if sep==';' and re.search(r'\d,\d', '\n'.join(lines[line+1:line+8])) else '.'}

def numeric_series(s,decimal='.'):
    if pd.api.types.is_numeric_dtype(s): return pd.to_numeric(s,errors='coerce').to_numpy(dtype=float)
    v=s.astype(str).str.strip()
    if decimal==',': v=v.str.replace(',','.',regex=False)
    return pd.to_numeric(v,errors='coerce').to_numpy(dtype=float)

def time_seconds(s,name,decimal='.',override=None):
    v=numeric_series(s,decimal)
    if np.isfinite(v).mean()>0.99:
        if not np.isfinite(v).all(): raise ValueError('Missing numeric timestamps')
        n=norm(name); unit=(override or {}).get('time_unit')
        if unit is None:
            m=re.search(r'\[(s|ms|us|ns)\]',n)
            unit=m.group(1) if m else 's'
        scale={'s':1.0,'ms':1e-3,'us':1e-6,'ns':1e-9}.get(unit)
        if scale is None: raise ValueError('Unsupported explicit timestamp unit')
        t=(v-v[0])*scale
        return t,{'kind':'numeric','unit':unit,'unit_evidence':'override_or_header' if override or '[' in n else 'bare_Time_seconds_convention','origin':float(v[0])}
    raw=s.astype(str).str.strip()
    try: dt=pd.to_datetime(raw,utc=True,errors='coerce',format='mixed')
    except (TypeError,ValueError): dt=pd.to_datetime(raw,utc=True,errors='coerce')
    if dt.isna().any():
        dt=pd.to_datetime(raw,utc=True,errors='coerce')
    if dt.isna().any(): raise ValueError('Unparseable datetime timestamps; no row deletion is performed')
    ns=dt.astype('int64').to_numpy()
    return (ns-ns[0]).astype(float)*1e-9,{'kind':'datetime','unit':'s','origin':str(dt.iloc[0])}

def validate_clock(t):
    t=np.asarray(t,float)
    if len(t)<32 or not np.isfinite(t).all(): raise ValueError('Insufficient/invalid clock')
    dt=np.diff(t)
    if np.any(dt<=0): raise ValueError('Duplicate or reversed timestamps: automatic sorting/repair is prohibited')
    med=float(np.median(dt)); dev=np.abs(dt-med)/med
    # A tolerance covers timestamp quantisation; gaps and substantial jitter still block.
    if np.quantile(dev,0.999)>CFG.timestamp_relative_tolerance or np.max(dt)>1.5*med:
        raise ValueError(f'Irregular clock: median_dt={med:g}, relative_q999={np.quantile(dev,.999):g}, max_gap={max(dt):g}')
    fs=1/med
    if not 2<fs<1e6: raise ValueError(f'Implausible stored rate: {fs}')
    return fs,{'median_dt':med,'relative_q999':float(np.quantile(dev,.999)),'duration_s':float(t[-1]-t[0])}

def read_openlab(path,task,spec=None,nrows=None):
    spec=spec or csv_spec(path,task)
    df=pd.read_csv(path,sep=spec['sep'],encoding=spec['encoding'],skiprows=spec['header_line'],
                   header=0,nrows=nrows,dtype=str,keep_default_na=True,on_bad_lines='error')
    df.columns=[str(c).strip() for c in df.columns]
    tc=[c for c in df if time_header(c)]
    if len(tc)!=1: raise ValueError('Exactly one named time column is required')
    override=OVERRIDES.get(str(Path(path).resolve()),{})
    if override and not override.get('evidence'): raise ValueError('Override has no evidence/source')
    t,tmeta=time_seconds(df[tc[0]],tc[0],spec['decimal'],override)
    fs,clock=validate_clock(t)
    selected=[c for c in df if column_role(c,task)[0]=='response']
    if len(selected)<2: raise ValueError('Fewer than two documented response sensors')
    ch=[canonical_channel(c) for c in selected]
    x=np.column_stack([numeric_series(df[c],spec['decimal']) for c in selected])
    return x,ch,fs,t,{**tmeta,**clock,'csv_spec':spec,
        'roles':{c:column_role(c,task) for c in df},'time_column':tc[0]}

def excitation_family(s):
    n=norm(s)
    if 'random' in n:return 'random'
    if 'sweep' in n:return 'sweep'
    if 'ambient' in n:return 'ambient'
    if 'frequenz' in n or 'fixed' in n:return 'fixed_frequency'
    return 'unresolved'

def after_documented_shaker_exchange(stamp,log):
    changes=[key for key,entry in log.items() if 'getauscht' in norm(entry.get('remarks',''))]
    # Stamp strings are ISO-date ordered. Carry the changed regime forward;
    # the absence of a repeated note does not prove the original shaker was restored.
    return bool(changes and stamp and stamp>=min(changes))


def discover_openlab(root,task):
    root=Path(root);recs=[]
    if not root.is_dir(): note('missing',task,root);return recs
    log=load_openlab_log(root) if task=='openLAB2025' else {}
    for p in sorted(root.rglob('*.csv')):
        try:
            spec=csv_spec(p,task)
        except Exception as e:
            note('unsupported_csv_schema',p,e,task=task);continue
        try:
            x,ch,fs,t,meta=read_openlab(p,task,spec,nrows=2048)
            roles=[column_role(c,task) for c in ch]
            if task=='openLAB2025':
                stamp=stamp_key(p.name)
                if stamp not in log: raise ValueError(f'No unique campaign-log entry for {p.name}')
                entry=log[stamp]
                token=entry['state_raw'];y,state,src=classify_ol_state(token)
                if norm(token) not in norm(p.stem): raise ValueError('Filename and log state disagree')
                exc=excitation_family(entry['excitation_raw']);vehicle=norm(entry['vehicle_raw'])
                eligible=(y is not None and exc in CFG.openlab_excitation_primary and vehicle in CFG.openlab_vehicle_primary)
                reason='' if eligible else f'secondary_scope:{src};excitation={exc};vehicle={vehicle}'
                # Incompatible experimental changes are recorded; no claim of excitation invariance.
                if after_documented_shaker_exchange(stamp,log):
                    eligible=False;reason='at_or_after_documented_shaker_exchange_secondary'
                    meta['hardware_regime']='after_documented_shaker_exchange'
                else:meta['hardware_regime']='before_documented_shaker_exchange_or_no_note'
                key=f'openLAB2025/{stamp}';meta.update({'campaign_log':entry,'label_evidence':src})
                modality='velocity';axes=['unverified_direction']*len(ch)
            else:
                y=0;state='reference_2024';exc='event_triggered';vehicle='unknown';eligible=True;reason='reference_only_separate_task'
                if abs(fs-500)>1: raise ValueError(f'Unexpected openLAB reference rate {fs}; expected documented 500 Hz')
                key='openLAB2024/'+str(p.relative_to(root));modality='acceleration';axes=['vertical']*len(ch)
                meta['source_preprocessing']='reference_archive_curated; no repeated bandpass applied'
                meta['reference_label_evidence']=SOURCES['openLAB2024']['url']
            meta.update({'header_preview_rows':len(x),'source_native_fs':fs})
            recs.append(Record(task,str(p.resolve()),'',key,key,state,y,ch,[r[3] for r in roles],axes,[r[2] for r in roles],
                fs,'native_csv_'+meta['kind'],modality,excitation=exc,vehicle=vehicle,eligible=eligible,reason=reason,metadata=meta,
                sha256=register_file(p,task)))
        except Exception as e: note('loader_error',p,e,task=task,trace=traceback.format_exc(limit=2))
    return recs

## 07 — Full-record validation and disk cache

Audit checks bounded CSV previews; full/smoke validates **every selected record in full**. Raw arrays are never z-scored, padded, differentiated or repaired across long gaps. The primary velocity task stays in velocity units. Its PSD is not mislabeled as acceleration. A future physical PSD conversion would be a separate, explicit transformation.

In [8]:
def native_load(r):
    if r.task=='Z24':
        x,chs,_=mat_data(r.path);idx=[chs.index(c) for c in r.channels]
        return np.asarray(x[:,idx],float),float(r.fs),None
    if r.task=='Route345':
        with h5py.File(r.path,'r') as f:
            arr=[np.asarray(f[p][()]).reshape(-1) for p in r.metadata['paths']]
        if len({len(v) for v in arr})!=1: raise ValueError('Route345 channel lengths changed')
        return np.column_stack(arr),float(r.fs),None
    x,chs,fs,t,meta=read_openlab(r.path,r.task,r.metadata['csv_spec'])
    if chs!=r.channels: raise ValueError('Full CSV channel order differs from audit')
    if not np.isclose(fs,r.fs,rtol=0.005): raise ValueError('Full-record clock differs from preview')
    return x,fs,t

def cache_record(r):
    descriptor={'version':VERSION,'implementation':IMPLEMENTATION_HASH,'sha':r.sha256,'key':r.native_key,
                'channels':r.channels,'fs':r.fs,'overrides':OVERRIDES.get(r.path)}
    key=stable_hash(descriptor);path=CACHE/'native'/f'{key}.npy';meta_path=path.with_suffix('.json')
    path.parent.mkdir(parents=True,exist_ok=True)
    candidates=[(path,meta_path,'V1_2')]
    if CFG.reuse_verified_native_cache and CFG.mode!='selftest':
        reference_desc={**descriptor,'version':REFERENCE_VERSION,'implementation':REFERENCE_IMPLEMENTATION_HASH}
        lp=REFERENCE_CACHE/'native'/f'{stable_hash(reference_desc)}.npy'
        candidates.append((lp,lp.with_suffix('.json'),'V1_1_READ_ONLY'))
    for pp,mp,origin in candidates:
        if not (pp.is_file() and mp.is_file()):continue
        metadata=json.loads(mp.read_text());a=np.load(pp,mmap_mode='r',allow_pickle=False)
        if a.ndim!=2 or a.shape[1]!=len(r.channels):raise ValueError('Invalid cached channel shape: '+str(pp))
        if metadata.get('source_sha')!=r.sha256 or not np.isclose(float(metadata['fs']),r.fs):
            raise ValueError('Native cache source/rate provenance mismatch: '+str(pp))
        # Verify the waveform, not just the existence of a previous metadata file.
        h=hashlib.sha256()
        for start in range(0,len(a),131072):
            block=np.asarray(a[start:start+131072],dtype=np.float32)
            block=np.where(np.isnan(block),np.float32(np.nan),block)
            h.update(block.tobytes(order='C'))
        if h.hexdigest()!=metadata['waveform_sha256']:raise ValueError('Native cache waveform checksum mismatch: '+str(pp))
        r.n_samples=len(a);r.waveform_hash=h.hexdigest();r.cache_path=str(pp)
        if origin!='V1_2':note('native_cache_reused_read_only',r.acquisition_id,origin,path=str(pp))
        return r
    x,fs,t=native_load(r)
    if x.ndim!=2 or len(x)<64:raise ValueError('Empty/non-matrix acquisition')
    if x.shape[1]!=len(r.channels):raise ValueError('Channel count mismatch')
    if np.isinf(x).any():raise ValueError('Infinite source samples are invalid')
    fraction=np.isfinite(x).mean(axis=0)
    if np.sum(fraction>.5)<2:raise ValueError('Fewer than two channels with >50% finite samples')
    x=np.asarray(x,dtype=np.float32);canonical=np.where(np.isnan(x),np.float32(np.nan),x)
    wavehash=hashlib.sha256(canonical.tobytes(order='C')).hexdigest()
    tmp=path.with_name(path.name+f'.{os.getpid()}.tmp')
    with tmp.open('wb') as f:np.save(f,x,allow_pickle=False)
    tmp.replace(path)
    write_json({'shape':list(x.shape),'fs':fs,'finite_fraction':fraction.tolist(),'waveform_sha256':wavehash,
                'source':r.path,'source_sha':r.sha256,'clock_validated':t is not None,'implementation':IMPLEMENTATION_HASH},meta_path)
    r.n_samples=len(x);r.waveform_hash=wavehash;r.cache_path=str(path)
    return r

## 08 — Évaluation externe groupée à trois plis
Chaque acquisition primaire est testée **exactement une fois**, hors des ensembles entraînement/développement/calibration du pli concerné. Z24 : l'indice de setup est groupé sur tous les états. Route345 et openLAB : acquisition complète. Les six références disponibles de chacun de ces deux jeux peuvent ainsi toutes contribuer à l'évaluation hors échantillon, sans copier les fenêtres entre partitions.

Les autres acquisitions sont réparties entre entraînement, développement et calibration. Les petits effectifs de calibration restent explicitement signalés : trois plis ne créent pas de nouvelles campagnes ni de nouveaux ponts. Les modèles de plis différents partagent une partie de leurs données d'entraînement ; le bootstrap est **conditionnel**, sans prétention à une indépendance entre plis.

In [9]:
SPLIT_NAMES=('train','dev','cal','test')

def allocate_counts(n):
    if n<5: raise ValueError(f'Need >=5 acquisition groups per class for train/dev/cal/test; observed {n}')
    counts=[2,1,1,1]
    targets=np.asarray(CFG.split_fractions)*n
    while sum(counts)<n:
        idx=int(np.argmax(targets-np.asarray(counts)));counts[idx]+=1
    return counts

def split_records(records,task,seed=None):
    recs=copy.deepcopy(records);seed=CFG.seed if seed is None else seed
    if not recs: raise ValueError(f'No records for {task}')
    if task=='Z24':
        setups=sorted({r.setup for r in recs})
        if len(setups)<5: raise ValueError('Z24 requires >=5 distinct setup indices for 4-way evaluation')
        rng=np.random.default_rng(seed);rng.shuffle(setups)
        counts=allocate_counts(len(setups));mapping={};pos=0
        for s,n in zip(SPLIT_NAMES,counts):
            mapping.update({v:s for v in setups[pos:pos+n]});pos+=n
        for r in recs:r.split=mapping[r.setup]
    else:
        grouped=defaultdict(list)
        for r in recs:grouped[r.group_id].append(r)
        for g,rr in grouped.items():
            if len({r.y for r in rr})!=1: raise ValueError(f'Conflicting labels within group {g}')
        for y in (0,1):
            ids=sorted(g for g,rr in grouped.items() if rr[0].y==y)
            counts=allocate_counts(len(ids))
            rng=np.random.default_rng(seed+101*y);rng.shuffle(ids)
            # Round-robin context ordering increases training coverage without consulting waveforms or performance.
            buckets=defaultdict(list)
            for g in ids:
                r=grouped[g][0];context=r.metadata.get('campaign_log',{}).get('excitation_raw',r.state)
                buckets[context].append(g)
            ordered=[]
            while any(buckets.values()):
                for context in sorted(buckets):
                    if buckets[context]:ordered.append(buckets[context].pop())
            pos=0
            for s,n in zip(SPLIT_NAMES,counts):
                for g in ordered[pos:pos+n]:
                    for r in grouped[g]:r.split=s
                pos+=n
    for s in SPLIT_NAMES:
        if {r.y for r in recs if r.split==s}!={0,1}: raise ValueError(f'{task}/{s} does not contain both structural classes')
    assert len({r.acquisition_id for r in recs})==len(recs),'Duplicate acquisition IDs require grouping/reconciliation'
    audit_split(recs)
    return recs

def audit_split(recs):
    seen={};waves={}
    for r in recs:
        for key in (r.group_id,):
            if key in seen and seen[key]!=r.split: raise ValueError('Group leakage')
            seen[key]=r.split
        if r.waveform_hash:
            if r.waveform_hash in waves and waves[r.waveform_hash]!=r.split:
                raise ValueError(f'Exact waveform duplicated across splits: {r.path}')
            waves[r.waveform_hash]=r.split
    return True

def panel_from_train(records):
    tr=[r for r in records if r.split=='train'];count=Counter(c for r in tr for c in set(r.channels))
    panel=sorted(c for c,n in count.items() if n/max(1,len(tr))>=CFG.train_channel_coverage)
    if records[0].task=='Z24':panel=[c for c in CFG.z24_reference_channels if c in panel]
    if len(panel)<2:raise ValueError('Insufficient stable training-panel channels; no test-derived fallback')
    registry={}
    for r in records:
        for c,s,a,u in zip(r.channels,r.sensor_ids,r.axes,r.units):
            if c in registry and registry[c]!=(s,a,u):raise ValueError(f'Inconsistent physical identity/unit: {c}')
            registry[c]=(s,a,u)
    physical=[registry[c][0] for c in panel]
    if len(set(physical))<CFG.min_physical_sensors:raise ValueError('Insufficient physical sensors in training panel')
    return panel,physical,[registry[c][1] for c in panel]
def inner_counts(n):
    if n<4:raise ValueError(f'Need >=4 non-test units per class for train(>=2)/dev/cal; got {n}')
    counts=np.array([2,1,1]);target=np.array([.65,.175,.175])*n
    while counts.sum()<n:counts[int(np.argmax(target-counts))]+=1
    return counts.tolist()

def stratified_unit_buckets(units,contexts,k,seed):
    """No signal or model scores enter allocation. Round robin across shuffled contexts."""
    rng=np.random.default_rng(seed);buckets=[[] for _ in range(k)]
    mapping=defaultdict(list)
    for u,c in zip(units,contexts):mapping[str(c)].append(u)
    order=list(sorted(mapping));rng.shuffle(order);offset=0
    for c in order:
        a=sorted(mapping[c]);rng.shuffle(a)
        for j,u in enumerate(a):buckets[(offset+j)%k].append(u)
        offset=(offset+len(a))%k
    return buckets

def outer_protocols(records,task):
    """Three outer folds. Test units rotate once; inner train/dev/cal use only the remaining units."""
    k=CFG.outer_folds
    if task=='Z24':
        units=sorted(set(r.setup for r in records));rng=np.random.default_rng(CFG.seed);rng.shuffle(units)
        if len(units)<6:raise ValueError('Z24: at least six setup indices are needed for outer evaluation')
        test_buckets=[list(a) for a in np.array_split(units,k)]
        strata=None
    else:
        grouped=defaultdict(list)
        for r in records:grouped[r.group_id].append(r)
        if any(len(set(r.y for r in v))!=1 for v in grouped.values()):raise ValueError('Contradictory labels within a physical group')
        strata={}
        for y in (0,1):
            rr=[v[0] for g,v in sorted(grouped.items()) if v[0].y==y]
            if len(rr)<6:raise ValueError(f'{task}: need at least six independent acquisition groups for class {y}, found {len(rr)}')
            contexts=[r.metadata.get('campaign_log',{}).get('excitation_raw',r.state) for r in rr]
            strata[y]=stratified_unit_buckets([r.group_id for r in rr],contexts,k,CFG.seed+101*y)
    protocols=[];test_counts=Counter()
    for fold in range(k):
        rr=copy.deepcopy(records);mapping={}
        if task=='Z24':
            testing=set(test_buckets[fold]);remaining=sorted(set(units)-testing)
            rng=np.random.default_rng(CFG.seed+701*(fold+1));rng.shuffle(remaining)
            counts=inner_counts(len(remaining));at=0
            mapping={u:'test' for u in testing}
            for s,n in zip(('train','dev','cal'),counts):
                mapping.update({u:s for u in remaining[at:at+n]});at+=n
            for r in rr:r.split=mapping[r.setup]
        else:
            for y in (0,1):
                testing=set(strata[y][fold]);remaining=sorted(set(sum(strata[y],[]))-testing)
                rng=np.random.default_rng(CFG.seed+701*(fold+1)+101*y);rng.shuffle(remaining)
                counts=inner_counts(len(remaining));at=0
                mapping.update({u:'test' for u in testing})
                for s,n in zip(('train','dev','cal'),counts):
                    mapping.update({u:s for u in remaining[at:at+n]});at+=n
            for r in rr:r.split=mapping[r.group_id]
        for part in SPLIT_NAMES:
            if {r.y for r in rr if r.split==part}!={0,1}:raise ValueError(f'{task} fold {fold}/{part}: missing a structural class')
        audit_split(rr)
        for r in rr:
            if r.split=='test':test_counts[r.acquisition_id]+=1
        protocols.append(rr)
    if set(test_counts)!={r.acquisition_id for r in records} or any(v!=1 for v in test_counts.values()):
        raise AssertionError('Every included acquisition must occur in exactly one outer test fold')
    return protocols


## 09 — Quality-preserving preprocessing and spectral features

Two views are calculated from the same **source-native** window. The quality view precedes centering/interpolation. The structural view interpolates only short *internal* gaps, with both a duration limit and a missing-fraction limit. Larger gaps make that channel structurally unavailable. Invalid structural values are imputed from training statistics only, with an explicit validity mask; zero imputation is never interpreted as reconstructed vibration.

Primary structural preprocessing is Welch with segment demeaning and explicit 0.5–25 Hz feature extraction, **not repeated bandpass filtering of already processed archives**. All native sampling rates are retained. Power is represented as spectral shape plus relative/amplitude information; the quality branch retains gain/offset changes. Welch uses at least three averaged segments; `nfft=nperseg`. Grid interpolation is a fixed feature representation, not additional frequency resolution.

In [10]:
from scipy.integrate import trapezoid
TINY=np.finfo(np.float64).tiny
QBASE_NAMES=['missing_fraction','longest_gap_fraction','flat_step_fraction','longest_flat_fraction','rail_fraction',
             'spike_fraction','mean_over_rms','crest_factor','kurtosis','normalized_drift','hf_energy_ratio','spectral_flatness','peer_incoherence']
QREF_NAMES=['log_rms_reference_ratio','mean_reference_shift','log_std_reference_ratio']
SF_NAMES=[f'log_psd_shape_{i:02d}' for i in range(CFG.psd_bins)]+['log_band_energy','centroid_hz','bandwidth_hz','spectral_entropy',
             'peak_hz','skewness','kurtosis','crest_factor','mean_peer_coherence']


def longest_true(mask):
    z=np.r_[False,np.asarray(mask,bool),False].astype(np.int8);d=np.diff(z)
    starts=np.flatnonzero(d==1);ends=np.flatnonzero(d==-1)
    return int(np.max(ends-starts)) if len(starts) else 0

def repair_short_gaps(x,fs):
    y=np.asarray(x,float).copy();bad=~np.isfinite(y)
    if not bad.any():return y,True
    if bad[0] or bad[-1] or bad.mean()>CFG.max_missing_fraction or longest_true(bad)>max(1,int(CFG.max_interpolation_gap_seconds*fs)):
        return y,False
    good=~bad;ix=np.arange(len(y));y[bad]=np.interp(ix[bad],ix[good],y[good])
    return y,True

def quality_basic(x,fs):
    x=np.asarray(x,float);finite=np.isfinite(x);n=len(x);miss=1-finite.mean()
    q=np.full(len(QBASE_NAMES),np.nan,float);q[0]=miss;q[1]=longest_true(~finite)/max(n,1)
    aux=np.array([np.nan,np.nan,np.nan],float) # RMS, mean, std for training references
    if finite.sum()<8:return q,aux
    y=x[finite];rms=float(np.sqrt(np.mean(y*y)));mu=float(np.mean(y));std=float(np.std(y));scale=max(rms,1e-30)
    med=np.median(y);mad=max(1.4826*np.median(np.abs(y-med)),std*0.01,1e-30)
    consecutive=finite[:-1]&finite[1:];diff=np.diff(x);tol=1e-7*max(std,rms,1e-30)
    flats=consecutive & (np.abs(diff)<=tol)
    q[2]=flats.sum()/max(consecutive.sum(),1);q[3]=longest_true(flats)/max(n-1,1)
    lo,hi=np.min(y),np.max(y);q[4]=np.mean((np.abs(y-lo)<=tol)|(np.abs(y-hi)<=tol))
    q[5]=np.mean(np.abs(y-med)>8*mad);q[6]=mu/scale;q[7]=np.max(np.abs(y))/scale
    q[8]=float(stats.kurtosis(y,bias=False)) if std>1e-30 else 0.
    t=np.flatnonzero(finite)/fs
    slope=np.polyfit(t-t[0],y,1)[0] if len(t)>2 else 0
    q[9]=slope*(n/fs)/max(std,rms*.01,1e-30)
    if finite.all() and std>1e-30:
        yy=y-y.mean();power=np.abs(np.fft.rfft(yy))**2;power=np.maximum(power,1e-30*max(power.max(),1e-30))
        q[10]=power[int(.75*len(power)):].sum()/max(power.sum(),TINY)
        q[11]=np.exp(np.mean(np.log(power)))/max(np.mean(power),TINY)
    aux[:]=[rms,mu,std]
    return q,aux

def welch_parameters(n,fs):
    seg=min(int(round(CFG.welch_segment_seconds*fs)),n//2)
    if seg<16:raise ValueError('Window too short for an averaged spectrum')
    return {'fs':fs,'window':signal.windows.hann(seg,sym=False),'nperseg':seg,'noverlap':seg//2,'nfft':seg,'detrend':'constant','scaling':'density'}

def features_window(raw,fs,present,axes):
    raw=np.asarray(raw,float);S=raw.shape[1]
    sf=np.full((S,len(SF_NAMES)),np.nan,np.float32);q=np.full((S,len(QBASE_NAMES)),np.nan,np.float32)
    aux=np.full((S,3),np.nan,np.float32);valid=np.zeros(S,np.float32);clean={}
    low,high=map(float,CFG.band_hz)
    if high>=fs/2:raise ValueError(f'Feature band {CFG.band_hz} exceeds Nyquist at fs={fs}')
    pars=welch_parameters(len(raw),fs)
    grid=np.linspace(low,high,CFG.psd_bins)
    for j in range(S):
        if not present[j]:continue
        qb,aa=quality_basic(raw[:,j],fs);q[j]=qb;aux[j]=aa
        y,ok=repair_short_gaps(raw[:,j],fs)
        if not ok or not np.isfinite(y).all() or np.std(y)<1e-30:continue
        f,p=signal.welch(y,**pars)
        band=(f>=low)&(f<=high)
        if band.sum()<4:continue
        pp=np.interp(grid,f,p);energy=float(trapezoid(pp,grid))
        if energy<=TINY:continue
        dens=pp/energy;shape=np.log10(np.maximum(dens,1e-20))
        centroid=float(trapezoid(grid*dens,grid));width=float(np.sqrt(max(0,trapezoid((grid-centroid)**2*dens,grid))))
        mass=pp/pp.sum();entropy=float(-np.sum(mass*np.log(mass+1e-30))/np.log(len(mass)))
        yy=y-y.mean();rms=np.sqrt(np.mean(yy**2))
        sc=[np.log10(energy),centroid,width,entropy,float(grid[np.argmax(pp)]),float(stats.skew(yy,bias=False)),
            float(stats.kurtosis(yy,bias=False)),float(np.max(np.abs(yy))/max(rms,1e-30)),np.nan]
        sf[j]=np.r_[shape,sc];valid[j]=1;clean[j]=y
    # Same-axis peers only; unverified directions are not claimed as physical axis matches.
    for j in range(S):
        if j not in clean:continue
        peers=[k for k in clean if k!=j and axes[k]==axes[j] and axes[j] not in {'unverified_direction','unknown',''}][:2]
        co=[]
        for k in peers:
            with np.errstate(divide='ignore',invalid='ignore'):
                f,c=signal.coherence(clean[j],clean[k],fs=fs,window=pars['window'],nperseg=pars['nperseg'],noverlap=pars['noverlap'],nfft=pars['nfft'],detrend='constant')
            mask=(f>=low)&(f<=high)&np.isfinite(c)
            if mask.any():co.append(float(np.mean(c[mask])))
        if co:sf[j,-1]=np.mean(co);q[j,-1]=1-np.mean(co)
    return sf,q,aux,valid

## 10 — Perturbations contrôlées et reproductibles
Les perturbations sont appliquées aux **fenêtres source après séparation des acquisitions et avant nos caractéristiques**. Le même capteur physique est ciblé dans les fenêtres d'une acquisition/scénario de test ; toutes ses directions sélectionnées sont affectées. L'intervalle du défaut reste local à chaque fenêtre : ce protocole teste des perturbations de fenêtre, **pas une panne continue reconstituée sur tout l'enregistrement**.

Les versions nominales sont source-native, pas certifiées exemptes de pannes. Familles connues : lacune, gel, gain, saturation et bruit ; la dérive reste entièrement absente d'entraînement, développement et calibration. Les trois sévérités sont appliquées à tous les parents admissibles, pas seulement aux cas faciles.

In [11]:
@dataclass(frozen=True)
class Fault:
    kind:str
    severity:float
    physical_sensor:str
    start:int
    stop:int
    seed:int
    parameter:float


def make_fault(n,physical,present,kind,severity,key):
    seed=int(stable_hash([CFG.seed,key,kind,severity])[:8],16)
    rng=np.random.default_rng(seed)
    available=sorted(set(p for p,m in zip(physical,present) if m))
    if not available:raise ValueError('No physical sensor to corrupt')
    sensor=str(rng.choice(available))
    duration=max(1,int(n*(.10+.65*severity)));start=int(rng.integers(0,max(1,n-duration+1)))
    parameter=float(severity)
    if kind=='gain':parameter=float(np.exp((1 if rng.random()<.5 else -1)*np.log(4)*severity))
    if kind=='noise':parameter=float(30-30*severity) # SNR dB, lower is more severe
    return Fault(kind,float(severity),sensor,start,min(n,start+duration),seed,parameter)


def inject_fault(raw,physical,fault):
    x=np.array(raw,dtype=float,copy=True);rng=np.random.default_rng(fault.seed)
    affected=np.zeros(x.shape[1],np.float32)
    for j,p in enumerate(physical):
        if p!=fault.physical_sensor:continue
        v=x[:,j];good=np.isfinite(v)
        if not good.any():continue
        yy=v[good];scale=max(float(np.std(yy)),1e-30);mu=float(np.median(yy));a,b=fault.start,fault.stop
        if fault.kind=='missing':v[a:b]=np.nan
        elif fault.kind=='freeze':v[a:b]=v[a-1] if a>0 and np.isfinite(v[a-1]) else mu
        elif fault.kind=='gain':v[a:b]*=fault.parameter
        elif fault.kind=='clipping':
            bound=max(scale*(3.0-2.75*fault.severity),1e-30);v[a:b]=np.clip(v[a:b],mu-bound,mu+bound)
        elif fault.kind=='noise':
            sigma=scale*10**(-fault.parameter/20);v[a:b]+=rng.normal(0,sigma,b-a)
        elif fault.kind=='drift':v[a:b]+=np.linspace(0,8*fault.severity*scale,b-a)
        else:raise ValueError(fault.kind)
        x[:,j]=v;affected[j]=1
    return x,affected

## 11 — Caches de fenêtres et magasins de caractéristiques
Les signaux natifs sont vérifiés puis mappés sur disque. Les caractéristiques réutilisables ne sont pas recalculées à chaque pli. Le cache dépend des empreintes de données, du code et du protocole. Les prédictions, calibrations et références saines, qui sont **apprises**, sont en revanche propres à chaque pli. Les limites mémoire ne déclenchent jamais une réduction silencieuse du test.

In [12]:
def cached_window_features(r,start,raw,present,axes,panel,fault):
    # Transforms learned from data are NOT stored here. This cache contains deterministic extraction only.
    key=stable_hash({'code':IMPLEMENTATION_HASH,'source':r.sha256,'native':r.native_key,'start':start,
          'n':len(raw),'fs':r.fs,'panel':panel,'present':present,'axes':axes,
          'fault':None if fault is None else asdict(fault),'band':CFG.band_hz,'bins':CFG.psd_bins,
          'welch_s':CFG.welch_segment_seconds,'gap':CFG.max_interpolation_gap_seconds,
          'missing':CFG.max_missing_fraction})
    p=CACHE/'windows'/key[:2]/(key+'.npz')
    if p.exists():
        try:
            with np.load(p,allow_pickle=False) as z:return tuple(z[k].copy() for k in ('sf','qb','aux','valid'))
        except Exception as e:note('cache_rebuild',p,'Unreadable deterministic feature cache: '+str(e))
    answer=features_window(raw,r.fs,present,axes)
    p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+f'.{os.getpid()}.tmp')
    with tmp.open('wb') as f:np.savez_compressed(f,**dict(zip(('sf','qb','aux','valid'),answer)))
    tmp.replace(p)
    return answer

@dataclass
class Store:
    meta:pd.DataFrame
    sf:np.ndarray
    qb:np.ndarray
    aux:np.ndarray
    present:np.ndarray
    valid:np.ndarray
    yf:np.ndarray
    seq:object=None
    physical:object=None


def slice_store(st,indices):
    ix=np.asarray(indices)
    return Store(st.meta.iloc[ix].reset_index(drop=True),*[getattr(st,k)[ix] for k in ('sf','qb','aux','present','valid','yf')],seq=None if st.seq is None else st.seq[ix],physical=st.physical)


def starts_for(n,w,split):
    overlap=CFG.train_overlap if split=='train' else CFG.eval_overlap
    starts=list(range(0,n-w+1,max(1,round(w*(1-overlap))))) if n>=w else []
    if CFG.mode in {'smoke','selftest'} and len(starts)>CFG.smoke_windows_per_acquisition:
        starts=[starts[i] for i in np.unique(np.linspace(0,len(starts)-1,CFG.smoke_windows_per_acquisition).astype(int))]
    return starts


def store_key(records,panel,seconds,policy):
    return stable_hash({'version':VERSION,'implementation':IMPLEMENTATION_HASH,'data':[(r.sha256,r.native_key,r.acquisition_id,r.split,r.y,r.fs,r.channels) for r in records],
        'panel':panel,'seconds':seconds,'policy':policy,'config':{k:v for k,v in asdict(CFG).items() if k not in {'mode','project_root','conference','conference_url','study_name','cpu_threads','device_preference','resume','progress_every','allow_local_compute','enable_quality_transfer','enable_minirocket','bootstrap_replicates','show_figures'}},
        'effective_smoke_cap':CFG.smoke_windows_per_acquisition if CFG.mode in {'smoke','selftest'} else None})


def build_store(records,panel,physical,axes,seconds,policy='nominal'):
    if not records:raise ValueError('Empty feature request')
    key=store_key(records,panel,seconds,policy);cache=CACHE/'features'/key;cache.mkdir(parents=True,exist_ok=True)
    pp=cache/'arrays.npz';mp=cache/'meta.json'
    if pp.exists() and mp.exists():
        z=np.load(pp,allow_pickle=False)
        return Store(pd.DataFrame(json.loads(mp.read_text())),*[z[k] for k in ('sf','qb','aux','present','valid','yf')],seq=z['seq'] if 'seq' in z else None,physical=list(physical))
    rows=[];arrays={k:[] for k in ('sf','qb','aux','present','valid','yf')};seqs=[]
    seen_faults=[f for f in CFG.fault_types if f!=CFG.heldout_fault]
    for ri,r in enumerate(records):
        native=np.load(r.cache_path,mmap_mode='r',allow_pickle=False)
        present=np.array([c in r.channels for c in panel],np.float32)
        nwin=round(seconds*r.fs);starts=starts_for(len(native),nwin,r.split)
        if not starts:raise ValueError(f'{r.acquisition_id} shorter than selected {seconds}s; duration must be revised on development, not silently dropped')
        for start in starts:
            x=np.full((nwin,len(panel)),np.nan,float)
            for j,c in enumerate(panel):
                if c in r.channels:x[:,j]=native[start:start+nwin,r.channels.index(c)]
            parent=f'{r.acquisition_id}::{start}:{start+nwin}'
            variants=[('nominal',None)]
            if policy=='train_aug':
                for k in range(CFG.train_fault_repeats):
                    # Hash independent of y; reproducible without using test features.
                    rng=np.random.default_rng(int(stable_hash([parent,k,CFG.seed])[:8],16))
                    ft=str(rng.choice(seen_faults));sev=float(rng.choice(CFG.fault_severities))
                    variants.append((f'{ft}_{sev}_r{k}',make_fault(nwin,physical,present,ft,sev,[parent,k,'train'])))
            elif policy in {'seen_grid','test_grid'}:
                families=seen_faults if policy=='seen_grid' else list(CFG.fault_types)
                for ft in families:
                    for sev in CFG.fault_severities:
                        for k in range(CFG.eval_fault_repeats):
                            variants.append((f'{ft}_{sev}_r{k}',make_fault(nwin,physical,present,ft,sev,[r.acquisition_id,k,r.split])))
            elif policy!='nominal':raise ValueError(policy)
            for tag,fault in variants:
                raw=x if fault is None else inject_fault(x,physical,fault)[0]
                yf=np.zeros(len(panel),np.float32) if fault is None else np.array([p==fault.physical_sensor for p in physical],np.float32)
                sf,qb,aux,valid=cached_window_features(r,start,raw,present,axes,panel,fault)
                arrays['sf'].append(sf);arrays['qb'].append(qb);arrays['aux'].append(aux)
                arrays['present'].append(present.copy());arrays['valid'].append(valid);arrays['yf'].append(yf)
                rows.append({'variant_id':parent+'::'+tag,'parent_id':parent,'group_id':r.group_id,'acquisition_id':r.acquisition_id,
                    'task':r.task,'split':r.split,'y':int(r.y),'state':r.state,'start_sample':start,'stop_sample':start+nwin,'fs':r.fs,
                    'seconds':seconds,'fault_type':'nominal' if fault is None else fault.kind,'severity':0 if fault is None else fault.severity,
                    'fault_sensor':'' if fault is None else fault.physical_sensor,'fault_seed':None if fault is None else fault.seed,
                    'fault_start':None if fault is None else fault.start,'fault_stop':None if fault is None else fault.stop,
                    'fault_parameter':None if fault is None else fault.parameter,
                    'scenario':('D' if r.y else 'H')+('-N' if fault is None else '-F'),
                    'excitation':r.metadata.get('campaign_log',{}).get('excitation_raw',r.excitation),
                    'required_physical_sensors':CFG.min_physical_sensors,
                    'usable_physical_sensors':len(set(p for p,m in zip(physical,valid) if m))})
                if CFG.enable_minirocket:
                    # Optional temporal baseline: within-task common grid with anti-alias resampling.
                    target_fs=64.0;frac=Fraction(target_fs/r.fs).limit_denominator(10000)
                    seq=np.zeros((len(panel),int(round(seconds*target_fs))),np.float32)
                    for j in range(len(panel)):
                        rr,ok=repair_short_gaps(raw[:,j],r.fs)
                        if not ok or not np.isfinite(rr).all():continue
                        yy=signal.resample_poly(rr-rr.mean(),frac.numerator,frac.denominator)
                        if len(yy)<seq.shape[1]:raise ValueError('Unexpected resampling length')
                        seq[j]=yy[:seq.shape[1]]
                    seqs.append(seq)
                approx=sum(np.asarray(v[-1]).nbytes for v in arrays.values())*len(rows)
                if seqs:approx+=seqs[-1].nbytes*len(rows)
                if approx>CFG.max_feature_store_gb*1024**3:raise MemoryError('Feature-store memory cap exceeded; reduce configured grid or use more memory explicitly')
        if ri%CFG.progress_every==0 or ri==len(records)-1:
            print(f'  features {r.task} {seconds:g}s {policy}: {ri+1}/{len(records)} acquisitions; {len(rows)} variants',flush=True)
    if not rows:raise ValueError('No extracted windows')
    if len({r['variant_id'] for r in rows})!=len(rows):raise ValueError('Duplicate variant IDs')
    aa={k:np.stack(v).astype(np.float32) for k,v in arrays.items()}
    if seqs:aa['seq']=np.stack(seqs)
    tmp=pp.with_name(pp.name+f'.{os.getpid()}.tmp')
    with tmp.open('wb') as f:np.savez_compressed(f,**aa)
    tmp.replace(pp);write_json(rows,mp)
    return Store(pd.DataFrame(rows),*[aa[k] for k in ('sf','qb','aux','present','valid','yf')],seq=aa.get('seq'),physical=list(physical))

## 12 — Références issues uniquement de l'entraînement et agrégation par acquisition

Le prétraitement appris reste spécifique au pont/pli. Chaque membre de l'ensemble est agrégé sur les fenêtres de l'acquisition **avant** de calculer sa dispersion. Les colonnes par membre sont conservées pour audit ; plus aucun maximum des écarts-types fenêtre n'entre dans la décision V1.1.

In [13]:
class FeatureTransform:
    def fit(self,st):
        if set(st.meta.split)!={'train'}:raise ValueError('FeatureTransform.fit only accepts the training partition')
        nominal=(st.meta.fault_type=='nominal').to_numpy();healthy=nominal&(st.meta.y.to_numpy()==0)
        if not healthy.any():raise ValueError('No training reference windows')
        refs=[];spectral_refs=[]
        for group in st.meta.loc[healthy,'group_id'].unique():
            ids=np.flatnonzero(healthy&(st.meta.group_id.to_numpy()==group))
            with warnings.catch_warnings():
                warnings.simplefilter('ignore',RuntimeWarning)
                refs.append(np.nanmedian(st.aux[ids],axis=0))
                v=st.sf[ids].copy();v[st.valid[ids]<=0]=np.nan;spectral_refs.append(np.nanmedian(v,axis=0))
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',RuntimeWarning)
            self.reference=np.nanmedian(np.stack(refs),axis=0)
            self.healthy_spectrum=np.nanmedian(np.stack(spectral_refs),axis=0)
        # Mask channels lacking a healthy reference only in relative features, never invent SI units.
        self.healthy_spectrum=np.where(np.isfinite(self.healthy_spectrum),self.healthy_spectrum,0.)
        x=st.sf[nominal].copy();x[st.valid[nominal]<=0]=np.nan
        self.sm,self.ss=self._stats(x)
        q=self.quality_raw(st);self.qm,self.qs=self._stats(q)
        self.train_groups=sorted(st.meta.group_id.unique().tolist())
        self.shape_features=CFG.psd_bins
        return self
    @staticmethod
    def _stats(x):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',RuntimeWarning)
            med=np.nanmedian(x,axis=0);mad=1.4826*np.nanmedian(np.abs(x-med),axis=0);std=np.nanstd(x,axis=0)
        med=np.where(np.isfinite(med),med,0.)
        scale=np.where(np.isfinite(mad)&(mad>1e-6),mad,np.where(np.isfinite(std)&(std>1e-6),std,1.))
        return med.astype(np.float32),scale.astype(np.float32)
    def quality_raw(self,st):
        a=st.aux.astype(float);r=self.reference.astype(float)
        rms_ref=np.maximum(r[:,0],1e-30);std_ref=np.maximum(r[:,2],rms_ref*.01)
        add=np.stack([np.log(np.maximum(a[:,:,0],1e-30)/rms_ref),
            (a[:,:,1]-r[:,1])/std_ref,np.log(np.maximum(a[:,:,2],1e-30)/np.maximum(r[:,2],1e-30))],axis=-1)
        return np.concatenate([st.qb,add.astype(np.float32)],axis=-1)
    def apply(self,st):
        x=(st.sf-self.sm)/self.ss
        # Explicit training-reference contrast. Spatial calibration remains bridge- and fold-specific.
        delta=(st.sf[:,:,:self.shape_features]-self.healthy_spectrum[None,:,:self.shape_features])/self.ss[None,:,:self.shape_features]
        x=np.concatenate([x,delta],axis=-1)
        q=(self.quality_raw(st)-self.qm)/self.qs
        x=np.nan_to_num(x,nan=0.,posinf=0.,neginf=0.);q=np.nan_to_num(q,nan=0.,posinf=0.,neginf=0.)
        return np.clip(x,-30,30).astype(np.float32),np.clip(q,-30,30).astype(np.float32)


def example_weights(meta,balance=True):
    # Equal mass for each independent acquisition, regardless of windows or corruptions.
    count=meta.groupby('group_id').size().to_dict();groups=meta[['group_id','y']].drop_duplicates()
    byclass=groups.groupby('y').size().to_dict()
    w=np.array([1/count[g]/(byclass[y] if balance else 1) for g,y in zip(meta.group_id,meta.y)],float)
    return w/max(w.mean(),1e-30)


def flat_struct(st,tr):
    x,_=tr.apply(st)
    return np.concatenate([x.reshape(len(x),-1),st.valid],axis=1)


EVENT_KEYS=['acquisition_id','group_id','task','split','y','state','fault_type','severity','excitation']

def aggregate_predictions(meta,p,**extra):
    """One row per acquisition/fault/severity. Preserve member means before dispersion."""
    df=meta.copy();p=np.asarray(p,float)
    if len(p)!=len(df) or not np.isfinite(p).all():raise ValueError('Invalid window predictions')
    df['p']=p
    for k,v in extra.items():
        a=np.asarray(v,float)
        if a.shape!=(len(df),) or not np.isfinite(a).all():raise ValueError('Invalid prediction field: '+k)
        df[k]=a
    cols={'p':'mean','usable_physical_sensors':'min'}
    for k in extra:cols[k]='max' if k=='qmax' else 'mean'
    out=df.groupby(EVENT_KEYS,dropna=False,sort=False).agg(cols).reset_index()
    out['event_id']=out['acquisition_id']+'::'+out['fault_type']+'::'+out['severity'].astype(str)
    if out.event_id.duplicated().any():raise ValueError('Conflicting acquisition/event metadata')
    out['n_windows']=df.groupby(EVENT_KEYS,dropna=False,sort=False).size().to_numpy()
    return out

def aggregate_member_predictions(meta,members,plain,qmax):
    members=np.asarray(members,float);plain=np.asarray(plain,float)
    if members.ndim!=2 or members.shape!=plain.shape or members.shape[1]!=len(meta):
        raise ValueError('Expected [member, window] aligned arrays')
    if not np.isfinite(members).all() or not np.isfinite(plain).all():raise FloatingPointError('Nonfinite probabilities')
    if np.min(members)<0 or np.max(members)>1:raise ValueError('Member probabilities outside [0,1]')
    extra={'qmax':qmax}
    for k in range(len(members)):
        extra[f'p_member_{k:02d}']=members[k];extra[f'plain_member_{k:02d}']=plain[k]
    out=aggregate_predictions(meta,members.mean(0),**extra)
    pm=out[[f'p_member_{k:02d}' for k in range(len(members))]].to_numpy()
    pp=out[[f'plain_member_{k:02d}' for k in range(len(members))]].to_numpy()
    out['p']=pm.mean(1)
    out['u_ensemble']=pm.std(axis=1,ddof=0)
    out['disagreement']=np.abs(pm.mean(1)-pp.mean(1))
    out['n_ensemble_members']=len(members)
    # Persist a deliberately labelled legacy score for the frozen-predictor audit only.
    # The original used maxima. Reconstruct only as a diagnostic, NEVER for V1.1 decisions.
    temp=meta[EVENT_KEYS].copy();temp['reference_max_window_std']=members.std(0,ddof=0)
    temp['reference_max_window_disagreement']=np.abs(members.mean(0)-plain.mean(0))
    old=temp.groupby(EVENT_KEYS,dropna=False,sort=False)[['reference_max_window_std','reference_max_window_disagreement']].max().reset_index()
    out=out.merge(old,on=EVENT_KEYS,validate='one_to_one',sort=False)
    return out

## 13 — Development-only duration and baseline selection

Candidate durations are compared using nominal train/development acquisitions. The test is not loaded for this selection. Macro performance is computed after averaging windows within an acquisition. A duration that cannot represent every requested training/development record is infeasible, rather than silently deleting the difficult records.

In [14]:
def dev_score(st,p):
    d=aggregate_predictions(st.meta,p)
    if d.y.nunique()!=2:return -np.inf
    return float(balanced_accuracy_score(d.y,(d.p>=.5).astype(int)))


def choose_duration(records,panel,physical,axes):
    rows=[];best=None
    requested=[r for r in records if r.split in {'train','dev'}]
    for sec in CFG.window_seconds:
        try:
            st=build_store(requested,panel,physical,axes,float(sec),'nominal')
            tr=slice_store(st,np.flatnonzero(st.meta.split.to_numpy()=='train'))
            dv=slice_store(st,np.flatnonzero(st.meta.split.to_numpy()=='dev'))
            tx=FeatureTransform().fit(tr);x=flat_struct(tr,tx);xv=flat_struct(dv,tx);w=example_weights(tr.meta)
            clf=LogisticRegression(C=.3,max_iter=1500,solver='lbfgs',random_state=CFG.seed).fit(x,tr.meta.y,sample_weight=w)
            score=dev_score(dv,clf.predict_proba(xv)[:,1])
            rows.append({'seconds':sec,'status':'OK','dev_acquisition_balanced_accuracy':score,'n_train_windows':len(tr.meta),'n_dev_windows':len(dv.meta)})
            # Predeclared tie-break: nearer 20 s, then shorter duration.
            ranking=(score,-abs(float(sec)-20.),-float(sec))
            if best is None or ranking>best[0]:best=(ranking,float(sec))
        except Exception as e:
            rows.append({'seconds':sec,'status':'INFEASIBLE','error':str(e)});note('duration_infeasible',records[0].task,e,seconds=sec)
    table(rows,f'tables/{records[0].task}_DURATION_SELECTION.csv')
    if best is None:raise RuntimeError('No feasible duration; see explicit duration diagnostics')
    return best[1]


def train_tabular(train,dev,transform):
    x=flat_struct(train,transform);xv=flat_struct(dev,transform);y=train.meta.y.to_numpy();w=example_weights(train.meta)
    trees=48 if CFG.mode in {'smoke','selftest'} else CFG.tree_estimators
    grids={
        'Logistic':[LogisticRegression(C=c,max_iter=2000,random_state=CFG.seed) for c in (.03,.3,3.)],
        'RandomForest':[RandomForestClassifier(n_estimators=trees,min_samples_leaf=m,max_features='sqrt',n_jobs=CFG.cpu_threads,random_state=CFG.seed) for m in (1,3)],
        'ExtraTrees':[ExtraTreesClassifier(n_estimators=trees,min_samples_leaf=m,max_features=.7,n_jobs=CFG.cpu_threads,random_state=CFG.seed) for m in (1,3)],
    }
    fitted={};rows=[]
    for name,candidates in grids.items():
        best=None
        for k,clf in enumerate(candidates):
            clf.fit(x,y,sample_weight=w);score=dev_score(dev,clf.predict_proba(xv)[:,1])
            rows.append({'model':name,'candidate':k,'dev_balanced_accuracy':score,'params':str(clf.get_params())})
            if best is None or score>best[0]:best=(score,clf)
        fitted[name]=best[1]
    return fitted,rows


class HealthyDistance:
    def pooled(self,st,tr):
        x,_=tr.apply(st);m=st.valid[:,:,None]
        count=np.maximum(m.sum(1),1.)
        mu=(x*m).sum(1)/count;sd=np.sqrt(((x-mu[:,None,:])**2*m).sum(1)/count)
        return np.concatenate([mu,sd],axis=1)
    def fit(self,st,tr):
        self.transform=tr;x=self.pooled(st,tr)
        use=(st.meta.y.to_numpy()==0)&(st.meta.fault_type.to_numpy()=='nominal')
        if use.sum()<3:raise ValueError('Insufficient training-reference windows for healthy-only model')
        self.est=LedoitWolf().fit(x[use]);scores=self.score_array(x[use])
        self.center=float(np.quantile(scores,.95));self.scale=max(float(np.std(scores)),1.)
        return self
    def score_array(self,x):
        d=x-self.est.location_
        return np.sqrt(np.maximum(0,np.einsum('bi,ij,bj->b',d,self.est.precision_,d)))
    def raw_probability(self,st):
        return expit((self.score_array(self.pooled(st,self.transform))-self.center)/self.scale)

## 14 — Qualité instrumentale : disponibilité de la décision et scores hors groupe
La calibration et le test utilisent exactement le même score au niveau acquisition. Aucun seuil utile : seuil `None` et décision `UNDETERMINED`, jamais un faux négatif rassurant. Les métriques rapportent disponibilité, performances conditionnelles et fraction de corruptions effectivement signalées parmi toutes les requêtes.

La comparaison Route345 utilise deux réducteurs fixes : maximum, et quantile temporel empirique 0,90 **après maximum des axes d’un même capteur**. Elle ne modifie pas la fusion structurelle. La qualité hors groupe exclut chaque acquisition tenue à l’écart de l’apprentissage du détecteur ET des références/normalisations.


In [15]:
class QualityEstimator:
    def fit(self,st,tr):
        self.transform=tr;_,q=tr.apply(st);mask=st.present.astype(bool)
        x=q[mask];y=st.yf[mask].astype(int)
        if len(np.unique(y))<2:raise ValueError('Quality training needs nominal and injected-fault examples')
        base=example_weights(st.meta,balance=False)
        w=np.repeat(base[:,None]/np.maximum(st.present.sum(1)[:,None],1),st.present.shape[1],axis=1)[mask]
        for c in (0,1):w[y==c]/=max(w[y==c].sum(),1e-30)
        self.model=ExtraTreesClassifier(n_estimators=48 if CFG.mode in {'smoke','selftest'} else CFG.quality_tree_estimators,
            min_samples_leaf=3,max_features=.8,n_jobs=CFG.cpu_threads,random_state=CFG.seed)
        self.model.fit(x,y,sample_weight=w/w.mean())
        return self
    def predict(self,st):
        _,q=self.transform.apply(st);p=np.ones(st.present.shape,np.float32);mask=st.present.astype(bool)
        if mask.any():p[mask]=self.model.predict_proba(q[mask])[:,1]
        return p

def binary_quality_summary(y,pred,weights=None):
    """Rates concern injected labels, never certified real hardware faults."""
    y=np.asarray(y,int);pred=np.asarray(pred,bool)
    w=np.ones(len(y),float) if weights is None else np.asarray(weights,float)
    if len(y)!=len(w) or len(y)!=len(pred):raise ValueError('Incompatible binary metric inputs')
    if np.any(w<0) or not np.isfinite(w).all():raise ValueError('Invalid weights')
    pos=y==1;neg=~pos
    tp=float(w[pos&pred].sum());fp=float(w[neg&pred].sum());fn=float(w[pos&~pred].sum());tn=float(w[neg&~pred].sum())
    def div(a,b):return float(a/b) if b>0 else np.nan
    return {'n_observations':len(y),'positive_prevalence_weighted':div(tp+fn,w.sum()),
        'precision':div(tp,tp+fp),'recall':div(tp,tp+fn),'false_positive_rate':div(fp,fp+tn),
        'f1':div(2*tp,2*tp+fp+fn),'alert_fraction':div(tp+fp,w.sum()),
        'tp_weight':tp,'fp_weight':fp,'fn_weight':fn,'tn_weight':tn}

def quality_event_frames(st,qmodel,aggregation='max'):
    """Cal and test share this exact reducer. Axes are combined within physical sensors FIRST."""
    if aggregation not in {'max','physical_q90'}:raise ValueError('Unsupported predeclared aggregation')
    p=np.asarray(qmodel.predict(st),float);present=st.present.astype(bool)
    if p.shape!=present.shape or not np.isfinite(p[present]).all() or np.any((p[present]<0)|(p[present]>1)):
        raise ValueError('Invalid quality score array')
    physical=list(st.physical) if st.physical is not None else [f'channel_{j}' for j in range(p.shape[1])]
    if len(physical)!=p.shape[1]:raise ValueError('Physical mapping length mismatch')
    frames=[]
    for j in range(p.shape[1]):
        take=np.flatnonzero(present[:,j])
        if len(take)==0:continue
        fields=EVENT_KEYS+[c for c in ('parent_id','variant_id','start_sample','stop_sample','fs','seconds','fault_start','fault_stop','fault_sensor') if c in st.meta]
        d=st.meta.iloc[take][fields].copy().reset_index(drop=True)
        d['_row_index']=take;d['channel_index']=j;d['physical_sensor']=physical[j]
        d['window_id']=d['parent_id'].astype(str) if 'parent_id' in d else pd.Series(take.astype(str))
        d['quality_score']=p[take,j];d['injected_fault']=st.yf[take,j].astype(int);frames.append(d)
    if not frames:raise ValueError('No available measurement channels for quality evaluation')
    win=pd.concat(frames,ignore_index=True)
    reducer='max' if aggregation=='max' else (lambda x:empirical_upper_quantile(x,CFG.route345_temporal_quantile))
    ch=win.groupby(EVENT_KEYS+['channel_index','physical_sensor'],dropna=False,sort=False).agg(
        quality_score=('quality_score',reducer),injected_fault=('injected_fault','max'),n_windows=('window_id','nunique')).reset_index()
    # No treating x/y/z axes of one device as independent instruments.
    pw=win.groupby(EVENT_KEYS+['physical_sensor','window_id'],dropna=False,sort=False).agg(
        quality_score=('quality_score','max'),injected_fault=('injected_fault','max'),n_axes=('channel_index','nunique')).reset_index()
    phy=pw.groupby(EVENT_KEYS+['physical_sensor'],dropna=False,sort=False).agg(
        quality_score=('quality_score',reducer),injected_fault=('injected_fault','max'),n_windows=('window_id','nunique'),n_axes=('n_axes','max')).reset_index()
    ev=phy.groupby(EVENT_KEYS,dropna=False,sort=False).agg(
        quality_score=('quality_score','max'),injected_fault=('injected_fault','max'),n_physical_sensors=('physical_sensor','nunique'),
        minimum_windows_per_sensor=('n_windows','min'),maximum_windows_per_sensor=('n_windows','max')).reset_index()
    for df in (win,ch,phy,ev):
        df['event_id']=df.acquisition_id.astype(str)+'::'+df.fault_type.astype(str)+'::'+df.severity.astype(str)
        df['aggregation']=aggregation
    if ev.event_id.duplicated().any():raise ValueError('Conflicting quality event identity')
    ev['temporal_resolution_warning']=np.where(ev.minimum_windows_per_sensor<10,'FEWER_THAN_10_WINDOWS_QUANTILE_MAY_EQUAL_MAX','')
    return ev,phy,ch,win


def quality_acquisition_calibration(events,budget=None):
    """Empirical constrained calibration. No usable threshold -> None, not a reassuring negative."""
    if 'split' in events and set(events.split)!={'cal'}:raise ValueError('Quality calibration must use cal acquisitions only')
    budget=CFG.quality_alarm_budget if budget is None else float(budget)
    if not 0<=budget<1:raise ValueError('Budget must be in [0,1)')
    q=events.quality_score.to_numpy(float);nom=events.fault_type.to_numpy()=='nominal';fault=events.injected_fault.to_numpy()==1
    if not np.isfinite(q).all() or np.any(q<0) or np.any(q>1):raise ValueError('Quality scores must be finite and in [0,1]')
    if 'aggregation' in events and events.aggregation.nunique()!=1:raise ValueError('Mixed reducers in calibration')
    agg=str(events.aggregation.iloc[0]) if 'aggregation' in events and len(events) else 'max'
    ng=int(events.loc[nom,'group_id'].nunique());groups=sorted(events.group_id.unique())
    common={'aggregation':agg,'budget_nominal_acquisition':budget,'nominal_calibration_groups':ng,
        'empirical_resolution_one_nominal_alert':1/ng if ng else None,
        'population_false_alarm_guarantee':False,'nominal_truth':'no_added_corruption; not certified hardware health',
        'cal_groups':groups,'uses_test_labels':False,'warnings':['SMALL_N'] if ng<CFG.min_quality_cal_groups else [],
        'temporal_quantile':CFG.route345_temporal_quantile if agg=='physical_q90' else None}
    columns=['threshold','nominal_alert_fraction','injected_detection_fraction','meets_empirical_budget','within_score_range']
    if ng==0 or not fault.any():
        diag={**common,'status':'INSUFFICIENT_CALIBRATION_SUPPORT','threshold':None,'available':False,
              'achieved_nominal_alert_fraction':None,'achieved_injected_detection_fraction':None,
              'reason':'Nominal and injected calibration acquisitions are both required.'}
        return None,diag,pd.DataFrame(columns=columns)
    def weights(mask):
        w=np.zeros(len(events),float);g=events.group_id.to_numpy()
        for group in np.unique(g[mask]):
            ix=mask&(g==group);w[ix]=1/int(ix.sum())
        return w/w.sum()
    wn,wf=weights(nom),weights(fault)
    candidates=np.unique(np.r_[0.,q,np.nextafter(q.max(),np.inf)])
    rows=[]
    for t in candidates:
        a=q>=t;nr=float(wn@a);dr=float(wf@a)
        rows.append({'threshold':float(t),'nominal_alert_fraction':nr,'injected_detection_fraction':dr,
                     'meets_empirical_budget':bool(nr<=budget+1e-12),'within_score_range':bool(0<=t<=1)})
    curve=pd.DataFrame(rows);feasible=curve[curve.meets_empirical_budget]
    best=feasible.sort_values(['injected_detection_fraction','nominal_alert_fraction','threshold'],ascending=[False,True,False],kind='mergesort').iloc[0]
    available=bool(best.injected_detection_fraction>1e-12 and best.within_score_range)
    diagnostic_threshold=float(best.threshold)
    diag={**common,'status':'EMPIRICAL_THRESHOLD_AVAILABLE' if available else 'NO_USEFUL_SENSOR_ALARM_THRESHOLD',
        'threshold':diagnostic_threshold if available else None,'available':available,
        'diagnostic_candidate_threshold':diagnostic_threshold,
        'achieved_nominal_alert_fraction':float(best.nominal_alert_fraction),
        'achieved_injected_detection_fraction':float(best.injected_detection_fraction),
        'reason':'Detection >0 on cal under the nominal budget; NOT a guarantee of field usefulness.' if available else
                 'No admissible candidate detects added calibration corruptions. Instrumental decision UNDETERMINED.',
        'max_nominal_score':float(q[nom].max()),'max_injected_score':float(q[fault].max())}
    return diag['threshold'],diag,curve


def quality_local_calibration(windows):
    """Local F1 calibration is diagnostic and NEVER used for the acquisition sensor_check."""
    groups=windows.groupby('group_id').size().to_dict()
    w=np.array([1/groups[g] for g in windows.group_id]);y=windows.injected_fault.to_numpy(int);p=windows.quality_score.to_numpy()
    if len(np.unique(y))<2:return .5,{'status':'INSUFFICIENT_LOCAL_LABELS','threshold':.5}
    best=max((f1_score(y,p>=t,sample_weight=w,zero_division=0),-abs(t-.5),float(t)) for t in np.linspace(.1,.9,33))
    return best[2],{'status':'DIAGNOSTIC_ONLY','threshold':best[2],'cal_local_weighted_f1':float(best[0]),'decision_level':'channel_window'}

def quality_threshold_cal(st,qmodel):
    ev,phy,ch,win=quality_event_frames(st,qmodel,'max')
    th,diag,curve=quality_acquisition_calibration(ev)
    local,ld=quality_local_calibration(win);diag['local_diagnostic']=ld
    task=str(st.meta.task.iloc[0]);table(ev,f'predictions/{task}_QUALITY_CAL_ACQUISITIONS.csv')
    table(curve,f'tables/{task}_QUALITY_CALIBRATION_CURVE.csv')
    quality_origin_diagnostics(st,qmodel,ev,win,'CAL')
    if task=='Route345' and CFG.run_route345_aggregation_study:
        ae,_,_,_=quality_event_frames(st,qmodel,'physical_q90')
        at,ad,ac=quality_acquisition_calibration(ae)
        table(ae,f'predictions/{task}_QUALITY_CAL_Q90_ACQUISITIONS.csv');table(ac,f'tables/{task}_QUALITY_CALIBRATION_Q90_CURVE.csv')
        diag['alternative_aggregation']={**ad,'threshold':at,'calibration_only':True,'used_for_structural_fusion':False}
    if not diag['available']:note('instrument_threshold_unavailable',task,diag['status'],details=diag)
    return th,diag


def evaluate_quality_levels(st,qmodel,threshold,diag):
    ev,phy,ch,win=quality_event_frames(st,qmodel,'max');task=str(st.meta.task.iloc[0]);rows=[]
    ev=attach_instrument_decision(ev,threshold,diag);table(ev,f'predictions/{task}_QUALITY_TEST_ACQUISITIONS.csv')
    quality_origin_diagnostics(st,qmodel,ev,win,'TEST')
    for subset in ['all','nominal','seen_faults','unseen_fault','all_faults']:
        dd=ev[subset_mask(ev,subset)]
        if dd.empty:continue
        counts=dd.groupby('group_id').size().to_dict();w=np.array([1/counts[g] for g in dd.group_id])
        rows.append({'task':task,'level':'acquisition','subset':subset,'threshold':threshold,
            'status':diag['status'],'aggregation':'max',**quality_service_summary(dd,w)})
    local=float(diag.get('local_diagnostic',{}).get('threshold',.5))
    for level,df in [('physical_sensor_event',phy),('channel_event',ch),('channel_window',win)]:
        for subset in ['all','nominal','seen_faults','unseen_fault']:
            d=df[subset_mask(df,subset)]
            if d.empty:continue
            counts=d.groupby('group_id').size().to_dict();w=np.array([1/counts[g] for g in d.group_id])
            rows.append({'task':task,'level':level,'subset':subset,'threshold':local,'aggregation':'max',
                'threshold_role':'local_diagnostic_only_not_an_acquisition_service',
                'n_acquisitions':int(d.group_id.nunique()),**binary_quality_summary(d.injected_fault,d.quality_score>=local,w)})
    table(phy,f'predictions/{task}_QUALITY_TEST_PHYSICAL_SENSORS.csv');table(rows,f'tables/{task}_QUALITY_TEST_LEVELS.csv')
    if task=='Route345' and CFG.run_route345_aggregation_study:
        ad=diag['alternative_aggregation'];alt,_,_,_=quality_event_frames(st,qmodel,'physical_q90')
        alt=attach_instrument_decision(alt,ad['threshold'],ad)
        if set(alt.event_id)!=set(ev.event_id):raise ValueError('Aggregation study event sets differ')
        table(alt,f'predictions/{task}_QUALITY_TEST_Q90_ACQUISITIONS.csv')
        comparison=[]
        for agg,df,cal in [('max',ev,diag),('physical_q90',alt,ad)]:
            specifications=[('nominal',None,None),('seen_faults',None,None),('unseen_fault',None,None),('all_faults',None,None)]
            specifications += [('family_severity',ft,float(sev)) for ft,sev in df[df.fault_type!='nominal'].groupby(['fault_type','severity']).groups]
            for sub,ft,sev in specifications:
                d=df[(df.fault_type==ft)&(df.severity==sev)] if ft is not None else df[subset_mask(df,sub)]
                if d.empty:continue
                counts=d.groupby('group_id').size().to_dict();w=np.array([1/counts[g] for g in d.group_id])
                comparison.append({'task':task,'aggregation':agg,'subset':sub,'fault_type':ft,'severity':sev,'threshold':cal['threshold'],
                    'calibration_status':cal['status'],'nominal_calibration_groups':cal['nominal_calibration_groups'],
                    'quantile':CFG.route345_temporal_quantile if agg=='physical_q90' else None,'selected_using_test':False,
                    'minimum_windows_per_sensor':int(d.minimum_windows_per_sensor.min()),
                    **quality_service_summary(d,w)})
        table(comparison,f'tables/{task}_AGGREGATION_COMPARISON.csv')
    return {'acquisition_threshold':threshold,'availability_status':diag['status'],'local_threshold':local,'results':rows,
            'truth':'added synthetic corruption; nominal archive health unverified'}


def training_quality_crossfit(st):
    """Strict training-only group exclusion, including reference spectra and quality normalizers."""
    if set(st.meta.split)!={'train'}:raise ValueError('Crossfit accepts train acquisitions only')
    if CFG.heldout_fault in set(st.meta.fault_type):raise ValueError('Held-out fault entered training crossfit')
    labels=st.meta[['group_id','y']].drop_duplicates()
    if labels.group_id.duplicated().any():raise ValueError('Conflicting labels in an acquisition group')
    counts=labels.groupby('y').size().to_dict()
    k=min(CFG.crossfit_quality_folds,counts.get(0,0),counts.get(1,0))
    task=str(st.meta.task.iloc[0])
    if k<2:
        write_json({'status':'INSUFFICIENT_TRAINING_GROUPS','counts':counts,'no_fallback_to_cal_or_test':True},RUN/f'audit/{task}_QUALITY_CROSSFIT.json')
        raise ValueError('At least two training groups per class needed for strict quality crossfit; calibration/test cannot replace them')
    signature=stable_hash({'code':IMPLEMENTATION_HASH,'groups':st.meta[['group_id','y','variant_id']].to_dict('records'),
        'k':k,'sf':array_fingerprint(st.sf),'qb':array_fingerprint(st.qb),'aux':array_fingerprint(st.aux),
        'yf':array_fingerprint(st.yf),'present':array_fingerprint(st.present),'physical':st.physical,
        'quality_trees':48 if CFG.mode in {'smoke','selftest'} else CFG.quality_tree_estimators,'seed':CFG.seed})
    cache=RUN/'features'/f'{task}_TRAIN_QUALITY_OOF.npz';stamp=cache.with_suffix('.json')
    if CFG.resume and cache.is_file() and stamp.is_file():
        m=json.loads(stamp.read_text())
        if m['signature']!=signature:raise ValueError('Crossfit checkpoint changed: use a new study name')
        if file_hash(cache)!=m['sha256']:raise ValueError('Crossfit checkpoint corrupted')
        with np.load(cache,allow_pickle=False) as z:pred=z['quality'].copy()
        if pred.shape!=st.present.shape or not np.isfinite(pred).all():raise ValueError('Invalid crossfit cached array')
        print('  Scores qualité hors groupe vérifiés et réutilisés :',task,flush=True)
        return pred
    assignment={}
    for y in (0,1):
        groups=sorted(labels.loc[labels.y==y,'group_id']);rng=np.random.default_rng(CFG.seed+997*y);rng.shuffle(groups)
        for i,g in enumerate(groups):assignment[g]=i%k
    fold_ids=st.meta.group_id.map(assignment).to_numpy();pred=np.full(st.present.shape,np.nan,np.float32)
    covered=np.zeros(len(st.meta),int);ledger=[];rows=[]
    for fold in range(k):
        ti=np.flatnonzero(fold_ids!=fold);hi=np.flatnonzero(fold_ids==fold)
        train=slice_store(st,ti);held=slice_store(st,hi)
        fitgroups=set(train.meta.group_id);heldgroups=set(held.meta.group_id)
        if fitgroups&heldgroups:raise ValueError('Crossfit group overlap')
        ft=FeatureTransform().fit(train);qm=QualityEstimator().fit(train,ft)
        pred[hi]=qm.predict(held);covered[hi]+=1
        refgroups=sorted(train.meta.loc[(train.meta.y==0)&(train.meta.fault_type=='nominal'),'group_id'].unique())
        if not refgroups or heldgroups&set(refgroups):raise ValueError('Invalid crossfit healthy reference provenance')
        ledger.append({'crossfit_fold':fold+1,'fit_groups':sorted(fitgroups),'heldout_groups':sorted(heldgroups),
            'reference_fit_groups':refgroups,'fit_rows':len(ti),'heldout_rows':len(hi),'no_group_overlap':True,
            'transform_fit_groups':ft.train_groups})
        for group in sorted(heldgroups):
            rows.append({'task':task,'crossfit_fold':fold+1,'heldout_group':group,'n_fit_groups':len(fitgroups),
                'n_reference_fit_groups':len(refgroups),'fit_groups_json':json.dumps(sorted(fitgroups)),
                'reference_groups_json':json.dumps(refgroups),'heldout_excluded_from_model':True,
                'heldout_excluded_from_transform':group not in ft.train_groups,'cal_or_test_used':False})
    if not np.all(covered==1) or not np.isfinite(pred).all():raise RuntimeError('Incomplete quality OOF coverage')
    table(rows,f'tables/{task}_CROSSFIT_GROUP_AUDIT.csv')
    write_json({'status':'COMPLETE_TRAIN_ONLY','k':k,'ledger':ledger,'prediction_sha256':array_fingerprint(pred),
        'heldout_fault_absent':True,'each_training_variant_predicted_once':True,
        'quality_final_model_separately_fitted_on_all_train':True},RUN/f'audit/{task}_QUALITY_CROSSFIT.json')
    temp=cache.with_name(cache.name+'.tmp')
    with temp.open('wb') as f:np.savez_compressed(f,quality=pred)
    temp.replace(cache);write_json({'signature':signature,'sha256':file_hash(cache)},stamp)
    return pred

def empirical_upper_quantile(values, q):
    """Inverse empirical CDF, without invented intermediate scores or random tie breaking."""
    a=np.sort(np.asarray(values,float))
    if a.size==0 or not np.isfinite(a).all():raise ValueError('Finite nonempty quantile input required')
    if not 0<q<=1:raise ValueError('Quantile outside (0,1]')
    return float(a[max(0,min(len(a)-1,int(np.ceil(q*len(a)))-1))])

def attach_instrument_decision(df,threshold,diag,score_col='quality_score'):
    out=df.copy();score=out[score_col].to_numpy(float)
    available=bool(diag.get('available',threshold is not None)) and threshold is not None
    if available and not (np.isfinite(threshold) and 0<=threshold<=1):raise ValueError('An available alarm must have a threshold within [0,1]')
    out['instrument_decision_available']=available
    out['sensor_check']=pd.array(score>=threshold if available else [pd.NA]*len(out),dtype='boolean')
    out['instrument_decision']=np.where(score>=threshold,'CHECK','NO_CHECK') if available else 'UNDETERMINED'
    out['quality_calibration_status']=diag.get('status','EMPIRICAL_THRESHOLD_AVAILABLE' if available else 'INSUFFICIENT_CALIBRATION_SUPPORT')
    out['quality_calibration_warnings']=';'.join(diag.get('warnings',[]))
    out['instrument_threshold']=float(threshold) if available else np.nan
    out['alarm_aggregation']=diag.get('aggregation','max')
    return out

def quality_service_summary(df,weights=None):
    """Conditional performance + end-to-end detection: undefined decisions are never negatives or successes."""
    y=df.injected_fault.to_numpy(int);known=df.instrument_decision_available.to_numpy(bool)
    nullable=df.sensor_check.astype('boolean')
    if nullable.notna().to_numpy().tolist()!=known.tolist():raise ValueError('Availability and nullable decisions disagree')
    pred=nullable.fillna(False).to_numpy(bool)
    w=np.ones(len(df),float) if weights is None else np.asarray(weights,float)
    if len(w)!=len(df) or np.any(w<0) or not np.isfinite(w).all() or w.sum()<=0:raise ValueError('Invalid service weights')
    def ratio(a,b):return float(w[a].sum()/w[b].sum()) if w[b].sum()>0 else np.nan
    allmask=np.ones(len(df),bool);pos=y==1;neg=y==0
    result={'n_observations':len(df),'n_acquisitions':int(df.group_id.nunique()),
        'decision_availability_fraction':ratio(known,allmask),'undetermined_fraction':ratio(~known,allmask),
        'undetermined_fraction_corrupted':ratio(~known&pos,pos),'undetermined_fraction_nominal':ratio(~known&neg,neg),
        'positive_prevalence_weighted':ratio(pos,allmask),
        'service_detection_fraction_all_corrupted':ratio(known&pred&pos,pos),
        'issued_alert_fraction_all_requests':ratio(known&pred,allmask),
        'issued_alert_fraction_all_nominal':ratio(known&pred&neg,neg),
        'conditional_metric_population':'available decisions only; availability always reported separately'}
    if known.any():
        local=binary_quality_summary(y[known],pred[known],w[known])
        for key in ('precision','recall','false_positive_rate','f1','alert_fraction'):
            result[key+'_conditional_available']=local[key]
    else:
        for key in ('precision','recall','false_positive_rate','f1','alert_fraction'):result[key+'_conditional_available']=np.nan
    return result

def quality_origin_diagnostics(st,qmodel,ev,windows,partition):
    """Diagnostic provenance only. No test diagnostic enters learning, calibration or fusion."""
    task=str(st.meta.task.iloc[0]);qraw=qmodel.transform.quality_raw(st);_,qscaled=qmodel.transform.apply(st)
    top=(windows.sort_values(['event_id','quality_score','start_sample','channel_index'],ascending=[True,False,True,True],kind='mergesort')
          .groupby('event_id',sort=False).head(CFG.quality_diagnostic_top_windows).copy())
    top['score_origin_rank']=top.groupby('event_id').cumcount()+1
    protocol_path=RUN/f'audit/{task}_SELECTED_PROTOCOL.json'
    if protocol_path.is_file():
        names=json.loads(protocol_path.read_text())['panel']
        top['channel_name']=[names[j] for j in top.channel_index.to_numpy(int)]
    else:top['channel_name']=['channel_index_'+str(j) for j in top.channel_index]
    for feature in QBASE_NAMES+QREF_NAMES:
        # Exact feature names come from the retained V1.1 arrays, not assumed synonyms.
        names=QBASE_NAMES+QREF_NAMES
        if feature in names:
            k=names.index(feature);ii=top._row_index.to_numpy(int);jj=top.channel_index.to_numpy(int)
            top['raw_'+feature]=qraw[ii,jj,k];top['scaled_'+feature]=qscaled[ii,jj,k]
    # Always include drift by its actual source key.
    k=QBASE_NAMES.index('normalized_drift')
    ii=top._row_index.to_numpy(int);jj=top.channel_index.to_numpy(int)
    top['raw_normalized_drift']=qraw[ii,jj,k];top['scaled_normalized_drift']=qscaled[ii,jj,k]
    top['start_seconds']=top.start_sample/top.fs;top['stop_seconds']=top.stop_sample/top.fs
    top['real_sensor_failure_label']=False
    table(top.drop(columns=['_row_index']),f'tables/{task}_{partition}_QUALITY_SCORE_ORIGIN.csv')
    nominal=ev[ev.fault_type=='nominal'][['acquisition_id','quality_score']].rename(columns={'quality_score':'nominal_quality_score'})
    if nominal.acquisition_id.duplicated().any():raise ValueError('Duplicate nominal event for pairing')
    pairs=ev[ev.fault_type!='nominal'].merge(nominal,on='acquisition_id',validate='many_to_one')
    if len(pairs)!=int((ev.fault_type!='nominal').sum()):raise ValueError('Missing nominal counterpart')
    pairs['delta_quality_score']=pairs.quality_score-pairs.nominal_quality_score
    pairs['diagnostic_only_clean_counterpart_unavailable_in_deployment']=True
    table(pairs,f'tables/{task}_{partition}_QUALITY_NOMINAL_CORRUPTED_PAIRS.csv')
    # Paired window/axis sensitivity to the held-out drift (not fed back into the model).
    if partition=='TEST':
        clean=windows[windows.fault_type=='nominal'][['acquisition_id','window_id','channel_index','_row_index','quality_score']]
        drift=windows[(windows.fault_type==CFG.heldout_fault)&(windows.injected_fault==1)]
        joined=drift.merge(clean,on=['acquisition_id','window_id','channel_index'],suffixes=('','_nominal'),validate='many_to_one')
        if len(joined):
            i=joined._row_index.to_numpy(int);j=joined.channel_index.to_numpy(int);n=joined._row_index_nominal.to_numpy(int)
            joined['raw_drift_nominal']=qraw[n,j,k];joined['raw_drift_corrupted']=qraw[i,j,k]
            joined['scaled_drift_nominal']=qscaled[n,j,k];joined['scaled_drift_corrupted']=qscaled[i,j,k]
            joined['delta_window_quality']=joined.quality_score-joined.quality_score_nominal
            grouped=joined.groupby(['acquisition_id','state','severity'],dropna=False).agg(
                n_sensor_windows=('window_id','size'),median_nominal_drift=('raw_drift_nominal','median'),
                median_corrupted_drift=('raw_drift_corrupted','median'),median_scaled_nominal=('scaled_drift_nominal','median'),
                median_scaled_corrupted=('scaled_drift_corrupted','median'),median_delta_quality=('delta_window_quality','median'),
                fraction_score_increased=('delta_window_quality',lambda x:float(np.mean(np.asarray(x)>0)))).reset_index()
            grouped['diagnostic_only']=True;table(grouped,f'tables/{task}_DRIFT_WINDOW_DIAGNOSTICS.csv')

## 15 — Réseaux identiques sauf fusion de fiabilité
Réseau multicapteur léger, identité du canal, même capacité nominale pour StandardNN et FaultAware, trois graines. Le détecteur de qualité est figé avant le diagnostic structurel ; le gradient structurel ne peut pas modifier les labels de qualité. La branche non pondérée est conservée pour détecter des contradictions. GPU automatique, CPU de secours, checkpoints d'époque et meilleure époque de développement. Aucun early stopping sur le test.

In [16]:
class SensorNet(nn.Module):
    def __init__(self,n_sensors,n_features,fault_aware):
        super().__init__();self.fault_aware=fault_aware;h=CFG.hidden_dim;l=CFG.latent_dim
        self.encoder=nn.Sequential(nn.Linear(n_features,h),nn.LayerNorm(h),nn.GELU(),nn.Dropout(CFG.dropout),nn.Linear(h,l),nn.GELU())
        self.identity=nn.Embedding(n_sensors,l)
        self.head=nn.Sequential(nn.Linear(2*l,h),nn.GELU(),nn.Dropout(CFG.dropout),nn.Linear(h,1))
    @staticmethod
    def pool(z,weight):
        den=weight.sum(1,keepdim=True).clamp_min(1e-6)
        mu=(z*weight.unsqueeze(-1)).sum(1)/den
        var=((z-mu[:,None,:])**2*weight.unsqueeze(-1)).sum(1)/den
        return torch.cat([mu,torch.sqrt(var+1e-6)],dim=1)
    def forward(self,x,valid,q):
        ids=torch.arange(x.shape[1],device=x.device)
        z=self.encoder(x)+self.identity(ids)[None,:,:]
        plain=self.head(self.pool(z,valid)).squeeze(-1)
        reliability=(.05+.95*(1-q.detach())) if self.fault_aware else torch.ones_like(q)
        weighted=self.head(self.pool(z,valid*reliability)).squeeze(-1)
        return weighted,plain


def train_sensor_ensemble(train,dev,tr,qmodel,aware,train_quality=None,checkpoint_tag=None):
    x,_=tr.apply(train);xv,_=tr.apply(dev)
    q=qmodel.predict(train) if train_quality is None else np.asarray(train_quality,np.float32)
    if q.shape!=train.present.shape or not np.isfinite(q).all():raise ValueError("Invalid cross-fitted training quality scores")
    qv=qmodel.predict(dev)
    y=train.meta.y.to_numpy(np.float32);yv=dev.meta.y.to_numpy(np.float32)
    w=example_weights(train.meta).astype(np.float32);wv=example_weights(dev.meta,balance=False).astype(np.float32)
    train_ds=TensorDataset(*[torch.as_tensor(a,dtype=torch.float32) for a in (x,train.valid,q,y,w)])
    dev_ds=TensorDataset(*[torch.as_tensor(a,dtype=torch.float32) for a in (xv,dev.valid,qv,yv,wv)])
    epochs=min(3,CFG.max_epochs) if CFG.mode in {'smoke','selftest'} else CFG.max_epochs
    seeds=CFG.seeds[:1] if CFG.mode in {'smoke','selftest'} else CFG.seeds
    models=[];history_all=[]
    signature=stable_hash({'study':SIGNATURE,'train':train.meta.variant_id.tolist(),'dev':dev.meta.variant_id.tolist(),'aware':aware,
        'shape':list(x.shape),'transform':tr.train_groups,'epochs':epochs,
        'train_x_hash':array_fingerprint(x),'train_q_hash':array_fingerprint(q),
        'dev_x_hash':array_fingerprint(xv),'dev_q_hash':array_fingerprint(qv),
        'train_labels':array_fingerprint(y),'dev_labels':array_fingerprint(yv)})
    name=checkpoint_tag or ('FaultAware' if aware else 'StandardNN')
    for seed in seeds:
        checkpoint=RUN/'checkpoints'/f'{name}_seed_{seed}.pt'
        def train_attempt():
            seed_all(int(seed));model=SensorNet(x.shape[1],x.shape[2],aware).to(DEVICE)
            opt=torch.optim.AdamW(model.parameters(),lr=CFG.lr,weight_decay=CFG.weight_decay)
            best=None;best_loss=np.inf;wait=0;first_epoch=1;hist=[];finished=False
            if CFG.resume and checkpoint.exists():
                try:state=torch.load(checkpoint,map_location='cpu',weights_only=False)
                except TypeError:state=torch.load(checkpoint,map_location='cpu')
                if state['signature']!=signature:raise ValueError('Checkpoint incompatible with fold/configuration')
                model.load_state_dict(state['model']);opt.load_state_dict(state['optimizer'])
                for values in opt.state.values():
                    for k,v in values.items():
                        if torch.is_tensor(v):values[k]=v.to(DEVICE)
                best=state['best'];best_loss=state['best_loss'];wait=state['wait'];first_epoch=state['epoch']+1
                hist=state['history'];finished=state['finished']
                print(f'  reprise {name} graine {seed} : époque {first_epoch-1}, terminé={finished}',flush=True)
            dl=DataLoader(dev_ds,batch_size=CFG.batch_size,shuffle=False,num_workers=0)
            for epoch in ([] if finished else range(first_epoch,epochs+1)):
                # An epoch has its own deterministic RNG, making checkpoint-at-epoch continuation reproducible.
                seed_all(int(seed)+epoch*100003)
                tl=DataLoader(train_ds,batch_size=CFG.batch_size,shuffle=True,
                    generator=torch.Generator().manual_seed(int(seed)+epoch*100003),num_workers=0)
                model.train();total=0.;weight_sum=0.
                for xx,mm,qq,yy,ww in tl:
                    xx,mm,qq,yy,ww=[v.to(DEVICE) for v in (xx,mm,qq,yy,ww)]
                    pred,plain=model(xx,mm,qq);per=F.binary_cross_entropy_with_logits(pred,yy,reduction='none')
                    if aware:per=per+.25*F.binary_cross_entropy_with_logits(plain,yy,reduction='none')
                    loss=(per*ww).sum()/ww.sum().clamp_min(1e-6)
                    if not torch.isfinite(loss):raise FloatingPointError('Non-finite neural training loss')
                    opt.zero_grad(set_to_none=True);loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5.);opt.step()
                    total+=float((per*ww).sum().detach().cpu());weight_sum+=float(ww.sum().detach().cpu())
                model.eval();vs=0.;vw=0.
                with torch.no_grad():
                    for xx,mm,qq,yy,ww in dl:
                        xx,mm,qq,yy,ww=[v.to(DEVICE) for v in (xx,mm,qq,yy,ww)]
                        pred,_=model(xx,mm,qq);per=F.binary_cross_entropy_with_logits(pred,yy,reduction='none')
                        vs+=float((per*ww).sum().cpu());vw+=float(ww.sum().cpu())
                value=vs/max(vw,1e-6)
                if not np.isfinite(value):raise FloatingPointError('Non-finite development loss')
                hist.append({'seed':seed,'epoch':epoch,'train_loss':total/max(weight_sum,1e-6),'dev_loss':value,'fault_aware':aware,'device':str(DEVICE)})
                if value<best_loss-1e-5:
                    best_loss=value;wait=0;best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                else:wait+=1
                finished=wait>=CFG.patience or epoch>=epochs
                # Model/optimizer copied to CPU storage by torch.save; restore to target device on resume.
                atomic_torch({'signature':signature,'epoch':epoch,'model':model.state_dict(),'optimizer':opt.state_dict(),
                    'best':best,'best_loss':best_loss,'wait':wait,'history':hist,'finished':finished,'device':str(DEVICE)},checkpoint)
                if epoch==1 or epoch%5==0 or finished:
                    print(f'  {name} | graine={seed} | époque={epoch}/{epochs} | dev={value:.5f} | {DEVICE}',flush=True)
                if finished:break
            if best is None:raise RuntimeError('No valid neural checkpoint')
            model.load_state_dict(best);return model.cpu().eval(),hist
        try:model,hist=train_attempt()
        except RuntimeError as e:
            if str(DEVICE).startswith('cuda') and gpu_error(e):
                use_cpu(repr(e));model,hist=train_attempt()
            else:raise
        models.append(model);history_all.extend(hist)
        gc.collect()
        if str(DEVICE).startswith('cuda'):torch.cuda.empty_cache()
    return models,history_all


@torch.no_grad()
def predict_neural(models,st,tr,qmodel):
    x,_=tr.apply(st);q=qmodel.predict(st);pp=[];plain=[]
    for model in models:
        def predict_attempt():
            model.to(DEVICE).eval();a=[];b=[]
            for start in range(0,len(x),CFG.batch_size):
                stop=start+CFG.batch_size
                xx=torch.as_tensor(x[start:stop],device=DEVICE);mm=torch.as_tensor(st.valid[start:stop],device=DEVICE);qq=torch.as_tensor(q[start:stop],device=DEVICE)
                p0,p1=model(xx,mm,qq)
                a.extend(torch.sigmoid(p0).cpu().numpy());b.extend(torch.sigmoid(p1).cpu().numpy())
            return a,b
        try:a,b=predict_attempt()
        except RuntimeError as e:
            if str(DEVICE).startswith('cuda') and gpu_error(e):
                model.cpu();use_cpu(repr(e));a,b=predict_attempt()
            else:raise
        model.cpu();pp.append(a);plain.append(b)
    p=np.asarray(pp,dtype=float);pplain=np.asarray(plain,dtype=float)
    if p.ndim!=2 or not np.isfinite(p).all() or not np.isfinite(pplain).all():raise FloatingPointError('Invalid member predictions')
    return p,pplain,q


class MiniRocketBaseline:
    def fit(self,st):
        from sktime.transformations.panel.rocket import MiniRocketMultivariate
        if st.seq is None:raise ValueError('Temporal sequence cache not enabled')
        self.rocket=MiniRocketMultivariate(num_kernels=5000,random_state=CFG.seed,n_jobs=CFG.cpu_threads)
        features=np.asarray(self.rocket.fit_transform(st.seq))
        self.scaler=StandardScaler(with_mean=False).fit(features)
        self.ridge=RidgeClassifier(alpha=10.).fit(self.scaler.transform(features),st.meta.y,sample_weight=example_weights(st.meta))
        return self
    def raw_probability(self,st):
        f=np.asarray(self.rocket.transform(st.seq))
        return expit(self.ridge.decision_function(self.scaler.transform(f)))

## 16 — Calibration indépendante, ex æquo et couverture atteignable

L'affine logit n'est ajustée que lorsque le nombre total de groupes **et le nombre par classe** suffisent. Sinon : identité et seuil structurel 0,5 pré-déclaré.

Sélecteur principal commun : entropie de la probabilité. Variante commune aux deux réseaux : maximum(entropie, `2 × std des probabilités d'acquisition`). Le désaccord weighted/plain est une ablation distincte.

Aucun départage aléatoire des scores identiques : cible, couverture inférieure/supérieure atteignable, dépassement et limites de disponibilité sont enregistrés.

In [17]:
def calibration_weights(df):
    # Each acquisition has equal total mass. Within it, nominal and perturbed conditions each have half.
    w=np.zeros(len(df),float)
    for _,ids in df.groupby('group_id',sort=False).indices.items():
        ix=np.asarray(ids);nom=df.iloc[ix].fault_type.to_numpy()=='nominal'
        if nom.any() and (~nom).any():
            w[ix[nom]]=.5/nom.sum();w[ix[~nom]]=.5/(~nom).sum()
        else:w[ix]=1/len(ix)
    return w/max(w.sum(),1e-30)

@dataclass
class Calibrator:
    # Keep legacy attribute names for safe reading of trusted local V1.0 bundles.
    slope:float
    intercept:float
    structural_threshold:float
    quality_threshold:object
    selectors:dict
    model:str
    cal_groups:list
    coverage_note:str='Empirical target; ties remain intact; no population guarantee'
    probability_method:str='regularized_affine_logit'
    independent_calibration_groups:int=0
    structural_threshold_method:str='calibration_balanced_accuracy'
    selector:str='confidence'
    class_group_counts:dict=field(default_factory=dict)
    calibration_partition:str='cal'
    quality_available:bool=True
    quality_status:str='EMPIRICAL_THRESHOLD_AVAILABLE'
    quality_warnings:list=field(default_factory=list)
    quality_aggregation:str='max'
    def prob(self,p):return expit(self.slope*logit(np.clip(p,1e-6,1-1e-6))+self.intercept)

def uncertainty_components(df,p):
    p=np.asarray(p,float)
    if not np.isfinite(p).all() or np.min(p)<0 or np.max(p)>1:raise ValueError('Invalid probabilities for selection')
    clipped=np.clip(p,1e-15,1-1e-15)
    entropy=-(clipped*np.log(clipped)+(1-clipped)*np.log1p(-clipped))/np.log(2.)
    std=np.asarray(df.get('u_ensemble',np.zeros(len(df))),float)
    if not np.isfinite(std).all() or np.any(std< -1e-12) or np.any(std>.5+1e-7):
        raise ValueError('Acquisition member std is not within the probability bound [0, 0.5]')
    dis=np.asarray(df.get('disagreement',np.zeros(len(df))),float)
    if not np.isfinite(dis).all() or np.any(dis<0) or np.any(dis>1+1e-7):raise ValueError('Invalid acquisition disagreement')
    # Clipping below only removes numerical roundoff at mathematical bounds, not a multiplier saturation.
    return {'confidence_entropy':np.clip(entropy,0,1),'ensemble_dispersion':np.clip(2*std,0,1),
            'weighted_plain_disagreement':np.clip(dis,0,1)}

def uncertainty_score(df,p,aware=False,selector=None):
    # Backward compatible positional argument for software fixtures, not legacy behavior.
    selector=('confidence_ensemble' if aware else 'confidence') if selector is None else selector
    c=uncertainty_components(df,p)
    if selector=='confidence':return c['confidence_entropy']
    if selector=='confidence_ensemble':return np.maximum(c['confidence_entropy'],c['ensemble_dispersion'])
    if selector=='confidence_ensemble_disagreement':
        return np.maximum(np.maximum(c['confidence_entropy'],c['ensemble_dispersion']),c['weighted_plain_disagreement'])
    raise ValueError('Unknown selector: '+str(selector))

def coverage_selector(u,hard,weights,y,target):
    """Smallest deterministic threshold reaching the target; tie overshoot is explicitly recorded."""
    u=np.asarray(u,float);hard=np.asarray(hard,bool);w=np.asarray(weights,float);y=np.asarray(y,int)
    if not (len(u)==len(hard)==len(w)==len(y)) or not np.isfinite(u).all():raise ValueError('Invalid selector inputs')
    if np.any(w<0) or w.sum()<=0:raise ValueError('Invalid coverage weights')
    w=w/w.sum()
    possible=np.unique(u[hard]);values=np.array([float(w@(hard&(u<=t))) for t in possible])
    lower=np.flatnonzero(values<target-1e-12);upper=np.flatnonzero(values>=target-1e-12)
    below=float(values[lower[-1]]) if len(lower) else 0.
    above=float(values[upper[0]]) if len(upper) else None
    if len(upper):th=float(possible[upper[0]])
    elif len(possible):th=float(possible[-1])
    else:th=-1.
    accepted=hard&(u<=th);cov=float(w@accepted)
    if not len(possible):status='NO_PHYSICALLY_ELIGIBLE_CALIBRATION_EVENTS'
    elif not len(upper):status='LIMITED_BY_SENSOR_AVAILABILITY'
    elif abs(cov-target)<=1e-12:status='TARGET_EXACTLY_ATTAINED'
    else:status='TARGET_REACHED_WITH_OVERSHOOT'
    byclass={}
    for label,name in [(0,'reference'),(1,'altered')]:
        use=y==label;byclass['cal_coverage_'+name]=float(w[use]@accepted[use]/w[use].sum()) if w[use].sum()>0 else None
    return {'threshold':th,'achieved_cal_coverage':cov,'feasible':bool(cov>=target-1e-12),
            'requested_coverage':float(target),'status':status,'coverage_below_target':below,'coverage_above_target':above,
            'target_exactly_attainable':bool(abs(cov-target)<=1e-12),'overshoot':max(0.,cov-target),
            'max_cal_attainable_coverage':float(w@hard),'unique_scores':int(len(possible)),
            'max_tie_block_events':int(max(Counter(u[hard]).values(),default=0)),**byclass}

def fit_calibrator(df,model,quality_threshold=.5,selector='confidence',probability_from=None,quality_diagnostic=None):
    if 'split' in df and set(df.split)!={'cal'}:raise ValueError('Calibration can only consume the cal partition')
    if df.y.nunique()!=2:raise ValueError('Calibration needs both structural classes')
    groups=df[['group_id','y']].drop_duplicates()
    if groups.group_id.duplicated().any():raise ValueError('Conflicting group labels')
    counts={int(k):int(v) for k,v in groups.groupby('y').size().items()};ng=len(groups)
    w=calibration_weights(df);y=df.y.to_numpy(int);z=logit(np.clip(df.p.to_numpy(),1e-6,1-1e-6))
    small=(ng<CFG.min_calibration_groups_for_affine or min(counts.values())<CFG.min_calibration_groups_per_class)
    if probability_from is not None:
        a,b,threshold=probability_from.slope,probability_from.intercept,probability_from.structural_threshold
        method=probability_from.probability_method;tmethod=probability_from.structural_threshold_method
    elif small:
        a,b,threshold=1.,0.,.5;method='IDENTITY_SMALL_INDEPENDENT_N';tmethod='PREDECLARED_0.5_SMALL_N'
        note('small_calibration',model,f'Groups by class {counts}: no flexible probability refit; fixed structural threshold 0.5')
    else:
        def nll(ab):
            zz=ab[0]*z+ab[1]
            return float(np.sum(w*(np.logaddexp(0,zz)-y*zz))+.01*((ab[0]-1)**2+ab[1]**2))
        opt=optimize.minimize(nll,[1.,0.],bounds=[(.05,10.),(-8.,8.)],method='L-BFGS-B')
        if not opt.success or not np.isfinite(opt.fun):raise RuntimeError('Probability calibration failed: '+str(opt.message))
        a,b=map(float,opt.x);method='REGULARIZED_AFFINE_LOGIT';p=expit(a*z+b)
        candidates=np.unique(np.r_[.5,np.linspace(.05,.95,91)]);scores=[]
        for t in candidates:
            pred=p>=t
            score=.5*(np.sum(w[(y==0)&(~pred)])/max(w[y==0].sum(),1e-30)+np.sum(w[(y==1)&pred])/max(w[y==1].sum(),1e-30))
            scores.append((score,-abs(t-.5),float(t)))
        threshold=max(scores)[2];tmethod='CALIBRATION_BALANCED_ACCURACY'
    p=expit(a*z+b);u=uncertainty_score(df,p,selector=selector)
    hard=df.usable_physical_sensors.to_numpy()>=CFG.min_physical_sensors
    selectors={str(float(t)):coverage_selector(u,hard,w,y,float(t)) for t in CFG.target_coverages}
    qd=quality_diagnostic or {'available':quality_threshold is not None,'status':'EMPIRICAL_THRESHOLD_AVAILABLE' if quality_threshold is not None else 'INSUFFICIENT_CALIBRATION_SUPPORT'}
    return Calibrator(a,b,float(threshold),None if quality_threshold is None else float(quality_threshold),selectors,model,sorted(df.group_id.unique()),
        probability_method=method,independent_calibration_groups=ng,structural_threshold_method=tmethod,
        selector=selector,class_group_counts=counts,quality_available=bool(qd['available']),quality_status=qd['status'],
        quality_warnings=qd.get('warnings',[]),quality_aggregation=qd.get('aggregation','max'))

def decide(df,cal,coverage=None):
    out=df.copy();out['p_cal']=cal.prob(out.p.to_numpy());parts=uncertainty_components(out,out.p_cal.to_numpy())
    for key,value in parts.items():out[key]=value
    out['uncertainty']=uncertainty_score(out,out.p_cal.to_numpy(),selector=cal.selector)
    target=CFG.primary_coverage if coverage is None else float(coverage)
    sel=cal.selectors[str(float(target))];th=sel['threshold']
    hard=out.usable_physical_sensors.to_numpy()>=CFG.min_physical_sensors
    accepted=hard&(out.uncertainty.to_numpy()<=th)
    pred=(out.p_cal.to_numpy()>=cal.structural_threshold).astype(int)
    out['accepted']=accepted;out['binary_prediction']=pred;out['physical_gate_passed']=hard
    out['structural_decision']=np.where(~accepted,'ABSTAIN',np.where(pred==1,'ALERT','NO_ALERT'))
    out=attach_instrument_decision(out,cal.quality_threshold,{'available':cal.quality_available,'status':cal.quality_status,
        'warnings':cal.quality_warnings,'aggregation':cal.quality_aggregation},score_col='qmax')
    out['coverage_target']=target;out['selector_rule']=cal.selector
    out['selection_status_calibration']=sel['status'];out['coverage_achieved_calibration']=sel['achieved_cal_coverage']
    out['uncertainty_threshold']=th;out['instrument_threshold']=np.nan if cal.quality_threshold is None else cal.quality_threshold
    out['calibration_nominal_class_groups']=cal.class_group_counts.get(0,0)
    return out

## 17 — Métriques par classe, points de couverture exacts et bootstrap

Couverture référence et couverture état modifié sont prioritaires. Le risque équilibré parmi les décisions acceptées reste indéfini si une classe n'est plus représentée.

Les courbes sont constituées de points réellement atteignables. L'intégrale conventionnelle sur blocs d'ex æquo et la couverture maximale sont indiquées ensemble. Le bootstrap est conditionnel aux campagnes observées et aux prédicteurs, avec sensibilité par état ; les plis n'ont pas des ensembles d'apprentissage indépendants.

In [18]:
def metric_dict(d):
    y=d.y.to_numpy(int);p=d.p_cal.to_numpy(float);a=d.accepted.to_numpy(bool);pred=d.binary_prediction.to_numpy(int)
    def rate(num,den):return float(np.sum(num)/np.sum(den)) if np.sum(den)>0 else np.nan
    pos=y==1;neg=y==0;both=len(np.unique(y))==2;accepted_both=a.any() and len(np.unique(y[a]))==2
    return {'n_events':len(d),'n_acquisitions':d.group_id.nunique(),'coverage':float(a.mean()),
        'coverage_reference':rate(a&neg,neg),'coverage_altered':rate(a&pos,pos),
        'selective_risk':rate(a&(pred!=y),a),'false_structural_alarm_rate':rate(a&(pred==1)&neg,neg),
        'false_no_alert_rate':rate(a&(pred==0)&pos,pos),'structural_detection_rate':rate(a&(pred==1)&pos,pos),
        'abstention_altered':rate((~a)&pos,pos),'abstention_reference':rate((~a)&neg,neg),
        'accuracy_forced_all':float(np.mean(pred==y)),
        'balanced_accuracy_forced_all':float(balanced_accuracy_score(y,pred)) if both else np.nan,
        'f1_forced_all':float(f1_score(y,pred,zero_division=0)) if both else np.nan,
        'f1_accepted':float(f1_score(y[a],pred[a],zero_division=0)) if accepted_both else np.nan,
        'balanced_accuracy_accepted':float(balanced_accuracy_score(y[a],pred[a])) if accepted_both else np.nan,
        'balanced_selective_error':1-float(balanced_accuracy_score(y[a],pred[a])) if accepted_both else np.nan,
        'accepted_both_classes':bool(accepted_both),
        'n_reference_events':int(neg.sum()),'n_altered_events':int(pos.sum()),
        'n_accepted_reference':int((a&neg).sum()),'n_accepted_altered':int((a&pos).sum()),
        'n_unique_reference_groups':int(d.loc[neg,'group_id'].nunique()),'n_unique_altered_groups':int(d.loc[pos,'group_id'].nunique()),
        'auroc_all':float(roc_auc_score(y,p)) if both else np.nan,
        'average_precision_all':float(average_precision_score(y,p)) if both else np.nan,
        'brier_all':float(np.mean((p-y)**2)),
        'sensor_check_fraction':float(d.sensor_check.mean()) if d.sensor_check.notna().any() else np.nan,
        'instrument_decision_availability':float(d.get('instrument_decision_available',d.sensor_check.notna()).mean()),
        'instrument_undetermined_fraction':float(d.sensor_check.isna().mean())}


def risk_coverage_exact(d):
    hard=d.usable_physical_sensors.to_numpy()>=CFG.min_physical_sensors
    order=np.argsort(d.uncertainty.to_numpy()[hard],kind='stable')
    scores=d.uncertainty.to_numpy()[hard][order]
    err=(d.binary_prediction.to_numpy()[hard][order]!=d.y.to_numpy()[hard][order]).astype(float)
    if len(order)==0:return pd.DataFrame(),np.nan,0.
    endpoints=np.r_[np.flatnonzero(np.diff(scores)!=0)+1,len(scores)]
    ce=np.cumsum(err);risk=ce[endpoints-1]/endpoints;coverage=endpoints/len(d)
    reference=(d.y.to_numpy()[hard][order]==0);altered=~reference
    cr=np.cumsum(reference);ca=np.cumsum(altered)
    nr=max(int((d.y==0).sum()),1);na=max(int((d.y==1).sum()),1)
    curve=pd.DataFrame({'coverage':coverage,'risk':risk,'uncertainty_threshold':scores[endpoints-1],
        'accepted_events':endpoints,'tie_block_size':np.diff(np.r_[0,endpoints]),
        'coverage_reference':cr[endpoints-1]/nr if (d.y==0).any() else np.nan,
        'coverage_altered':ca[endpoints-1]/na if (d.y==1).any() else np.nan,
        'attainable_point':True})
    # Conventional sum over complete tie blocks (risk after each accepted block).
    # Intermediate coverages within a tie are NOT attainable by a deterministic threshold.
    aurc=float(np.sum(np.diff(np.r_[0.,coverage])*risk))
    return curve,aurc,float(coverage[-1])


def paired_bootstrap(a,b,metric,replicates,seed):
    """Paired cluster bootstrap of rates via sufficient counts, avoiding repeated DataFrame/ROC work."""
    if set(a.event_id)!=set(b.event_id):raise ValueError('Models were not evaluated on identical events')
    b=b.set_index('event_id').loc[a.event_id].reset_index()
    a=a.reset_index(drop=True)
    if not np.array_equal(a.group_id.to_numpy(),b.group_id.to_numpy()):raise ValueError('Pairing does not preserve acquisition groups')
    groups=sorted(a.group_id.unique());point=metric_dict(a)[metric]-metric_dict(b)[metric]
    if len(groups)<2:return {'difference_A_minus_B':point,'ci95_low':None,'ci95_high':None,'n_clusters':len(groups)}
    def counts(df):
        y=df.y.to_numpy(int);pred=df.binary_prediction.to_numpy(int);ok=df.accepted.to_numpy(bool)
        if metric=='coverage':num=ok;den=np.ones(len(df),bool)
        elif metric=='selective_risk':num=ok&(pred!=y);den=ok
        elif metric=='false_structural_alarm_rate':num=ok&(pred==1)&(y==0);den=y==0
        elif metric=='false_no_alert_rate':num=ok&(pred==0)&(y==1);den=y==1
        else:raise ValueError('This fast bootstrap supports predeclared rate metrics only: '+metric)
        z=pd.DataFrame({'group':df.group_id,'num':num.astype(float),'den':den.astype(float)})
        return z.groupby('group')[['num','den']].sum().reindex(groups).to_numpy()
    ca,cb=counts(a),counts(b);rng=np.random.default_rng(seed)
    multiplicities=rng.multinomial(len(groups),np.full(len(groups),1/len(groups)),size=replicates)
    ta=multiplicities@ca;tb=multiplicities@cb
    valid=(ta[:,1]>0)&(tb[:,1]>0)
    values=ta[valid,0]/ta[valid,1]-tb[valid,0]/tb[valid,1]
    return {'difference_A_minus_B':point,'ci95_low':float(np.quantile(values,.025)) if len(values) else None,
        'ci95_high':float(np.quantile(values,.975)) if len(values) else None,'n_clusters':len(groups),'finite_replicates':len(values),
        'interpretation':'paired acquisition bootstrap, conditional on campaign and fitted models; not independent-state generalization'}

## 18 — Référence V1.1 figée et inférence à l’échelle acquisition
Les poids locaux de votre propre V1.1 peuvent être réutilisés après vérification des empreintes, versions, tableaux de partitions, canaux et transformations. Aucun fichier provenant d’un tiers ne doit être chargé par `joblib`.

`FaultAware_Control` désigne le modèle avec scores qualité in-sample (poids V1.1 s’ils sont compatibles). `FaultAware` désigne la nouvelle variante hors groupe. Le rapport distingue explicitement réutilisation vérifiée et réentraînement.


In [19]:
def raw_predictions(name,model,st,tr,qmodel):
    q=qmodel.predict(st);qmax=np.where(st.present>0,q,0.).max(1)
    if name in {'StandardNN','FaultAware','FaultAware_Control'}:
        p,plain,q=predict_neural(model,st,tr,qmodel)
        return aggregate_member_predictions(st.meta,p,plain,qmax)
    if name in {'HealthyMahalanobis','MiniRocket'}:p=model.raw_probability(st)
    else:p=model.predict_proba(flat_struct(st,tr))[:,1]
    if not np.isfinite(p).all():raise FloatingPointError(f'Non-finite predictions from {name}')
    result=aggregate_predictions(st.meta,p,qmax=qmax)
    result['u_ensemble']=0.;result['disagreement']=0.;result['n_ensemble_members']=0
    return result

REFERENCE_TRAINING_FIELDS=(
    'seed','z24_reference_channels','z24_exclude_installation_state2','route345_axes','route345_excluded_sensors',
    'openlab_excitation_primary','openlab_vehicle_primary','openlab_allow_postload_as_change','min_physical_sensors',
    'train_channel_coverage','window_seconds','train_overlap','eval_overlap','band_hz','psd_bins','welch_segment_seconds',
    'max_missing_fraction','max_interpolation_gap_seconds','timestamp_relative_tolerance','outer_folds',
    'fault_types','heldout_fault','fault_severities','train_fault_repeats','eval_fault_repeats','max_faulty_physical_sensors',
    'seeds','max_epochs','patience','batch_size','lr','weight_decay','latent_dim','hidden_dim','dropout',
    'tree_estimators','quality_tree_estimators','enable_minirocket')

def validate_reference_records(old,records):
    """Compare native identities, labels, partitions, and full waveform hashes before any unpickling."""
    if old.acquisition_id.duplicated().any():raise ValueError('Duplicate legacy acquisition IDs')
    lookup={r.acquisition_id:r for r in records}
    if set(old.acquisition_id)!=set(lookup):raise ValueError('Legacy/current acquisition sets differ')
    mismatches=[]
    for _,row in old.iterrows():
        r=lookup[row.acquisition_id]
        for key in ('group_id','state','split','sha256','waveform_hash'):
            if str(row[key])!=str(getattr(r,key)):mismatches.append((r.acquisition_id,key))
        if int(row.y)!=int(r.y) or not np.isclose(float(row.fs),r.fs) or int(row.n_samples)!=int(r.n_samples):
            mismatches.append((r.acquisition_id,'label/rate/length'))
        for key in ('channels','sensor_ids','axes','units'):
            if json.loads(row[key])!=list(getattr(r,key)):mismatches.append((r.acquisition_id,key))
        nk='' if pd.isna(row.native_key) else str(row.native_key)
        if nk!=str(r.native_key or ''):mismatches.append((r.acquisition_id,'native_key'))
    if mismatches:raise ValueError('Legacy/current data or split mismatch: '+str(mismatches[:10]))
    return True

def reference_candidate(records,task,panel,physical,axes):
    """Returns metadata only; no model is deserialized until provenance checks pass."""
    if CFG.reference_policy=='never':return None
    # Synthetic mode never silently loads real-project models.
    if CFG.mode=='selftest' and not CFG.reference_run_root:return None
    fold_name=RUN.name
    folder=REFERENCE_RUN/'folds'/task/fold_name
    modeldir=folder/'models'/task
    need=[modeldir/'tabular_and_calibration.joblib',modeldir/'StandardNN.pt',modeldir/'FaultAware.pt',
          modeldir/'PRE_TEST_FREEZE.json',folder/'audit/LOCKED_SPLIT.csv',folder/f'audit/{task}_SELECTED_PROTOCOL.json',
          REFERENCE_RUN/'STUDY_LOCK.json',REFERENCE_RUN/'environment.json']
    missing=[str(p) for p in need if not p.is_file()]
    if missing:
        if CFG.reference_policy=='require_compatible':raise FileNotFoundError('Required trusted V1.1 bundle missing: '+str(missing[:4]))
        note('reference_not_available',task,'Refitting required; no V1.1 weights loaded',missing=missing[:4])
        return None
    if not CFG.trust_own_local_reference_models:raise ValueError('Local model deserialization disabled: choose reference_policy=never or explicitly trust your own output')
    lock=json.loads((REFERENCE_RUN/'STUDY_LOCK.json').read_text())
    if lock.get('implementation')!=REFERENCE_IMPLEMENTATION_HASH:raise ValueError('Unsupported legacy implementation fingerprint')
    oc=lock['config'];nc=asdict(CFG)
    mismatch=[k for k in REFERENCE_TRAINING_FIELDS if json_safe(oc.get(k))!=json_safe(nc.get(k))]
    if mismatch:raise ValueError('V1.1 predictors incompatible with requested training protocol: '+str(mismatch)+'. Use a new study with reference_policy=never for deliberate refitting.')
    oenv=json.loads((REFERENCE_RUN/'environment.json').read_text())
    # Pickled sklearn objects are version-sensitive. Do not silently cross incompatible environments.
    for package,current in [('sklearn',sklearn.__version__),('numpy',np.__version__)]:
        prior=str(oenv.get('packages',{}).get(package,''))
        if package=='sklearn' and prior!=current:raise ValueError(f'Legacy sklearn {prior} != kernel {current}; restore original environment or set reference_policy=never in a new study')
        if package=='numpy' and prior.split('.')[0]!=current.split('.')[0]:raise ValueError('Legacy NumPy major version differs; explicit refit required')
    old=pd.read_csv(folder/'audit/LOCKED_SPLIT.csv');validate_reference_records(old,records)
    protocol=json.loads((folder/f'audit/{task}_SELECTED_PROTOCOL.json').read_text())
    if protocol['panel']!=panel or protocol['physical_sensor_ids']!=physical or protocol['axes']!=axes:
        raise ValueError('Legacy panel or physical identity differs')
    if tuple(protocol['band_hz'])!=tuple(CFG.band_hz) or float(protocol['seconds']) not in CFG.window_seconds:
        raise ValueError('Legacy spectral/window settings differ')
    freeze=json.loads((modeldir/'PRE_TEST_FREEZE.json').read_text());hashes={}
    for p in need[:3]:
        rel=str(p.relative_to(folder));expected=freeze.get('model_sha256',{}).get(rel)
        if expected is None or file_hash(p)!=expected:raise ValueError('Unverified or changed legacy model: '+str(p))
        hashes[rel]=expected
    for split in ('train','dev','cal'):
        actual=sorted({r.group_id for r in records if r.split==split})
        if freeze.get(split+'_groups')!=actual:raise ValueError('Legacy freeze split mismatch: '+split)
    return {'folder':folder,'modeldir':modeldir,'protocol':protocol,'hashes':hashes,
            'reference_config':oc,'reference_freeze_sha256':file_hash(modeldir/'PRE_TEST_FREEZE.json')}

def deserialize_trusted_legacy(candidate,train,task):
    # joblib / pickle may execute code. This path is restricted to explicitly trusted local project outputs,
    # validated against the saved own-run model hashes above. Never pass an untrusted downloaded path.
    bundle=joblib.load(candidate['modeldir']/'tabular_and_calibration.joblib')
    if bundle.get('version')!=REFERENCE_VERSION:raise ValueError('Unexpected model bundle version')
    tr=bundle['transform'];qm=bundle['quality_model']
    if sorted(tr.train_groups)!=sorted(train.meta.group_id.unique()):raise ValueError('Legacy transform training groups differ')
    # Fit only a check transform on current TRAIN inputs to verify the transformation actually matches.
    chk=FeatureTransform().fit(train)
    for name in ('reference','healthy_spectrum','sm','ss','qm','qs'):
        if not np.allclose(getattr(chk,name),getattr(tr,name),equal_nan=True,rtol=1e-6,atol=1e-7):
            raise ValueError('Legacy transform values differ: '+name)
    models=dict(bundle['tabular']);nfeat=tr.apply(train)[0].shape[-1]
    for name,aware in [('StandardNN',False),('FaultAware',True)]:
        p=candidate['modeldir']/f'{name}.pt'
        try:obj=torch.load(p,map_location='cpu',weights_only=True)
        except TypeError:obj=torch.load(p,map_location='cpu') # trusted own-output path, older PyTorch only
        if obj.get('version')!=REFERENCE_VERSION or obj['panel']!=candidate['protocol']['panel'] or obj['input_feature_dim']!=nfeat:
            raise ValueError('Legacy neural architecture/panel incompatible')
        nnmodels=[]
        for state in obj['states']:
            model=SensorNet(len(obj['panel']),nfeat,aware);model.load_state_dict(state,strict=True)
            nnmodels.append(model.cpu().eval())
        expected_n=1 if CFG.mode in {'smoke','selftest'} else len(CFG.seeds)
        if len(nnmodels)!=expected_n:raise ValueError('Legacy ensemble seed count differs from requested execution budget')
        models[name]=nnmodels
    required={'Logistic','RandomForest','ExtraTrees','HealthyMahalanobis','StandardNN','FaultAware'}
    if not required.issubset(models):raise ValueError('Incomplete legacy comparator bundle')
    return tr,qm,models


def save_bundle(task,panel,tr,qmodel,models,calibrators,cal_hash):
    dest=RUN/'models'/task;dest.mkdir(parents=True,exist_ok=True)
    atomic_joblib({'version':VERSION,'transform':tr,'quality_model':qmodel,
                 'tabular':{k:v for k,v in models.items() if k not in {'StandardNN','FaultAware','FaultAware_Control'}},
                 'calibrators':calibrators,'panel':panel,'calibration_hash':cal_hash},dest/'tabular_and_calibration.joblib')
    for name in ('StandardNN','FaultAware','FaultAware_Control'):
        atomic_torch({'version':VERSION,'states':[m.state_dict() for m in models[name]],'panel':panel,
                    'input_feature_dim':models[name][0].encoder[0].in_features,'n_channels':len(panel),'features':SF_NAMES+[f'reference_delta_shape_{i}' for i in range(CFG.psd_bins)],'quality_features':QBASE_NAMES+QREF_NAMES,'config':asdict(CFG)},dest/f'{name}.pt')
    write_json({k:asdict(v) for k,v in calibrators.items()},dest/'calibrators.json')


def calibration_fingerprint(cals):return stable_hash({k:asdict(v) for k,v in cals.items()})


def record_model_results(task,name,pred,cal):
    decisions={}
    for coverage in CFG.target_coverages:
        d=decide(pred,cal,float(coverage));decisions[float(coverage)]=d
        if float(coverage)==float(CFG.primary_coverage):table(d,f'predictions/{task}_{name}_TEST.csv')
        for (kind,sev),g in d.groupby(['fault_type','severity'],dropna=False):
            m=metric_dict(g);curve,aurc,attainable=risk_coverage_exact(g)
            METRICS.append({'task':task,'model':name,'fault_type':kind,'severity':sev,'coverage_target':coverage,
                            'calibration_coverage_feasible':cal.selectors[str(float(coverage))]['feasible'],
                            'selection_status':cal.selectors[str(float(coverage))]['status'],
                            'coverage_achieved_calibration':cal.selectors[str(float(coverage))]['achieved_cal_coverage'],
                            'selector_rule':cal.selector,
                            'aurc_attainable':aurc,'max_attainable_coverage':attainable,**m})
        if float(coverage)==float(CFG.primary_coverage):
            for kind in ('nominal','ALL_FAULTS'):
                sub=d[d.fault_type=='nominal'] if kind=='nominal' else d[d.fault_type!='nominal']
                if len(sub):
                    curve,_,_=risk_coverage_exact(sub)
                    table(curve,f'tables/{task}_{name}_{kind}_RISK_COVERAGE.csv')
    return decisions

def resolve_frozen_window(records,task,panel,physical,axes):
    """V1.1 windows and partitions are fixed; absence of old models never triggers a new duration search."""
    folder=REFERENCE_RUN/'folds'/task/RUN.name
    path=folder/f'audit/{task}_SELECTED_PROTOCOL.json';splitfile=folder/'audit/LOCKED_SPLIT.csv'
    if path.is_file():
        if not splitfile.is_file():raise ValueError('Parent protocol exists without its partition provenance')
        validate_reference_records(pd.read_csv(splitfile),records)
        obj=json.loads(path.read_text())
        if obj['panel']!=panel or obj['physical_sensor_ids']!=physical or obj['axes']!=axes:raise ValueError('Parent physical panel differs')
        if tuple(obj['band_hz'])!=tuple(CFG.band_hz):raise ValueError('Parent analysis band differs')
        seconds=float(obj['seconds'])
        if not np.isfinite(seconds) or seconds<=0:raise ValueError('Invalid parent window duration')
        status='FROZEN_FROM_VERIFIED_V1_1_PROTOCOL';source=str(path)
    else:
        if CFG.require_reference_protocol:raise FileNotFoundError('V1.1 protocol required: '+str(path))
        seconds=float(CFG.frozen_window_seconds)
        status='PREDECLARED_20_SECONDS_NO_LOCAL_PARENT_PROTOCOL';source=None
        note('frozen_duration_no_parent_protocol',task,status,seconds=seconds)
    write_json({'status':status,'seconds':seconds,'source':source,'source_sha256':file_hash(path) if path.is_file() else None,
        'new_window_search_performed':False,'test_scores_used':False},RUN/f'audit/{task}_WINDOW_PROVENANCE.json')
    return seconds

## 19 — Calcul contrôlé par pont et pli
Les comparateurs suivent les règles V1.1. La variante Fault-Aware hors groupe est nouvellement entraînée. Calibration et test restent indépendants des sous-plis qualité. Les seuils des deux agrégateurs Route345 sont figés avant les prédictions test. Un avertisseur sans seuil utile n’interrompt pas le diagnostic structurel : il produit une sortie instrumentale non déterminée.


In [20]:
TRANSFER_PAYLOADS={};ALL_DECISIONS={}


def run_task(records):
    task=records[0].task
    panel,physical,axes=panel_from_train(records)
    candidate=reference_candidate(records,task,panel,physical,axes)
    seconds=resolve_frozen_window(records,task,panel,physical,axes)
    if candidate is not None and not np.isclose(seconds,candidate['protocol']['seconds']):raise ValueError('Reference window disagreement')
    write_json({'panel':panel,'physical_sensor_ids':physical,'axes':axes,'seconds':seconds,'band_hz':CFG.band_hz,
                'welch_segment_seconds':CFG.welch_segment_seconds,'native_modality':records[0].modality},RUN/f'audit/{task}_SELECTED_PROTOCOL.json')
    train=build_store([r for r in records if r.split=='train'],panel,physical,axes,seconds,'train_aug')
    dev=build_store([r for r in records if r.split=='dev'],panel,physical,axes,seconds,'seen_grid')
    assert not set(train.meta.group_id)&set(dev.meta.group_id)
    assert CFG.heldout_fault not in set(train.meta.fault_type)|set(dev.meta.fault_type)
    origin_key=str(RUN.relative_to(BASE_RUN))
    if candidate is not None:
        tr,qmodel,models=deserialize_trusted_legacy(candidate,train,task)
        origin={'mode':'V1_1_FROZEN_COMPARATORS','version':REFERENCE_VERSION,'source':str(candidate['folder']),
                'verified_model_hashes':candidate['hashes'],'reference_freeze_sha256':candidate['reference_freeze_sha256'],
                'new_probability_and_selection_calibration':True,'test_previously_inspected':True}
        table([{'status':'REUSED_FROZEN_V1_1','source':str(candidate['folder'])}],f'tables/{task}_MODEL_SELECTION.csv')
        print('  Poids V1.1 vérifiés et réutilisés ; nouvelle inférence/calibration V1.2.',flush=True)
    else:
        tr=FeatureTransform().fit(train);qmodel=QualityEstimator().fit(train,tr)
        models,hpt=train_tabular(train,dev,tr);table(hpt,f'tables/{task}_MODEL_SELECTION.csv')
        models['HealthyMahalanobis']=HealthyDistance().fit(train,tr)
        # Same V1.1 control training protocol. Crossfit only changes the new FaultAware arm below.
        for name,aware in [('StandardNN',False),('FaultAware',True)]:
            models[name],history=train_sensor_ensemble(train,dev,tr,qmodel,aware,train_quality=None,
                checkpoint_tag='FaultAware_Control' if aware else 'StandardNN')
            table(history,f'tables/{task}_{name}_CONTROL_TRAINING.csv')
        if CFG.enable_minirocket:
            try:models['MiniRocket']=MiniRocketBaseline().fit(train)
            except ImportError as e:note('optional_dependency','MiniRocket',e)
        origin={'mode':'REFIT_V11_COMPARATORS_THEN_CROSSFIT' if CFG.variant=='quality_crossfit' else 'REFIT_REFERENCE_TRAINING_PROTOCOL',
                'version':VERSION,'source':None,'test_previously_inspected':CFG.test_was_previously_inspected,
                'reason':'explicit variant/policy or legacy model files unavailable'}
    models['FaultAware_Control']=models.pop('FaultAware')
    origin['control_source']=origin['mode']
    origin['primary_faultaware']='TRAIN_GROUP_OOF_QUALITY' if CFG.variant=='quality_crossfit' else 'CONTROL_IDENTICAL_DIAGNOSTIC'
    if CFG.variant=='quality_crossfit':
        train_quality=training_quality_crossfit(train)
        models['FaultAware'],history=train_sensor_ensemble(train,dev,tr,qmodel,True,train_quality=train_quality,checkpoint_tag='FaultAware_Crossfit')
        table(history,f'tables/{task}_FaultAware_CROSSFIT_TRAINING.csv')
    else:
        models['FaultAware']=models['FaultAware_Control']
    origin['quality_final_estimator_changed_by_crossfit']=False
    origin['primary_new_training']=CFG.variant=='quality_crossfit'
    EXPERIMENT_STATUS[f'{task}/MiniRocket']='FITTED' if 'MiniRocket' in models else ('UNAVAILABLE_DEPENDENCY' if CFG.enable_minirocket else 'DISABLED_BY_CONFIG')
    PREDICTOR_ORIGINS[origin_key]=origin
    write_json(origin,RUN/f'models/{task}_PREDICTOR_ORIGIN.json')
    # No model/hyperparameter choice below this line.
    calstore=build_store([r for r in records if r.split=='cal'],panel,physical,axes,seconds,'seen_grid')
    assert not(set(calstore.meta.group_id)&(set(train.meta.group_id)|set(dev.meta.group_id)))
    assert CFG.heldout_fault not in set(calstore.meta.fault_type)
    qthr,qdiag=quality_threshold_cal(calstore,qmodel)
    write_json(qdiag,RUN/f'tables/{task}_QUALITY_CALIBRATION.json')
    calibrators={};calpred={}
    for name,model in models.items():
        p=raw_predictions(name,model,calstore,tr,qmodel);calpred[name]=p
        calibrators[name]=fit_calibrator(p,name,qthr,quality_diagnostic=qdiag)
        table(p,f'predictions/{task}_{name}_CAL.csv')
    # Frozen predictor with confidence-only selection: isolates selector effects.
    aliases={
        'FaultAware_confidence_only':('FaultAware','confidence'),
        'StandardNN_confidence_ensemble':('StandardNN','confidence_ensemble'),
        'FaultAware_confidence_ensemble':('FaultAware','confidence_ensemble'),
        'FaultAware_confidence_ensemble_disagreement':('FaultAware','confidence_ensemble_disagreement'),
    }
    for alias,(source,rule) in aliases.items():
        calibrators[alias]=fit_calibrator(calpred[source],alias,qthr,selector=rule,probability_from=calibrators[source],quality_diagnostic=qdiag)
    selectors=[]
    for name,cal in calibrators.items():
        for target,details in cal.selectors.items():
            selectors.append({'model':name,'selector':cal.selector,'target':float(target),
                              'probability_method':cal.probability_method,'cal_reference_groups':cal.class_group_counts.get(0,0),
                              'cal_altered_groups':cal.class_group_counts.get(1,0),**details})
    table(selectors,f'tables/{task}_SELECTOR_CALIBRATION.csv')
    frozen_hash=calibration_fingerprint(calibrators)
    quality_frozen_hash=stable_hash(qdiag)
    save_bundle(task,panel,tr,qmodel,models,calibrators,frozen_hash)
    model_file_hashes={str(p.relative_to(RUN)):file_hash(p) for p in (RUN/'models'/task).glob('*') if p.is_file()}
    write_json({'phase':'FROZEN_BEFORE_TEST','quality_calibration_hash':quality_frozen_hash,'model_sha256':model_file_hashes,'hash':frozen_hash,'train_groups':sorted(train.meta.group_id.unique()),
                'dev_groups':sorted(dev.meta.group_id.unique()),'cal_groups':sorted(calstore.meta.group_id.unique()),
                'selected_seconds':seconds},RUN/f'models/{task}/PRE_TEST_FREEZE.json')
    test=build_store([r for r in records if r.split=='test'],panel,physical,axes,seconds,'test_grid')
    assert not set(test.meta.group_id)&(set(train.meta.group_id)|set(dev.meta.group_id)|set(calstore.meta.group_id))
    table(test.meta,f'audit/{task}_TEST_VARIANT_MANIFEST.csv')
    reference_events=None;permodel={};raw_test={}
    for name,model in models.items():
        pred=raw_predictions(name,model,test,tr,qmodel)
        raw_test[name]=pred
        table(pred,f'predictions/{task}_{name}_RAW_TEST_MEMBERS.csv')
        if reference_events is None:reference_events=set(pred.event_id)
        if set(pred.event_id)!=reference_events:raise ValueError('Unmatched model evaluation events')
        permodel[name]=record_model_results(task,name,pred,calibrators[name])
    for alias,(source,rule) in aliases.items():
        permodel[alias]=record_model_results(task,alias,raw_test[source],calibrators[alias])
    # Explicit hard-gate-only (no uncertainty rejection) ablation.
    p=raw_test['FaultAware']
    d=decide(p,calibrators['FaultAware'])
    d['accepted']=d.usable_physical_sensors>=CFG.min_physical_sensors
    d['structural_decision']=np.where(~d.accepted,'ABSTAIN',np.where(d.binary_prediction==1,'ALERT','NO_ALERT'))
    table(d,f'predictions/{task}_FaultAware_HARD_GATE_ONLY.csv')
    for kind,g in d.groupby('fault_type'):
        METRICS.append({'task':task,'model':'FaultAware_HARD_GATE_ONLY','fault_type':kind,'coverage_target':None,**metric_dict(g)})
    permodel['FaultAware_HARD_GATE_ONLY']={float(c):d.copy() for c in CFG.target_coverages}
    if CFG.seed_diagnostics:
        # Per-seed diagnostics, not seed-selection. Use the ensemble's frozen calibration for comparability.
        for name in ('StandardNN','FaultAware'):
            for i,model in enumerate(models[name]):
                seedpred=raw_predictions(name,[model],test,tr,qmodel)
                sd=decide(seedpred,calibrators[name]);table(sd,f'predictions/{task}_{name}_seed_{i}_TEST_DIAGNOSTIC.csv')
    qmetrics=evaluate_quality_levels(test,qmodel,qthr,qdiag)
    write_json(qmetrics,RUN/f'tables/{task}_QUALITY_TEST.json')
    if candidate is not None:
        comparisons=[]
        for name,pred in raw_test.items():
            if name=='FaultAware':continue # This is deliberately newly trained; never claim it is frozen.
            oldname='FaultAware' if name=='FaultAware_Control' else name
            oldfile=candidate['folder']/f'predictions/{task}_{oldname}_TEST.csv'
            if not oldfile.is_file():continue
            old=pd.read_csv(oldfile)
            if set(old.event_id)!=set(pred.event_id):raise ValueError('Legacy and revised evaluation event sets differ')
            paired=pred.set_index('event_id').loc[old.event_id]
            delta=np.abs(paired.p.to_numpy()-old.p.to_numpy())
            comparisons.append({'model':name,'max_abs_raw_prediction_difference':float(delta.max()),
                'same_frozen_predictor':True,'within_float_tolerance':bool(np.allclose(paired.p,old.p,rtol=1e-5,atol=1e-6)),
                'old_test_was_seen':True})
        table(comparisons,f'tables/{task}_V1_1_FROZEN_PREDICTION_CHECK.csv')
        if comparisons and not all(r['within_float_tolerance'] for r in comparisons):
            raise RuntimeError('Frozen predictor raw predictions differ beyond tolerance; review environment, preprocessing and weights')
    reps=min(40,CFG.bootstrap_replicates) if CFG.mode in {'smoke','selftest'} else CFG.bootstrap_replicates
    for comparator in ('FaultAware_Control','StandardNN','RandomForest','ExtraTrees','FaultAware_confidence_ensemble','FaultAware_confidence_ensemble_disagreement'):
        a=permodel['FaultAware'][float(CFG.primary_coverage)];b=permodel[comparator][float(CFG.primary_coverage)]
        for subset in ('nominal','seen_faults','unseen_fault'):
            mask=(a.fault_type=='nominal') if subset=='nominal' else ((a.fault_type==CFG.heldout_fault) if subset=='unseen_fault' else ~a.fault_type.isin(['nominal',CFG.heldout_fault]))
            if not mask.any():continue
            aa=a[mask].reset_index(drop=True);bb=b[b.event_id.isin(aa.event_id)].reset_index(drop=True)
            for metric in ('selective_risk','false_structural_alarm_rate','false_no_alert_rate','coverage'):
                result=paired_bootstrap(aa,bb,metric,reps,CFG.seed)
                PAIRWISE.append({'task':task,'A':'FaultAware','B':comparator,'subset':subset,'metric':metric,
                                 'cluster_level':'acquisition_conditional',**result})
                # Dependence sensitivity: all held-out records of one campaign/state move together.
                ast=aa.copy();bst=bb.copy()
                ast['group_id']=ast['state'];bst['group_id']=bst['state']
                state_result=paired_bootstrap(ast,bst,metric,reps,CFG.seed)
                state_result['interpretation']='paired state/campaign-cluster sensitivity; few observed states, fixed fitted model; NOT state-disjoint or causal validation'
                PAIRWISE.append({'task':task,'A':'FaultAware','B':comparator,'subset':subset,'metric':metric,
                                 'cluster_level':'state_campaign_sensitivity',**state_result})
    if stable_hash(qdiag)!=quality_frozen_hash:raise RuntimeError('Quality calibration mutated after test')
    if calibration_fingerprint(calibrators)!=frozen_hash:raise RuntimeError('Calibration mutated after test access')
    ALL_DECISIONS[task]=permodel
    # Keep only compact quality arrays for the independent transfer experiment.
    TRANSFER_PAYLOADS[task]={}
    for label,st in [('train',train),('cal',calstore),('test',test)]:
        mask=st.present.astype(bool)
        TRANSFER_PAYLOADS[task][label]={'x':st.qb[mask].copy(),'y':st.yf[mask].astype(int),
             'groups':np.repeat(st.meta.group_id.to_numpy()[:,None],st.qb.shape[1],axis=1)[mask],
             'fault':np.repeat(st.meta.fault_type.to_numpy()[:,None],st.qb.shape[1],axis=1)[mask],
             'physical_sensor':np.repeat(np.asarray(physical)[None,:],len(st.meta),axis=0)[mask]}
    for label in TRANSFER_PAYLOADS[task]:
        TRANSFER_PAYLOADS[task][label]=bounded_transfer_payload(TRANSFER_PAYLOADS[task][label])
    EXPERIMENT_STATUS[task]={'status':'COMPLETE','selected_seconds':seconds,'panel':panel,'physical_sensors':sorted(set(physical)),'n_train_acquisitions':train.meta.group_id.nunique(),
         'n_dev_acquisitions':dev.meta.group_id.nunique(),'n_cal_acquisitions':calstore.meta.group_id.nunique(),
         'n_test_acquisitions':test.meta.group_id.nunique(),'test_variants':len(test.meta),'test_events':len(reference_events),
         'calibration_immutable':True,'predictor_origin':origin,'quality_calibration_level':'acquisition',
         'quality_alarm_status':qdiag['status'],'quality_alarm_available':qdiag['available'],
         'route345_q90_study':qdiag.get('alternative_aggregation',{}).get('status','NOT_REQUESTED'),
         'primary_selector':'common_confidence','quality_crossfit':CFG.variant=='quality_crossfit','source_origin':'SYNTHETIC_FIXTURES' if CFG.mode=='selftest' else 'LOCAL_REAL_ARCHIVES'}
    flush_reports();table(METRICS,'tables/MAIN_METRICS.csv');table(PAIRWISE,'tables/PAIRED_BOOTSTRAP.csv')
    del train,dev,calstore,test,models;gc.collect()

## 20 — Transfert de qualité entre ponts (expérience distincte)
Descripteurs de qualité de base seulement, sans référence saine ni calibration du pont cible. Deux ponts sources fournissent entraînement et calibration du pli initial ; chaque acquisition cible est évaluée une fois via ses prédictions hors échantillon. Les familles de défauts restent des gabarits simulés partagés, donc ce test ne prouve pas un transfert sur des pannes réelles. Les poids équilibrent sources, acquisitions et labels de corruption.

In [21]:
def bounded_transfer_payload(data):
    groups=np.asarray(data['groups']);indices=[]
    for g in sorted(set(groups)):
        ids=np.flatnonzero(groups==g)
        if len(ids)>CFG.transfer_rows_per_acquisition:
            rng=np.random.default_rng(int(stable_hash(['transfer_limit',g,CFG.seed])[:8],16))
            ids=np.sort(rng.choice(ids,CFG.transfer_rows_per_acquisition,replace=False))
        indices.extend(ids.tolist())
    ix=np.array(indices,dtype=int)
    return {k:np.asarray(v)[ix] for k,v in data.items()}

def transfer_weight(data,source_id):
    groups=np.asarray(data['groups']);weights=np.zeros(len(groups))
    unique=set(groups)
    for g in unique:
        ix=np.flatnonzero(groups==g);weights[ix]=1/max(1,len(ix))/max(1,len(unique))
    return weights

def run_quality_transfer():
    if not CFG.enable_quality_transfer:
        EXPERIMENT_STATUS['quality_transfer']={'status':'DISABLED_BY_CONFIG'};return
    rows=[]
    for target in CFG.required_tasks:
        sources=[s for s in CFG.required_tasks if s!=target]
        if any(s not in FOLD_PAYLOADS or not FOLD_PAYLOADS[s] for s in [target]+sources):
            raise RuntimeError('Missing fold quality payload for transfer: '+target)
        # Source training and source calibration are disjoint within their first fold.
        trains=[FOLD_PAYLOADS[s][0]['train'] for s in sources]
        cals=[FOLD_PAYLOADS[s][0]['cal'] for s in sources]
        tests=[v['test'] for v in FOLD_PAYLOADS[target]]
        x=np.concatenate([v['x'] for v in trains]);y=np.concatenate([v['y'] for v in trains])
        xc=np.concatenate([v['x'] for v in cals]);yc=np.concatenate([v['y'] for v in cals])
        xt=np.concatenate([v['x'] for v in tests]);yt=np.concatenate([v['y'] for v in tests])
        tg=np.concatenate([v['groups'] for v in tests]);tf=np.concatenate([v['fault'] for v in tests])
        weight_blocks=[]
        for s,v in zip(sources,trains):
            b=transfer_weight(v,s)
            for label in (0,1):
                use=v['y']==label;b[use]/=max(b[use].sum(),1e-30)
            weight_blocks.append(b)
        w=np.concatenate(weight_blocks)
        wc=np.concatenate([transfer_weight(v,s) for s,v in zip(sources,cals)])
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',RuntimeWarning);med=np.nanmedian(x,axis=0)
        med=np.where(np.isfinite(med),med,0.)
        def impute(a):return np.clip(np.where(np.isfinite(a),a,med),-1e6,1e6)
        for label in (0,1):w[y==label]/=max(w[y==label].sum(),1e-30)
        clf=ExtraTreesClassifier(n_estimators=48 if CFG.mode in {'smoke','selftest'} else CFG.quality_tree_estimators,
            min_samples_leaf=3,random_state=CFG.seed,n_jobs=CFG.cpu_threads).fit(impute(x),y,sample_weight=w/w.mean())
        pc=clf.predict_proba(impute(xc))[:,1]
        threshold=max((f1_score(yc,pc>=t,zero_division=0,sample_weight=wc),-abs(t-.5),float(t)) for t in np.linspace(.1,.9,33))[2]
        freeze={'target':target,'source_tasks':sources,'source_training_groups':sorted(set(np.concatenate([v['groups'] for v in trains]))),
                'source_calibration_groups':sorted(set(np.concatenate([v['groups'] for v in cals]))),'threshold':threshold,'uses_target_calibration':False}
        write_json(freeze,RUN/f'models/QUALITY_TRANSFER_TO_{target}_FREEZE.json')
        pt=clf.predict_proba(impute(xt))[:,1]
        weights=np.zeros(len(tg))
        for g in set(tg):
            ix=np.flatnonzero(tg==g);weights[ix]=1/len(ix)
        pred=pd.DataFrame({'target':target,'group_id':tg,'fault_type':tf,'injected_fault':yt,'p_fault':pt,'predicted_fault':pt>=threshold,'acquisition_weight':weights})
        table(pred,f'predictions/QUALITY_TRANSFER_TO_{target}.csv')
        detail=[]
        for family in sorted(set(tf)):
            take=tf==family
            detail.append({'target':target,'fault_type':family,'unit':'available_channel_window',
                **binary_quality_summary(yt[take],pt[take]>=threshold,weights[take])})
        table(detail,f'tables/QUALITY_TRANSFER_TO_{target}_BY_FAMILY.csv')
        if all('physical_sensor' in v for v in tests):
            sensors=np.concatenate([v['physical_sensor'] for v in tests]);per_sensor=[]
            for sensor in sorted(set(sensors)):
                take=sensors==sensor
                per_sensor.append({'target':target,'physical_sensor':sensor,
                    **binary_quality_summary(yt[take],pt[take]>=threshold,weights[take])})
            table(per_sensor,f'tables/QUALITY_TRANSFER_TO_{target}_BY_SENSOR.csv')
        for subset in ('all','seen_family','unseen_family'):
            mask=np.ones(len(yt),bool) if subset=='all' else (np.isin(tf,['nominal',CFG.heldout_fault]) if subset=='unseen_family' else tf!=CFG.heldout_fault)
            yy=yt[mask];pp=pt[mask];ww=weights[mask]
            rows.append({'target':target,'sources':'+'.join(sources),'subset':subset,'status':'COMPLETE',
                'zero_target_calibration':True,'feature_count':len(QBASE_NAMES),'threshold_source_cal':threshold,
                'fault_f1_acquisition_weighted':float(f1_score(yy,pp>=threshold,zero_division=0,sample_weight=ww)),
                'fault_auroc_acquisition_weighted':float(roc_auc_score(yy,pp,sample_weight=ww)) if len(np.unique(yy))==2 else None,
                'target_acquisitions':len(np.unique(tg[mask])),'fault_truth':'synthetic window corruptions',
                'max_channel_windows_per_acquisition':CFG.transfer_rows_per_acquisition,'no_new_bridge_structural_claim':True,
                **binary_quality_summary(yy,pp>=threshold,ww)})
        atomic_joblib({'model':clf,'imputation_source_medians':med,'threshold':threshold,'features':QBASE_NAMES},RUN/f'models/QUALITY_TRANSFER_TO_{target}.joblib')
    table(rows,'tables/QUALITY_TRANSFER.csv')
    EXPERIMENT_STATUS['quality_transfer']={'status':'COMPLETE','results':rows,'target_tasks':list(CFG.required_tasks)}

## 21 — openLAB 2024 : deux unités d'analyse séparées

Analyse principale : seuil sur les scores des enregistrements de calibration, appliqué aux mêmes scores par enregistrement test. Analyse secondaire : médianes journalières calibrées puis évaluées par jour. L'intervalle du taux par enregistrement est obtenu par bootstrap regroupant les jours.

Il s'agit d'alertes sur une archive de référence, pas d'une détection de dommages validée ni d'un taux par heure de surveillance continue.

In [22]:
def reference_record_threshold(cal_events,quantile):
    valid=cal_events.usable_physical_sensors>=CFG.min_physical_sensors
    scores=cal_events.loc[valid,'score'].to_numpy(float)
    if len(scores)==0 or not np.isfinite(scores).all():raise ValueError('No finite eligible reference calibration records')
    return float(np.quantile(scores,quantile,method='higher'))

def bootstrap_reference_days(events,replicates,seed):
    counts=events.groupby('group_id').agg(alert=('alert','sum'),abstain=('abstain','sum'),n=('alert','size'))
    if len(counts)<2:return {'n_days':len(counts),'ci95_alarm_low':None,'ci95_alarm_high':None,'status':'TOO_FEW_TEST_DAYS'}
    rng=np.random.default_rng(seed);mult=rng.multinomial(len(counts),np.repeat(1/len(counts),len(counts)),size=replicates)
    sums=mult@counts[['alert','abstain','n']].to_numpy(float);vals=sums[:,0]/sums[:,2]
    return {'n_days':len(counts),'ci95_alarm_low':float(np.quantile(vals,.025)),
            'ci95_alarm_high':float(np.quantile(vals,.975)),
            'interpretation':'conditional day-cluster bootstrap of record-level alarm fraction; no hourly guarantee'}

def run_reference_only(records):
    if not CFG.run_reference_2024:EXPERIMENT_STATUS['openLAB2024']={'status':'DISABLED_BY_CONFIG'};return
    if len(records)<8:EXPERIMENT_STATUS['openLAB2024']={'status':'INSUFFICIENT_REFERENCE_RECORDS'};return
    recs=copy.deepcopy(records)
    for r in recs:
        date=str(r.metadata.get('origin',''))[:10]
        if not re.fullmatch(r'\d{4}-\d{2}-\d{2}',date):raise ValueError('Reference chronology unresolved')
        r.group_id=f'openLAB2024/day/{date}'
    days=sorted({r.group_id for r in recs})
    if len(days)<5:raise ValueError('Insufficient independent reference days')
    ntrain=max(2,int(.60*len(days)));ncal=max(1,int(.20*len(days)))
    mapping={d:('train' if i<ntrain else 'cal' if i<ntrain+ncal else 'test') for i,d in enumerate(days)}
    for r in recs:r.split=mapping[r.group_id]
    audit_split(recs);panel,physical,axes=panel_from_train(recs)
    train=build_store([r for r in recs if r.split=='train'],panel,physical,axes,CFG.reference_window_seconds,'nominal')
    cal=build_store([r for r in recs if r.split=='cal'],panel,physical,axes,CFG.reference_window_seconds,'nominal')
    tr=FeatureTransform().fit(train);md=HealthyDistance().fit(train,tr)
    def event_scores(st):
        scores=md.score_array(md.pooled(st,tr));df=st.meta.copy();df['score']=scores
        return df.groupby(['acquisition_id','group_id'],sort=False).agg(
            score=('score','mean'),usable_physical_sensors=('usable_physical_sensors','min')).reset_index()
    cc=event_scores(cal)
    threshold=reference_record_threshold(cc,CFG.reference_quantile)
    cc['valid']=cc.usable_physical_sensors>=CFG.min_physical_sensors
    # Separate, noninterchangeable daily analysis: valid records -> median day -> daily threshold.
    daycal=cc[cc.valid].groupby('group_id').score.median()
    daythreshold=float(np.quantile(daycal,CFG.reference_quantile,method='higher'))
    freeze={'record_threshold':threshold,'record_calibration_unit':'one_mean_score_per_eligible_record',
            'record_calibration_records':int(cc.valid.sum()),'record_nominal_cal_alert_fraction':float(((cc.score>threshold)&cc.valid).mean()),
            'daily_threshold':daythreshold,'daily_calibration_unit':'median_of_valid_record_scores_per_day',
            'cal_days':sorted(cc.group_id.unique()),'no_damage_labels':True,'quantile':CFG.reference_quantile,
            'small_N_warning':len(daycal)<20,'population_alarm_guarantee':False}
    write_json(freeze,RUN/'models/OPENLAB2024_FREEZE.json');freeze_hash=stable_hash(freeze)
    atomic_joblib({'transform':tr,'model':md,'panel':panel,'freeze':freeze},RUN/'models/OPENLAB2024_REFERENCE_MODEL.joblib')
    table(cc,'predictions/OPENLAB2024_CALIBRATION_EVENTS.csv')
    test=build_store([r for r in recs if r.split=='test'],panel,physical,axes,CFG.reference_window_seconds,'nominal')
    dd=event_scores(test);dd['valid']=dd.usable_physical_sensors>=CFG.min_physical_sensors
    dd['alert']=dd.valid&(dd.score>threshold);dd['abstain']=~dd.valid
    table(dd,'predictions/OPENLAB2024_REFERENCE_EVENTS.csv')
    daily=[]
    for g,x in dd.groupby('group_id'):
        valid=x[x.valid];score=float(valid.score.median()) if len(valid) else np.nan
        daily.append({'group_id':g,'score':score,'n_records':len(x),'n_valid_records':len(valid),
                      'alert':bool(len(valid) and score>daythreshold),'abstain':not bool(len(valid))})
    daily=pd.DataFrame(daily);table(daily,'predictions/OPENLAB2024_REFERENCE_DAYS.csv')
    reps=min(40,CFG.bootstrap_replicates) if CFG.mode in {'smoke','selftest'} else CFG.bootstrap_replicates
    result={'status':'COMPLETE_REFERENCE_ONLY','n_test_records':len(dd),'n_test_days':dd.group_id.nunique(),
            'alarm_fraction_per_record':float(dd.alert.mean()),'abstention_fraction':float(dd.abstain.mean()),
            'threshold':threshold,'calibration_unit':'same_acquisition_score_as_record_evaluation',
            'record_quantile':CFG.reference_quantile,'record_confidence_interval':bootstrap_reference_days(dd,reps,CFG.seed),
            'daily_analysis':{'threshold':daythreshold,'calibration_and_evaluation_unit':'day_median_of_valid_record_scores',
                              'alarm_fraction_per_day':float(daily.alert.mean()),'abstention_fraction_per_day':float(daily.abstain.mean())},
            'no_damage_detection_claim':True,'no_continuous_hourly_alarm_rate_claim':True,'population_false_alarm_guarantee':False}
    if stable_hash(freeze)!=freeze_hash:raise RuntimeError('Reference calibration mutated')
    EXPERIMENT_STATUS['openLAB2024']=result;write_json(result,RUN/'tables/OPENLAB2024_REFERENCE_RESULTS.json')

## 22 — Résultats révisés, graphiques et comparaison traçable

Tableaux complets : couvertures par classe, incertitudes et ex æquo, diagnostics de qualité par niveau, comparaison descriptive V1.0–V1.1 lorsque la source est présente. Ne pas sélectionner un seuil depuis ces tableaux test.

Tous les fichiers destinés à l'article sont créés à partir des prédictions de cette exécution et indiquent la provenance des modèles. Aucun signal brut ni poids pickle n'est inclus dans les paquets article.

In [23]:
def calibration_ece(d,bins=10):
    p=d.p_cal.to_numpy();y=d.y.to_numpy();value=0.
    for i in range(bins):
        mask=(p>=i/bins)&((p<(i+1)/bins) if i<bins-1 else (p<=1))
        if mask.any():value+=mask.mean()*abs(p[mask].mean()-y[mask].mean())
    return float(value)

def subset_mask(d,subset):
    if subset=='nominal':return d.fault_type=='nominal'
    if subset=='seen_faults':return ~d.fault_type.isin(['nominal',CFG.heldout_fault])
    if subset=='unseen_fault':return d.fault_type==CFG.heldout_fault
    if subset=='all_faults':return d.fault_type!='nominal'
    return np.ones(len(d),bool)

def pooled_oof_results():
    global METRICS,PAIRWISE,ALL_DECISIONS
    METRICS=[];PAIRWISE=[];ALL_DECISIONS={}
    summary=[];counts=[];matched=[]
    for task,folds in ALL_FOLD_DECISIONS.items():
        names=set(folds[0]);
        if any(set(v)!=names for v in folds):raise ValueError('Model sets differ across folds: '+task)
        ALL_DECISIONS[task]={}
        for name in sorted(names):
            ALL_DECISIONS[task][name]={}
            for coverage in CFG.target_coverages:
                parts=[]
                for fold,result in enumerate(folds):
                    d=result[name][float(coverage)].copy();d['outer_fold']=fold+1;parts.append(d)
                joined=pd.concat(parts,ignore_index=True)
                if joined.event_id.duplicated().any():raise ValueError('Outer tests duplicate an acquisition/scenario event')
                # No recalibration on the concatenated outer test predictions.
                ALL_DECISIONS[task][name][float(coverage)]=joined
                if coverage==CFG.primary_coverage:table(joined,f'predictions/{task}_{name}_OOF.csv')
                for (ft,sev),g in joined.groupby(['fault_type','severity']):
                    curve,auc,maxcov=risk_coverage_exact(g)
                    METRICS.append({'task':task,'model':name,'fault_type':ft,'severity':sev,'coverage_target':coverage,
                        'aurc_attainable':auc,'max_attainable_coverage':maxcov,'ece_all':calibration_ece(g),**metric_dict(g)})
                for subset in ('nominal','seen_faults','unseen_fault','all_faults'):
                    dd=joined[subset_mask(joined,subset)].copy()
                    if dd.empty:continue
                    curve,auc,maxcov=risk_coverage_exact(dd)
                    summary.append({'task':task,'model':name,'subset':subset,'coverage_target':coverage,
                        'outer_folds':len(folds),'aurc_attainable':auc,'max_attainable_coverage':maxcov,'ece_all':calibration_ece(dd),**metric_dict(dd)})
                    if coverage==CFG.primary_coverage:
                        table(curve,f'tables/{task}_{name}_{subset}_OOF_RISK_COVERAGE.csv')
                        cm=np.zeros((2,3),int)
                        for yy,dec in zip(dd.y,dd.structural_decision):
                            cm[int(yy),{'NO_ALERT':0,'ALERT':1,'ABSTAIN':2}[dec]]+=1
                        for y in (0,1):
                            for j,label in enumerate(('NO_ALERT','ALERT','ABSTAIN')):
                                counts.append({'task':task,'model':name,'subset':subset,'true_structural_class':y,'decision':label,'n_events':int(cm[y,j])})
                        for target in CFG.target_coverages:
                            feasible=curve[curve.coverage>=target] if not curve.empty else pd.DataFrame()
                            row={'task':task,'model':name,'subset':subset,'requested_coverage':target}
                            if feasible.empty:row['status']='UNATTAINABLE_WITH_PHYSICAL_AVAILABILITY'
                            else:
                                z=feasible.iloc[0];row.update(status='DESCRIPTIVE_TEST_RANKING_NOT_NEW_THRESHOLD',actual_coverage=float(z.coverage),risk=float(z.risk))
                            matched.append(row)
        reps=min(40,CFG.bootstrap_replicates) if CFG.mode in {'smoke','selftest'} else CFG.bootstrap_replicates
        for comparator in ('FaultAware_Control','StandardNN','RandomForest','ExtraTrees','FaultAware_confidence_ensemble'):
            a=ALL_DECISIONS[task]['FaultAware'][float(CFG.primary_coverage)]
            b=ALL_DECISIONS[task][comparator][float(CFG.primary_coverage)]
            for subset in ('nominal','seen_faults','unseen_fault'):
                aa=a[subset_mask(a,subset)].reset_index(drop=True);bb=b[b.event_id.isin(aa.event_id)].reset_index(drop=True)
                if aa.empty:continue
                for metric in ('selective_risk','false_structural_alarm_rate','false_no_alert_rate','coverage'):
                    for level in ('acquisition_conditional','state_campaign_sensitivity'):
                        x=aa.copy();y=bb.copy()
                        if level=='state_campaign_sensitivity':x['group_id']=x.state;y['group_id']=y.state
                        result=paired_bootstrap(x,y,metric,reps,CFG.seed)
                        result['interpretation']='Conditional descriptive cluster bootstrap of OOF predictions; folds have overlapping training sets. Not a full retraining bootstrap or new-state validation.'
                        PAIRWISE.append({'task':task,'A':'FaultAware','B':comparator,'subset':subset,'metric':metric,'cluster_level':level,**result})
    table(METRICS,'tables/MAIN_METRICS.csv');table(summary,'tables/ARTICLE_MAIN_RESULTS.csv')
    table(counts,'tables/CONFUSION_WITH_ABSTENTION.csv');table(PAIRWISE,'tables/PAIRED_BOOTSTRAP.csv')
    table(matched,'tables/MATCHED_COVERAGE_DESCRIPTIVE.csv')
    table(FOLD_METRICS,'tables/FOLD_METRICS.csv');table(FOLD_PAIRWISE,'tables/FOLD_BOOTSTRAP_DIAGNOSTICS.csv')
    if summary:
        df=pd.DataFrame(summary);macro=[]
        for key,g in df.groupby(['model','subset','coverage_target']):
            if set(g.task)!=set(CFG.required_tasks):continue
            row={'model':key[0],'subset':key[1],'coverage_target':key[2],'n_bridges':len(g)}
            for metric in ('coverage','coverage_reference','coverage_altered','balanced_selective_error','selective_risk','false_structural_alarm_rate','false_no_alert_rate','balanced_accuracy_forced_all','auroc_all','f1_forced_all'):
                row['macro_'+metric]=float(g[metric].mean()) if g[metric].notna().all() else None
            macro.append(row)
        table(macro,'tables/MACRO_THREE_BRIDGES.csv')
    return pd.DataFrame(summary)

def save_fig(fig,name):
    fig.tight_layout();fig.savefig(RUN/f'figures/{name}.png',dpi=180);fig.savefig(RUN/f'figures/{name}.pdf')
    plt.close(fig)

def finalize_plots_tables():
    summary=pooled_oof_results()
    if summary.empty:return
    for task,models in ALL_DECISIONS.items():
        for subset in ('nominal','seen_faults','unseen_fault'):
            fig,ax=plt.subplots(figsize=(6,4))
            for name in ('Logistic','RandomForest','ExtraTrees','StandardNN','FaultAware'):
                dd=models[name][float(CFG.primary_coverage)];dd=dd[subset_mask(dd,subset)]
                curve,_,_=risk_coverage_exact(dd)
                if not curve.empty:ax.plot(curve.coverage,curve.risk,marker='o',markersize=3,linestyle='none',label=name)
            ax.set(xlabel='Coverage',ylabel='Selective error rate',title=f'{task} — {subset} (attainable OOF thresholds)')
            ax.grid(alpha=.25);ax.legend(fontsize=8);save_fig(fig,f'{task}_RISK_COVERAGE_{subset}')
        p=models['FaultAware'][float(CFG.primary_coverage)]
        for metric in ('coverage','false_no_alert_rate','false_structural_alarm_rate'):
            fig,ax=plt.subplots(figsize=(6,4))
            for ft,g in p[p.fault_type!='nominal'].groupby('fault_type'):
                points=sorted((float(sev),metric_dict(gg)[metric]) for sev,gg in g.groupby('severity'))
                ax.plot([x[0] for x in points],[x[1] for x in points],marker='o',label=ft)
            ax.set(xlabel='Predeclared corruption severity',ylabel=metric.replace('_',' '),title=f'{task} — FaultAware');ax.legend(fontsize=8);ax.grid(alpha=.25)
            save_fig(fig,f'{task}_SEVERITY_{metric}')
        d=p[p.fault_type=='nominal'];fig,ax=plt.subplots(figsize=(5,3.8));cm=np.zeros((2,3),int)
        for y,decision in zip(d.y,d.structural_decision):cm[int(y),{'NO_ALERT':0,'ALERT':1,'ABSTAIN':2}[decision]]+=1
        ax.imshow(cm,aspect='auto')
        for (i,j),v in np.ndenumerate(cm):ax.text(j,i,str(v),ha='center',va='center')
        ax.set(xticks=range(3),xticklabels=['No alert','Alert','Abstain'],yticks=[0,1],yticklabels=['Reference','Altered'],title=f'{task} — nominal OOF events')
        save_fig(fig,f'{task}_CONFUSION_NOMINAL')
    extra_v12_reports(summary)
    display(summary[(summary.coverage_target==CFG.primary_coverage)&(summary.model.isin(['StandardNN','FaultAware']))][
        ['task','model','subset','n_acquisitions','coverage','coverage_reference','coverage_altered','selective_risk','false_structural_alarm_rate','false_no_alert_rate','auroc_all']])

LIMITATIONS=[
 'Endpoint: documented structural change versus reference, not RUL, optimal maintenance or safety certification.',
 'Nominal recordings may have unannotated sensor problems; truth for quality evaluation means added synthetic corruption.',
 'Three outer folds test every retained acquisition once. Model training sets overlap between folds; neither folds nor windows are independent new campaigns.',
 'Route345 and openLAB have few independent reference acquisitions. Low-N calibration uses identity probabilities and a fixed 0.5 structural threshold; abstention calibration remains empirical.',
 'Group/state-cluster bootstrap intervals are conditional descriptive intervals from frozen OOF predictions, not a population guarantee.',
 'The primary evaluation does not withhold entire damage states or entire new bridges from the structural model.',
 'openLAB 2025 uses only the two verified velocity response channels, random excitation, A40 and the accepted hardware regime; other recordings remain explicitly outside scope.',
 'Losing one of two openLAB response sensors may make a structural decision unavailable; rejection is not a correct diagnosis.',
 'Corruptions are local to source windows after source-native preprocessing; they do not reconstruct a persistent hardware failure across the complete recording.',
 'A14 exclusion is symmetric and based on previously reported project QC; no new independent physical-fault label is claimed.',
 'openLAB 2024 is a separate event-triggered reference experiment; alarm fractions are not continuous hourly rates.',
 'Cross-bridge quality transfer uses shared synthetic corruption templates. Good results do not establish field fault transfer.',
 'V1.2 revises a benchmark whose V1.0 test results have already been inspected. This is not a new independent confirmatory test.',
 'Sensor-check thresholds are acquisition-level empirical calibrations; small calibration cohorts do not establish a population false-alarm guarantee.',
 'Confidence scores use genuine ties; connecting points in ranking plots does not create attainable intermediate coverages.',
 'Quality cross-fitting is the primary separately labelled training experiment, not an automatic improvement claim.',
 'The same architecture/configuration is used on CPU and CUDA, but floating-point results need not be bit-for-bit identical.',
 'All performance values come from this execution. Statistical superiority, causal damage identification and conference acceptance are not guaranteed.'
]

def write_evidence(status,error=None):
    complete=[t for t in CFG.required_tasks if isinstance(EXPERIMENT_STATUS.get(t),dict) and EXPERIMENT_STATUS[t].get('status')=='COMPLETE' and EXPERIMENT_STATUS[t].get('completed_outer_folds')==CFG.outer_folds]
    mandatory_pass=bool(GATES) and all(g['passed'] for g in GATES if g['mandatory'])
    requested_ok=(not CFG.run_reference_2024 or EXPERIMENT_STATUS.get('openLAB2024',{}).get('status')=='COMPLETE_REFERENCE_ONLY') if isinstance(EXPERIMENT_STATUS.get('openLAB2024',{}),dict) else False
    requested_ok=requested_ok and (not CFG.enable_quality_transfer or EXPERIMENT_STATUS.get('quality_transfer',{}).get('status')=='COMPLETE')
    if CFG.enable_minirocket:
        requested_ok=requested_ok and all(EXPERIMENT_STATUS.get(f'{t}/MiniRocket')=='FITTED' for t in CFG.required_tasks)
    main_exists=(BASE_RUN/'tables/ARTICLE_MAIN_RESULTS.csv').is_file()
    available=(CFG.mode=='full' and status in {'COMPLETE','COMPLETE_WITH_LIMITATIONS'} and len(complete)==len(CFG.required_tasks) and mandatory_pass and main_exists)
    all_complete=bool(available and requested_ok)
    effective_status='COMPLETE_WITH_LIMITATIONS' if status=='COMPLETE' and not requested_ok else status
    evidence={'version':VERSION,'run_id':RUN_ID,'mode':CFG.mode,'status':effective_status,'error':error,
        'data_origin':'SYNTHETIC_SOFTWARE_FIXTURES' if CFG.mode=='selftest' else 'LOCAL_ARCHIVES',
        'interface':'Jupyter web, no shell required','implementation_hash':IMPLEMENTATION_HASH,'study_signature':SIGNATURE,
        'executed_on':platform.node(),'device_initial':ENV['device_initial'],'device_final':str(DEVICE),'device_events':DEVICE_EVENTS,
        'variant':CFG.variant,'predictor_origins':PREDICTOR_ORIGINS,'previous_test_inspection':CFG.test_was_previously_inspected,
        'confirmation_status':'REVISED_ANALYSIS_ON_PREVIOUSLY_INSPECTED_BENCHMARK',
        'quality_calibration_level':'acquisition_max_over_channels_and_windows',
        'primary_selector':'confidence_entropy_identical_rule_for_all_six_models',
        'dispersion_selector':'mean windows within each member then std(ddof=0) across members; normalized by 2',
        'elapsed_seconds_this_session':time.perf_counter()-RUNTIME_STARTED,
        'paper_results_available':available,'all_requested_experiments_complete':all_complete,
        'scientific_review_required':True,'performance_superiority':'NOT_ASSUMED',
        'completed_required_tasks':complete,'outer_folds':CFG.outer_folds,'tasks':EXPERIMENT_STATUS,'folds':FOLD_SUMMARIES,
        'gates':GATES,'software_tests':{'passed':sum(t['passed'] for t in SOFTWARE_TESTS),'total':len(SOFTWARE_TESTS)},
        'results_files':{},'sources':SOURCES,'limitations':LIMITATIONS,
        'paper_claim':'Calculated OOF evidence only; no automatic claim of superiority or manuscript acceptance.'}
    for rel in ['tables/ARTICLE_MAIN_RESULTS.csv','tables/MAIN_METRICS.csv','tables/PAIRED_BOOTSTRAP.csv','tables/QUALITY_TRANSFER.csv','tables/MACRO_THREE_BRIDGES.csv','tables/OPENLAB2024_REFERENCE_RESULTS.json','tables/QUALITY_ACQUISITION_TEST.csv','tables/QUALITY_CALIBRATION_SUMMARY.csv','tables/UNCERTAINTY_DIAGNOSTICS.csv','tables/V1_0_V1_1_COMPARISON.csv']:
        p=BASE_RUN/rel
        if p.exists():evidence['results_files'][rel]={'bytes':p.stat().st_size,'sha256':file_hash(p)}
    write_json(evidence,BASE_RUN/'EVIDENCE_SUMMARY.json')
    import html,zipfile
    headings=[f'# MultiBridge V1.2 — {effective_status}',f'Étude : `{RUN_ID}`',f'Mode : `{CFG.mode}`',
        f'Résultats principaux calculés : **{available}**',f'Toutes les expériences demandées terminées : **{all_complete}**',
        '', 'Analyse révisée sur un benchmark déjà examiné en V1.0 : aucune nouvelle confirmation indépendante. Les intervalles sont conditionnels aux campagnes observées.']
    main=BASE_RUN/'tables/ARTICLE_MAIN_RESULTS.csv'
    if main.exists():
        df=pd.read_csv(main);view=df[(df.coverage_target==CFG.primary_coverage)&df.model.isin(['StandardNN','FaultAware'])]
        headings+=['','## Résumé calculé','```csv',view[['task','model','subset','n_acquisitions','coverage','coverage_reference','coverage_altered','selective_risk','false_structural_alarm_rate','false_no_alert_rate','auroc_all']].to_csv(index=False),'```']
    headings+=['','## Limites']+['- '+v for v in LIMITATIONS]
    (BASE_RUN/'RESULTS_FOR_PAPER.md').write_text('\n'.join(headings),encoding='utf-8')
    page=['<!doctype html><html lang="fr"><meta charset="utf-8"><title>MultiBridge V1.2 — résultats</title>',
          '<style>body{font:16px/1.5 system-ui;max-width:1250px;margin:35px auto;padding:20px}table{border-collapse:collapse;font-size:12px}td,th{border:1px solid #ccc;padding:7px}img{max-width:700px;width:95%}pre{white-space:pre-wrap}</style>',
          '<h1>MultiBridge Fault-Aware SHM — V1.2</h1>',f'<p>Statut : <strong>{effective_status}</strong> ; mode {CFG.mode} ; périphérique {DEVICE}.</p>',
          '<p>Scores calculés, non garantis. Absence de résultats dans une table ne signifie pas succès. Les signaux bruts ne sont pas inclus.</p>']
    for filename,title in [('ARTICLE_MAIN_RESULTS.csv','Résultats hors échantillon'),('MACRO_THREE_BRIDGES.csv','Moyennes entre les trois ponts'),('QUALITY_TRANSFER.csv','Transfert de qualité')]:
        p=BASE_RUN/'tables'/filename
        if p.exists():
            d=pd.read_csv(p)
            if 'coverage_target' in d:d=d[d.coverage_target==CFG.primary_coverage]
            page+=['<h2>'+title+'</h2>',d.to_html(index=False,float_format=lambda x:f'{x:.4f}',escape=True)]
    page+=['<h2>Figures</h2>']
    for p in sorted((BASE_RUN/'figures').glob('*.png')):page.append(f'<h3>{html.escape(p.stem)}</h3><img src="figures/{p.name}">')
    page+=['<h2>Limites</h2><ul>']+['<li>'+html.escape(v)+'</li>' for v in LIMITATIONS]+['</ul></html>']
    (BASE_RUN/'REPORT_ARTICLE.html').write_text('\n'.join(page),encoding='utf-8')
    # Small diagnostic package; no raw signals, no native caches, no binary models.
    with zipfile.ZipFile(BASE_RUN/'AUDIT_PACKAGE.zip','w',zipfile.ZIP_DEFLATED) as z:
        for p in (BASE_RUN/'audit').rglob('*'):
            if p.is_file():z.write(p,p.relative_to(BASE_RUN))
        for name in ('environment.json','EVIDENCE_SUMMARY.json','STUDY_LOCK.json'):
            p=BASE_RUN/name
            if p.exists():z.write(p,p.name)
    return evidence

def article_package():
    import zipfile
    dest=BASE_RUN/'ARTICLE_RESULTS.zip'
    with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED) as z:
        for folder in ('tables','figures','predictions','audit'):
            for p in (BASE_RUN/folder).rglob('*'):
                if p.is_file():z.write(p,p.relative_to(BASE_RUN))
        for name in ('EVIDENCE_SUMMARY.json','environment.json','STUDY_LOCK.json','RESULTS_FOR_PAPER.md','REPORT_ARTICLE.html'):
            p=BASE_RUN/name
            if p.exists():z.write(p,p.name)
        # All inner split/calibration tables, but no raw data or model weights.
        for p in (BASE_RUN/'folds').rglob('*'):
            if p.is_file() and (p.suffix.lower() in {'.csv','.json'}) and 'checkpoints' not in p.parts:
                z.write(p,p.relative_to(BASE_RUN))
    print('\nPaquet article :',dest,flush=True)
    for name in ('EVIDENCE_SUMMARY.json','REPORT_ARTICLE.html','ARTICLE_RESULTS.zip','AUDIT_PACKAGE.zip'):
        p=BASE_RUN/name
        try:display(FileLink(str(p.relative_to(Path.cwd())),result_html_prefix=name+' : '))
        except ValueError:print(name,':',p)


def extra_v12_reports(summary):
    """Export diagnostics AFTER frozen-test evaluation. No report feeds a fit or threshold choice."""
    quality_cal=[];quality_levels=[];selector=[];unc=[];crossfit=[];origins=[];pairs=[];drift=[];route=[];reuse=[]
    def read_decisions(path):
        d=pd.read_csv(path)
        if 'sensor_check' in d:d['sensor_check']=d.sensor_check.map(lambda x: pd.NA if pd.isna(x) else (str(x).lower()=='true')).astype('boolean')
        if 'instrument_decision_available' in d:d['instrument_decision_available']=d.instrument_decision_available.map(lambda x:str(x).lower()=='true')
        return d
    for task in CFG.required_tasks:
        for folder in sorted((BASE_RUN/'folds'/task).glob('fold_*')):
            fold=int(folder.name.split('_')[-1]);p=folder/f'tables/{task}_QUALITY_CALIBRATION.json'
            if p.is_file():
                d=json.loads(p.read_text());quality_cal.append({'task':task,'outer_fold':fold,
                    **{k:v for k,v in d.items() if not isinstance(v,(dict,list))},'warnings':';'.join(d.get('warnings',[]))})
                if d.get('alternative_aggregation'):
                    ad=d['alternative_aggregation'];quality_cal.append({'task':task,'outer_fold':fold,
                        **{k:v for k,v in ad.items() if not isinstance(v,(dict,list))},'warnings':';'.join(ad.get('warnings',[]))})
            specifications=[(f'tables/{task}_QUALITY_TEST_LEVELS.csv',quality_levels),
                (f'tables/{task}_SELECTOR_CALIBRATION.csv',selector),(f'tables/{task}_CROSSFIT_GROUP_AUDIT.csv',crossfit),
                (f'tables/{task}_AGGREGATION_COMPARISON.csv',route),
                (f'tables/{task}_DRIFT_WINDOW_DIAGNOSTICS.csv',drift),
                (f'tables/{task}_V1_1_FROZEN_PREDICTION_CHECK.csv',reuse)]
            specifications += [(f'tables/{task}_{part}_QUALITY_SCORE_ORIGIN.csv',origins) for part in ('CAL','TEST')]
            specifications += [(f'tables/{task}_{part}_QUALITY_NOMINAL_CORRUPTED_PAIRS.csv',pairs) for part in ('CAL','TEST')]
            for rel,collection in specifications:
                pp=folder/rel
                if pp.is_file() and pp.stat().st_size>1:
                    df=pd.read_csv(pp);df['task']=task;df['outer_fold']=fold;df['source_table']=rel;collection.append(df)
        for name,bycoverage in ALL_DECISIONS.get(task,{}).items():
            d=bycoverage[float(CFG.primary_coverage)]
            for subset in ('nominal','seen_faults','unseen_fault','all_faults'):
                dd=d[subset_mask(d,subset)]
                if dd.empty:continue
                u=dd.uncertainty.to_numpy();c=Counter(u.tolist())
                unc.append({'task':task,'model':name,'subset':subset,'n_events':len(dd),'unique_scores':len(c),
                    'max_equal_score_events':max(c.values()),'fraction_exactly_one':float(np.mean(u==1)),
                    'std_member_acquisition_max':float(dd.u_ensemble.max()),'coverage_reference':metric_dict(dd)['coverage_reference'],
                    'coverage_altered':metric_dict(dd)['coverage_altered'],'score_is_probability_of_failure':False})
    table(quality_cal,'tables/QUALITY_CALIBRATION_FEASIBILITY.csv')
    table(quality_cal,'tables/QUALITY_CALIBRATION_SUMMARY.csv');table(unc,'tables/UNCERTAINTY_DIAGNOSTICS.csv')
    for parts,name in [(quality_levels,'QUALITY_TEST_LEVELS_ALL_FOLDS'),(selector,'SELECTOR_CALIBRATION_ALL_FOLDS'),
        (crossfit,'CROSSFIT_GROUP_AUDIT'),(origins,'QUALITY_SCORE_ORIGIN'),(pairs,'QUALITY_NOMINAL_CORRUPTED_PAIRS'),
        (drift,'DRIFT_WINDOW_DIAGNOSTICS'),(route,'ROUTE345_AGGREGATION_COMPARISON'),(reuse,'BASELINE_REUSE_AUDIT')]:
        if parts:table(pd.concat(parts,ignore_index=True),f'tables/{name}.csv')
    if crossfit:
        cf=pd.concat(crossfit,ignore_index=True)
        check=not cf.duplicated(['task','outer_fold','heldout_group']).any()
        check=check and cf.heldout_excluded_from_model.all() and cf.heldout_excluded_from_transform.all() and not cf.cal_or_test_used.any()
        gate('crossfit_all_heldout_training_groups_excluded',bool(check),{'rows':len(cf),'tasks':sorted(cf.task.unique())})
    # OOF instrumental availability: retain undetermined events in every denominator report.
    pooled_quality=[];route_pooled=[]
    for task in CFG.required_tasks:
        aggregations=['max','physical_q90'] if task=='Route345' and CFG.run_route345_aggregation_study else ['max']
        for agg in aggregations:
            parts=[]
            for folder in sorted((BASE_RUN/'folds'/task).glob('fold_*')):
                rel=f'predictions/{task}_QUALITY_TEST_'+('Q90_' if agg=='physical_q90' else '')+'ACQUISITIONS.csv'
                p=folder/rel
                if p.is_file():
                    d=read_decisions(p);d['outer_fold']=int(folder.name.split('_')[-1]);parts.append(d)
            if not parts:continue
            d=pd.concat(parts,ignore_index=True)
            if d.event_id.duplicated().any():raise ValueError('Duplicated OOF instrumental events')
            table(d,f'predictions/{task}_QUALITY_{agg}_ACQUISITION_OOF.csv')
            specifications=[('nominal',None,None),('seen_faults',None,None),('unseen_fault',None,None),('all_faults',None,None)]
            specifications += [('family_severity',ft,float(sev)) for ft,sev in d[d.fault_type!='nominal'].groupby(['fault_type','severity']).groups]
            for subset,ft,sev in specifications:
                dd=d[(d.fault_type==ft)&(d.severity==sev)] if ft else d[subset_mask(d,subset)]
                if dd.empty:continue
                counts=dd.groupby('group_id').size().to_dict();w=np.array([1/counts[g] for g in dd.group_id])
                result={'task':task,'aggregation':agg,'subset':subset,'fault_type':ft,'severity':sev,
                    'truth':'added corruption only; nominal source may contain unannotated issues','no_test_selection':True,
                    **quality_service_summary(dd,w)}
                if agg=='max':pooled_quality.append(result)
                if task=='Route345':route_pooled.append(result)
    table(pooled_quality,'tables/QUALITY_ACQUISITION_TEST.csv')
    if route_pooled:table(route_pooled,'tables/ROUTE345_AGGREGATION_OOF.csv')
    # Side-by-side parent comparison, only for reporting. The reference file is never used by the learner.
    oldfile=REFERENCE_RUN/'tables/ARTICLE_MAIN_RESULTS.csv'
    scope={'status':'NO_PARENT_TABLE','checks':[],'used_for_fitting':False}
    if oldfile.is_file():
        scope['status']='VERIFIED_SAME_EVENT_SCOPE'
        for task in CFG.required_tasks:
            op=REFERENCE_RUN/f'predictions/{task}_FaultAware_OOF.csv'
            current=ALL_DECISIONS.get(task,{}).get('FaultAware',{}).get(float(CFG.primary_coverage))
            if not op.is_file() or current is None:
                scope['status']='REFERENCE_SCOPE_NOT_VERIFIABLE';scope['checks'].append({'task':task,'verified':False,'reason':'Parent OOF events unavailable'})
                continue
            parent=pd.read_csv(op,usecols=['event_id','group_id','y','state'])
            ok=not parent.event_id.duplicated().any() and set(parent.event_id)==set(current.event_id)
            if ok:
                aligned=parent.set_index('event_id').loc[current.event_id]
                ok=(np.array_equal(aligned.group_id.astype(str),current.group_id.astype(str)) and
                    np.array_equal(aligned.y.to_numpy(int),current.y.to_numpy(int)) and
                    np.array_equal(aligned.state.astype(str),current.state.astype(str)))
            scope['checks'].append({'task':task,'verified':bool(ok),'parent_sha256':file_hash(op)})
            if not ok:scope['status']='REFERENCE_SCOPE_INCOMPATIBLE'
    write_json(scope,BASE_RUN/'audit/V1_1_COMPARISON_SCOPE.json')
    if oldfile.is_file() and scope['status']=='VERIFIED_SAME_EVENT_SCOPE':
        old=pd.read_csv(oldfile);keys=['task','model','subset','coverage_target']
        metrics=['coverage','coverage_reference','coverage_altered','selective_risk','balanced_accuracy_forced_all',
                 'false_no_alert_rate','false_structural_alarm_rate','sensor_check_fraction']
        shared=[m for m in metrics if m in old and m in summary]
        both=old[keys+shared].merge(summary[keys+shared],on=keys,suffixes=('_V1_1','_V1_2'),validate='one_to_one')
        for m in shared:both[m+'_difference']=both[m+'_V1_2']-both[m+'_V1_1']
        both['instrument_fraction_caveat']='V1.2 fraction conditional on available decisions; inspect decision availability'
        both['interpretation']='REVISED_PREVIOUSLY_INSPECTED_BENCHMARK_NOT_INDEPENDENT_CONFIRMATION'
        table(both,'tables/V1_1_V1_2_COMPARISON.csv')
        write_json({'source':str(oldfile),'sha256':file_hash(oldfile),'used_for_fitting':False},BASE_RUN/'audit/V1_1_REFERENCE_RESULTS.json')
    else:
        # Replace any earlier generated comparison, rather than leaving a stale numeric table on resume.
        table([{'status':scope['status'],'comparison_available':False,'reason':'Parent event scope not verified; no numeric version comparison'}],'tables/V1_1_V1_2_COMPARISON.csv')
        note('comparison_unavailable','V1_1_REFERENCE_RESULTS','Parent tables absent or event scope unverified; no numeric comparison',scope=scope['status'])
    for task in CFG.required_tasks:
        selected=summary[(summary.task==task)&(summary.subset=='nominal')&(summary.coverage_target==CFG.primary_coverage)&summary.model.isin(['ExtraTrees','StandardNN','FaultAware_Control','FaultAware'])]
        if len(selected):
            fig,ax=plt.subplots(figsize=(7,4));x=np.arange(len(selected))
            ax.plot(x,selected.coverage_reference,marker='o',label='Reference coverage')
            ax.plot(x,selected.coverage_altered,marker='s',label='Altered coverage')
            ax.set(xticks=x,xticklabels=selected.model,ylim=(0,1.05),ylabel='Fraction accepted',title=f'{task} — class-specific nominal coverage')
            ax.tick_params(axis='x',labelrotation=15);ax.legend();ax.grid(alpha=.25);save_fig(fig,f'{task}_COVERAGE_BY_CLASS')

# Extend evidence without weakening the inherited completed-fold checks.
_write_evidence_core=write_evidence

def write_evidence(status,error=None):
    issues=[];availability=[]
    for p in sorted((BASE_RUN/'folds').glob('*/fold_*/tables/*_QUALITY_CALIBRATION.json')):
        d=json.loads(p.read_text());task=p.parts[-4];fold=p.parts[-3]
        availability.append({'task':task,'outer_fold':fold,'aggregation':'max','status':d['status'],'available':d['available']})
        if not d['available']:issues.append({'task':task,'outer_fold':fold,'aggregation':'max','status':d['status']})
        ad=d.get('alternative_aggregation')
        if ad:
            availability.append({'task':task,'outer_fold':fold,'aggregation':'physical_q90','status':ad['status'],'available':ad['available']})
            if not ad['available']:issues.append({'task':task,'outer_fold':fold,'aggregation':'physical_q90','status':ad['status']})
    adjusted='COMPLETE_WITH_LIMITATIONS' if status=='COMPLETE' and issues else status
    ev=_write_evidence_core(adjusted,error)
    ev['instrument_decision_availability']=availability;ev['instrument_threshold_limitations']=issues
    ev['availability_note']='UNDETERMINED is not NO_CHECK; conditional performance and end-to-end service fractions are reported separately'
    ev['primary_training_variant']=CFG.variant
    ev['window_selection_policy']='verified V1.1 duration per fold, otherwise predeclared fixed 20 seconds; no new search'
    ev['reference_version']=REFERENCE_VERSION;ev['reference_root']=str(REFERENCE_RUN)
    ev['crossfit_does_not_itself_modify_final_quality_estimator']=True
    ev['route345_aggregation_study']={'enabled':CFG.run_route345_aggregation_study,'quantile':CFG.route345_temporal_quantile,
        'scope':'instrumental decisions only; max vs inverse empirical CDF after physical-axis pooling','no_best_test_method_selected':True}
    for name in ('CROSSFIT_GROUP_AUDIT','QUALITY_CALIBRATION_FEASIBILITY','QUALITY_SCORE_ORIGIN','QUALITY_NOMINAL_CORRUPTED_PAIRS',
        'DRIFT_WINDOW_DIAGNOSTICS','ROUTE345_AGGREGATION_COMPARISON','ROUTE345_AGGREGATION_OOF','BASELINE_REUSE_AUDIT','V1_1_V1_2_COMPARISON'):
        p=BASE_RUN/f'tables/{name}.csv'
        if p.is_file():ev['results_files'][str(p.relative_to(BASE_RUN))]={'bytes':p.stat().st_size,'sha256':file_hash(p)}
    write_json(ev,BASE_RUN/'EVIDENCE_SUMMARY.json')
    page=BASE_RUN/'REPORT_ARTICLE.html'
    if page.is_file():
        extra='<h2>V1.2 — disponibilité des avertisseurs instrumentaux</h2><p>UNDETERMINED n’est ni un succès ni une absence de défaut. Le calcul peut être achevé avec une calibration inutilisable.</p>'
        if availability:extra+=pd.DataFrame(availability).to_html(index=False,escape=True)
        for name,title in [('QUALITY_ACQUISITION_TEST','Qualité : disponibilité et performances conditionnelles'),('ROUTE345_AGGREGATION_OOF','Route345 : deux agrégations préspécifiées'),('V1_1_V1_2_COMPARISON','Comparaison révisée V1.1/V1.2')]:
            p=BASE_RUN/f'tables/{name}.csv'
            if p.is_file():
                d=pd.read_csv(p)
                if 'coverage_target' in d:d=d[d.coverage_target==CFG.primary_coverage]
                if 'subset' in d:d=d[d.subset.isin(['nominal','seen_faults','unseen_fault','all_faults'])]
                extra+='<h2>'+title+'</h2>'+d.to_html(index=False,escape=True,float_format=lambda x:f'{x:.4f}')
        text=page.read_text();page.write_text(text.replace('</body>','').replace('</html>',extra+'</html>'),encoding='utf-8')
    # Refresh the small archive after extending the JSON so both archives always describe the same run.
    import zipfile
    with zipfile.ZipFile(BASE_RUN/'AUDIT_PACKAGE.zip','w',zipfile.ZIP_DEFLATED) as z:
        for p in (BASE_RUN/'audit').rglob('*'):
            if p.is_file():z.write(p,p.relative_to(BASE_RUN))
        for name in ('environment.json','EVIDENCE_SUMMARY.json','STUDY_LOCK.json'):
            p=BASE_RUN/name
            if p.exists():z.write(p,p.name)
    return ev



## 23 — Tests logiciels intégrés et archives synthétiques explicites

Tests natifs hérités de V1.0, puis nouveaux tests de dispersion, ordre d'agrégation, ex æquo, budget d'alerte acquisitionnel, petits effectifs par classe et absence d'utilisation des étiquettes de test. Les fixtures ne sont générées qu'en mode `selftest`.

In [24]:
SOFTWARE_TESTS=[]

def check_test(name,condition,details=''):
    row={'test':name,'passed':bool(condition),'details':details}
    SOFTWARE_TESTS.append(row)
    write_json(SOFTWARE_TESTS,RUN/'audit/SOFTWARE_TESTS.json')
    if not condition:raise AssertionError(name+': '+str(details))


def expect_failure(name,fn):
    try:fn()
    except (ValueError,AssertionError,RuntimeError):check_test(name,True);return
    check_test(name,False,'Expected a scientific/format validation error')


def run_regression_tests():
    check_test('route345_baseline2_is_reference',classify_rt_state('baseline_2')[0]==0)
    check_test('route345_damage1_is_changed',classify_rt_state('damage_1')[0]==1)
    check_test('route345_unknown_not_guessed',classify_rt_state('scenario_4')[0] is None)
    fs,src,internal=rate_from_metadata({'/sensor:internal_sample_rate':512,'/sensor:sample_rate':128,'time':'09-Jun-2008 13:01:58'})
    check_test('effective128_internal512',fs==128 and internal==[512.])
    check_test('date_not_elapsed_duration',rate_from_metadata({'time':'09-Jun-2008 13:01:58','internal_sample_rate':512})[0] is None)
    expect_failure('conflicting_effective_rates_blocked',lambda:rate_from_metadata({'sample_rate':128,'fs':512}))
    check_test('openlab_reference_label',classify_ol_state('Ungeschädigt')[0]==0)
    check_test('openlab_cut_label',classify_ol_state('LitzePos_1_Spanndraht_01_getrennt')[0]==1)
    check_test('openlab_exposed_not_cut',classify_ol_state('LitzePos_1_Spanndraht_01_freigelegt')[0] is None)
    if not CFG.openlab_allow_postload_as_change:
        check_test('postload_not_invented_damage',classify_ol_state('Laststufe_03_entlastet')[0] is None)
    check_test('native_velocity_role',column_role('21_1k21_WZ_V47 [m/s]','openLAB2025')[0]=='response')
    check_test('shaker_alias_excluded',column_role('50_2K18__ACCEL_BAM_SH2 [m/s²]','openLAB2025')[0]=='excitation')
    check_test('laser_not_force_or_response',column_role('52_2k19_Laser_Distanz [m]','openLAB2025')[:2]==('auxiliary','displacement'))
    check_test('unknown_numeric_not_response',column_role('channel_999','openLAB2025')[0]=='unresolved')
    check_test('shaker_regime_propagates_to_later_records',after_documented_shaker_exchange('2025-05-07 20-28-40 (5)',{'2025-05-07 17-41-58 (4)':{'remarks':'Shaker getauscht'}}))
    check_test('mat_bytes_labels',labels_from_mat(np.array([b'R1V',b'R2L']))==['R1V','R2L'])
    check_test('mat_char_matrix_labels',labels_from_mat(np.array([list('R1V '),list('R2L ')]))==['R1V','R2L'])
    t,_=time_seconds(pd.Series(['2025-05-05T10:00:00.000+02:00','2025-05-05T10:00:00.005+02:00','2025-05-05T10:00:00.010+02:00']),'Time')
    check_test('datetime_subtraction_preserves_resolution',np.allclose(t,[0,.005,.01],rtol=0,atol=1e-10))
    t,_=time_seconds(pd.Series(np.arange(100)*2.),'Time [ms]')
    check_test('explicit_ms_rate',abs(validate_clock(t)[0]-500)<1e-5)
    expect_failure('clock_duplicate_blocked',lambda:validate_clock(np.r_[np.arange(99)*.01,.98]))
    expect_failure('clock_gap_blocked',lambda:validate_clock(np.r_[np.arange(50)*.01,1+np.arange(50)*.01]))
    fs=100.;t=np.arange(2000)/fs;x=np.sin(2*np.pi*4*t)+.05*np.cos(2*np.pi*11*t)
    short=x.copy();short[50:52]=np.nan
    check_test('short_gap_repaired',repair_short_gaps(short,fs)[1])
    long=x.copy();long[50:100]=np.nan
    check_test('long_gap_not_reconstructed',not repair_short_gaps(long,fs)[1])
    edge=x.copy();edge[0]=np.nan
    check_test('edge_gap_not_extrapolated',not repair_short_gaps(edge,fs)[1])
    raw=np.column_stack([x,x*.7,x*1.1]);physical=['P1','P1','P2']
    f=Fault('gain',1.,'P1',100,500,99,2.)
    corrupt,affected=inject_fault(raw,physical,f)
    check_test('physical_fault_affects_both_axes',np.allclose(corrupt[100:500,:2],2*raw[100:500,:2]) and affected.tolist()==[1.,1.,0.])
    check_test('fault_does_not_mutate_raw',np.array_equal(raw,np.column_stack([x,x*.7,x*1.1])))
    check_test('deterministic_fault_seed',make_fault(2000,physical,[1,1,1],'noise',.6,'a')==make_fault(2000,physical,[1,1,1],'noise',.6,'a'))
    sf,qb,aux,v=features_window(raw,fs,np.ones(3),['V','L','V'])
    check_test('spectral_feature_shape',sf.shape==(3,len(SF_NAMES)) and v.sum()==3)
    peak_index=SF_NAMES.index('peak_hz')
    check_test('known_tone_peak',abs(sf[0,peak_index]-4)<.6)
    sf2,q2,a2,v2=features_window(np.full_like(raw,np.nan),fs,np.ones(3),['V','L','V'])
    check_test('all_missing_not_valid_structure',not v2.any())
    expected=list(range(0,2001,1000))
    check_test('test_windows_nonoverlap',starts_for(3000,1000,'test')==expected)
    check_test('minimum_group_allocation',allocate_counts(5)==[2,1,1,1])
    expect_failure('too_few_groups_not_window_split',lambda:allocate_counts(4))
    mini=pd.DataFrame({'y':[0,0,1,1],'p_cal':[.1,.8,.2,.9],'accepted':[True,True,True,False],
                      'binary_prediction':[0,1,0,1],'group_id':['a','b','c','d'],'sensor_check':[False]*4})
    m=metric_dict(mini)
    check_test('false_alarm_denominator_all_reference',m['false_structural_alarm_rate']==.5)
    check_test('false_no_alert_denominator_all_changed',m['false_no_alert_rate']==.5)
    check_test('abstention_not_counted_correct',abs(m['selective_risk']-2/3)<1e-10 and m['abstention_altered']==.5)
    mini['uncertainty']=[.2]*4;mini['usable_physical_sensors']=[2]*4
    curve,aurc,maxcov=risk_coverage_exact(mini)
    check_test('risk_coverage_ties_not_optimistic',len(curve)==1 and maxcov==1 and abs(aurc-.5)<1e-10)
    mini['usable_physical_sensors']=0
    check_test('no_valid_sensors_zero_attainable_coverage',risk_coverage_exact(mini)[2]==0)
    print(f'Software regression tests (source contracts): {len(SOFTWARE_TESTS)} passed',flush=True)


def make_synthetic_archives():
    """Explicit software fixtures. Never called for real-data run modes."""
    if CFG.mode!='selftest':raise RuntimeError('Synthetic fixture creation requires selftest mode')
    from scipy.io import savemat
    base=RUN/'SYNTHETIC_FIXTURES';zr=base/'z24';orr=base/'openlab2024';od=base/'openlab2025'
    for p in (zr,orr,od,base/'route345'):p.mkdir(parents=True,exist_ok=True)
    rng=np.random.default_rng(67584)
    def wave(fs,n,channels,changed,seed):
        rr=np.random.default_rng(seed);tt=np.arange(n)/fs
        common=(np.sin(2*np.pi*(4.2-.55*changed)*tt+rr.uniform(0,6))+
                .4*np.sin(2*np.pi*(10.2-.7*changed)*tt+rr.uniform(0,6)))
        a=rr.uniform(.7,1.4,(1,channels))*common[:,None]+rr.normal(0,.20,(n,channels))
        return a*.002
    for sid in (1,3,4,8,9,10):
        for setup in range(1,10):
            n=4200+(setup%3)*7;xx=wave(100,n,7,int(sid not in {1,8}),sid*100+setup)
            p=zr/f'{sid:02d}'/'avt'/f'{sid:02d}setup{setup:02d}.mat';p.parent.mkdir(parents=True,exist_ok=True)
            savemat(p,{'data':xx,'labelshulp':np.array(['R1V ','R2L ','R2T ','R2V ','R3V ','V01 ','V02 ']),'fs':100.})
    rt=base/'route345/Rt345Bridge.h5'
    with h5py.File(rt,'w') as h5:
        for state in ['baseline_1','baseline_2']+[f'damage_{i}' for i in range(1,7)]:
            root=h5.create_group(state);root.attrs['damage_description']='synthetic software fixture: '+state
            for test in range(1,4 if state.startswith('baseline') else 3):
                tg=root.create_group(f'acceleration/test_{test}');tg.attrs['time']='09-Jun-2008 13:01:58'
                n=5500+test;tg.attrs['number of samples']=np.array([n],float)
                xx=wave(128,n,10,int(state.startswith('damage')),int(stable_hash([state,test])[:8],16))
                for j,sensor in enumerate(['A01','A02','A03','A04','A14']):
                    sg=tg.create_group(sensor);sg.attrs['internal_sample_rate']=np.array([512.]);sg.attrs['sample_rate']=np.array([128.]);sg.attrs['unit']='m/s^2'
                    for k,axis in enumerate(['y','z']):sg.create_dataset(axis,data=xx[:,2*j+k][None,:])
    logs=['Zeitstempel | Anregung | Wagen | Zustand | Bemerkung']
    cases=[]
    for i in range(6):cases.append(('Ungeschädigt','Shaker_Random_Feld_'+str(1+i%2)))
    for i in range(6):cases.append((f'LitzePos_1_Spanndraht_01_bis_{1+i//2:02d}_getrennt','Shaker_Random_Feld_'+str(1+i%2)))
    cases.extend([('Laststufe_03_entlastet','Shaker_Random_Feld_2'),('LitzePos_1_Spanndraht_01_freigelegt','Shaker_Random_Feld_2'),('Ungeschädigt','Shaker_Frequenz_3Hz')])
    for i,(state,exc) in enumerate(cases):
        instant=pd.Timestamp('2025-05-05T08:00:00+02:00')+pd.Timedelta(minutes=15*i)
        stamp=instant.strftime('%Y-%m-%d %H-%M-%S')+' (1)'
        fname=f'BAM_SYNTHETIC_{i:02d}_Anregung_{exc}_Wagen_Achse_A40_Laststufe_{state}_{stamp}.csv'
        n=42000+i;xx=wave(1000,n,2,int('getrennt' in state),30000+i)
        times=pd.date_range(instant,periods=n,freq='1ms').astype(str)
        frame=pd.DataFrame({'Time':times,'21_1k21_WZ_V47 [m/s]':xx[:,0],
            '33_1k33_WZ_V86 [m/s]':xx[:,1],'49_2k17_Accel_UR_SH1 [m/s²]':rng.normal(0,.01,n),
            '50_2k18__Accel_BAM_SH2 [m/s²]':rng.normal(0,.01,n),'52_2k19_Laser_Distanz [m]':np.full(n,2.)})
        frame.to_csv(od/fname,index=False,sep=';',decimal=',' if i==0 else '.',encoding='cp1252' if i==0 else 'utf-8')
        logs.append(f'{stamp} | {exc} | Achse_A40 | {state} | synthetic fixture')
    (od/'2025_05_05_Übersicht_Messungen.txt').write_text('\n'.join(logs),encoding='utf-8')
    for day in range(10):
        n=22000;instant=pd.Timestamp('2024-02-01T08:00:00Z')+pd.Timedelta(days=day)
        xx=wave(500,n,6,0,70000+day);df=pd.DataFrame({'Timestamp':pd.date_range(instant,periods=n,freq='2ms').astype(str)})
        for j,code in enumerate(('PE11','PE12','PE13','PE21','PE22','PE23')):df[f'G_ACCZ_{code}_CB0750_0']=xx[:,j]
        df.to_csv(orr/f'acceleration_{day:02d}.csv',index=False)
    (base/'WARNING.txt').write_text('SYNTHETIC SOFTWARE TEST DATA. NOT Z24, Route345 or openLAB experimental data. DO NOT PUBLISH THESE METRICS AS BRIDGE RESULTS.')
    return dict(z24_root=str(zr),route345_file=str(rt),openlab_2025_root=str(od),openlab_2024_root=str(orr))
def run_v10_regression_tests():
    # The legacy suite deliberately checks the retained legacy splitter as well as new contracts.
    make=lambda task,g,y,setup=0:Record(task,'/fixture','',g,g,'reference' if y==0 else 'changed',y,
                  ['a','b'],['a','b'],['V','V'],['m/s','m/s'],100.,'test','velocity',setup=setup)
    for task,nh,nd in [('Route345',6,19),('openLAB2025',6,6)]:
        records=[make(task,f'{task}/h{i}',0) for i in range(nh)]+[make(task,f'{task}/d{i}',1) for i in range(nd)]
        protocols=outer_protocols(records,task)
        testing=[r.acquisition_id for pp in protocols for r in pp if r.split=='test']
        check_test(task+'_outer_every_acquisition_once',len(testing)==len(records) and len(set(testing))==len(testing))
        check_test(task+'_outer_fourparts_both_classes',all({r.y for r in pp if r.split==s}=={0,1} for pp in protocols for s in SPLIT_NAMES))
    z=[make('Z24',f'z/{state}/{setup}',int(state>2),setup) for state in range(1,5) for setup in range(1,10)]
    zz=outer_protocols(z,'Z24')
    check_test('Z24_outer_same_setup_not_shared',all(all(len({r.split for r in pp if r.setup==j})==1 for j in range(1,10)) for pp in zz))
    bad=[make('openLAB2025',f'h{i}',0) for i in range(4)]+[make('openLAB2025',f'd{i}',1) for i in range(6)]
    expect_failure('outer_small_dataset_no_window_fallback',lambda:outer_protocols(bad,'openLAB2025'))
    d=pd.DataFrame({'y':[0,1],'p':[.1,.9],'group_id':['h','d'],'fault_type':['nominal','nominal'],
                    'usable_physical_sensors':[2,2],'disagreement':[0.,0.],'u_ensemble':[0.,0.]})
    c=fit_calibrator(d,'FaultAware')
    check_test('lowN_calibration_identity',c.slope==1 and c.intercept==0 and c.structural_threshold==.5 and c.probability_method=='IDENTITY_SMALL_INDEPENDENT_N')
    check_test('no_reference_environment_override',CFG.mode in {'full','audit','smoke','selftest'})
    check_test('GPU_policy_kernel_local',CFG.device_preference in {'auto','cpu'} and str(DEVICE).split(':')[0] in {'cpu','cuda'})
    tt=np.arange(2000)/100.;xx=np.column_stack([np.sin(2*np.pi*4*tt),np.sin(2*np.pi*4*tt+.2)])
    ss,qq,aa,vv=features_window(xx,100.,np.ones(2),['unverified_direction','unverified_direction'])
    check_test('unknown_axis_not_coherence_peer',np.isnan(ss[:,-1]).all() and np.isnan(qq[:,-1]).all())
    check_test('GPU_error_classification',gpu_error(RuntimeError('CUDA out of memory')) and not gpu_error(ValueError('bad source data')))
    check_test('array_hash_sensitive_to_data',array_fingerprint(np.array([1.,2.]))!=array_fingerprint(np.array([1.,3.])))
    check_test('holdout_drift_not_training_family',CFG.heldout_fault=='drift' and 'drift' not in [x for x in CFG.fault_types if x!=CFG.heldout_fault])
    print('Inherited V1.0 regression checks total:',len(SOFTWARE_TESTS),flush=True)


def run_v11_regression_tests():
    def frame(n=4):
        return pd.DataFrame({'acquisition_id':['a']*n,'group_id':['a']*n,'task':['Z24']*n,
            'split':['cal']*n,'y':[0]*n,'state':['reference']*n,'fault_type':['nominal']*n,
            'severity':[0.]*n,'excitation':['avt']*n,'usable_physical_sensors':[3]*n})
    meta=frame(4)
    # Window-level disagreement may cancel within an acquisition; aggregation order matters.
    members=np.array([[.1,.9,.1,.9],[.9,.1,.9,.1]])
    a=aggregate_member_predictions(meta,members,members,np.zeros(4))
    check_test('v11_member_means_before_std',a.u_ensemble.iloc[0]==0 and np.isclose(a.p.iloc[0],.5))
    b=aggregate_member_predictions(meta,np.tile([[.2],[.8]],(1,4)),np.tile([[.2],[.8]],(1,4)),np.zeros(4))
    check_test('v11_sigma030_maps060',np.isclose(uncertainty_components(b,np.array([.1]))['ensemble_dispersion'][0],.6))
    perm=np.array([3,1,0,2]);c=aggregate_member_predictions(meta.iloc[perm].reset_index(drop=True),members[::-1][:,perm],members[::-1][:,perm],np.zeros(4))
    check_test('v11_member_and_row_order_invariant',np.isclose(a.p.iloc[0],c.p.iloc[0]) and np.isclose(a.u_ensemble.iloc[0],c.u_ensemble.iloc[0]))
    dup=aggregate_member_predictions(pd.concat([meta,meta],ignore_index=True),np.tile(members,(1,2)),np.tile(members,(1,2)),np.zeros(8))
    check_test('v11_duplicated_windows_not_new_group',np.isclose(a.u_ensemble.iloc[0],dup.u_ensemble.iloc[0]) and len(dup)==1)
    d=pd.DataFrame({'u_ensemble':[.51],'disagreement':[0.]})
    expect_failure('v11_invalid_probability_std_blocked',lambda:uncertainty_components(d,np.array([.5])))
    sel=coverage_selector(np.array([.2]*4),np.ones(4,bool),np.ones(4),np.array([0,0,1,1]),.8)
    check_test('v11_tied_coverage_reports_overshoot',sel['status']=='TARGET_REACHED_WITH_OVERSHOOT' and sel['coverage_below_target']==0 and not sel['target_exactly_attainable'])
    limited=coverage_selector(np.arange(4)/4,np.array([1,0,0,0],bool),np.ones(4),np.array([0,0,1,1]),.8)
    check_test('v11_hard_gate_reports_limited_coverage',limited['status']=='LIMITED_BY_SENSOR_AVAILABILITY' and np.isclose(limited['max_cal_attainable_coverage'],.25))
    empty=coverage_selector(np.arange(4)/4,np.zeros(4,bool),np.ones(4),np.array([0,0,1,1]),.8)
    check_test('v11_no_sensors_no_acceptance',empty['threshold']==-1 and empty['achieved_cal_coverage']==0)
    q=pd.DataFrame({'group_id':['a','b','a','b'],'acquisition_id':['a','b','a','b'],'fault_type':['nominal','nominal','gain','gain'],
        'injected_fault':[0,0,1,1],'quality_score':[.6,.8,.9,.95],'split':['cal']*4})
    th,qd,_=quality_acquisition_calibration(q)
    check_test('v11_acquisition_budget_respected_on_cal',qd['achieved_nominal_alert_fraction']<=CFG.quality_alarm_budget+1e-12 and th>.8)
    th2,qd2,_=quality_acquisition_calibration(pd.concat([q,q],ignore_index=True))
    check_test('v11_cal_duplicates_do_not_change_threshold',th==th2 and qd2['nominal_calibration_groups']==2)
    qbad=q.copy();qbad['split']='test'
    expect_failure('v11_quality_test_labels_not_calibration',lambda:quality_acquisition_calibration(qbad))
    # Many examples but few reference groups: flexible calibration must still be disabled.
    n=20;cp=pd.DataFrame({'y':[0,0]+[1]*18,'p':[.1,.2]+[.8]*18,'group_id':[f'g{i}' for i in range(n)],
        'fault_type':['nominal']*n,'usable_physical_sensors':[2]*n,'disagreement':[0.]*n,'u_ensemble':[0.]*n,'qmax':[.1]*n,'split':['cal']*n})
    cal=fit_calibrator(cp,'FaultAware')
    check_test('v11_affine_requires_groups_per_class',cal.probability_method=='IDENTITY_SMALL_INDEPENDENT_N' and cal.class_group_counts[0]==2)
    sc=fit_calibrator(cp,'StandardNN')
    check_test('v11_all_models_same_primary_selector',cal.selector==sc.selector=='confidence')
    test=cp.copy();test['split']='test';fingerprint=stable_hash(asdict(cal));dd=decide(test,cal)
    test['y']=1-test.y;changed=decide(test,cal)
    check_test('v11_test_labels_do_not_change_decisions',np.array_equal(dd.accepted,changed.accepted) and np.array_equal(dd.binary_prediction,changed.binary_prediction) and stable_hash(asdict(cal))==fingerprint)
    expect_failure('v11_structural_test_cannot_fit_calibrator',lambda:fit_calibrator(test,'FaultAware'))
    no=test.copy();no['usable_physical_sensors']=1
    check_test('v11_one_sensor_always_abstains',not decide(no,cal).accepted.any())
    ref=pd.DataFrame({'group_id':['d1','d1','d2'],'score':[0.,100.,10.],'usable_physical_sensors':[2,2,2]})
    threshold=reference_record_threshold(ref,.95)
    check_test('v11_reference_calibration_same_record_unit',threshold==100.)
    counts=binary_quality_summary([0,0,1],[0,1,1])
    check_test('v11_quality_prevalence_and_false_positive',np.isclose(counts['positive_prevalence_weighted'],1/3) and counts['false_positive_rate']==.5)
    check_test('v11_source_version_separate',VERSION=='1.2.0' and 'runs_v12' in str(BASE_RUN))
    check_test('v11_default_never_claim_new_independent_test',CFG.test_was_previously_inspected)
    # Additional metadata invariants inherited unchanged from V1.0.
    check_test('v11_no_added_models_required_for_primary',not CFG.enable_minirocket or isinstance(CFG.enable_minirocket,bool))
    print('V1.1 software regression checks total:',len(SOFTWARE_TESTS),flush=True)

def run_v12_regression_tests():
    # Identical ordinal source-score values must not be jittered for apparent coverage or detection.
    check_test('v12_quantile_has_no_interpolated_scores',empirical_upper_quantile([.1,.2,.9],.90)==.9)
    check_test('v12_quantile_replication_invariant',empirical_upper_quantile([.1,.2,.3,.9],.90)==empirical_upper_quantile([.1,.2,.3,.9]*4,.90))
    def calframe(scores):
        return pd.DataFrame({'group_id':['a','b','c','a','b','c'],'acquisition_id':['a','b','c','a','b','c'],
            'fault_type':['nominal']*3+['gain']*3,'injected_fault':[0]*3+[1]*3,'quality_score':scores,'split':['cal']*6,'aggregation':['max']*6})
    q=calframe([.1,.2,1.,.8,.95,1.]);th,diag,curve=quality_acquisition_calibration(q,.05)
    check_test('v12_saturated_cal_no_useful_alarm',th is None and diag['status']=='NO_USEFUL_SENSOR_ALARM_THRESHOLD' and not diag['available'])
    check_test('v12_out_of_range_threshold_diagnostic_only',diag['diagnostic_candidate_threshold']>1 and diag['achieved_injected_detection_fraction']==0)
    check_test('v12_three_cal_groups_resolution',np.isclose(diag['empirical_resolution_one_nominal_alert'],1/3) and 'SMALL_N' in diag['warnings'])
    decisions=attach_instrument_decision(q,th,diag)
    check_test('v12_unknown_not_negative',decisions.sensor_check.isna().all() and (decisions.instrument_decision=='UNDETERMINED').all())
    ms=quality_service_summary(decisions)
    check_test('v12_unknown_stays_in_service_denominators',ms['undetermined_fraction']==1 and ms['service_detection_fraction_all_corrupted']==0 and np.isnan(ms['recall_conditional_available']))
    clear=calframe([.1,.2,.3,.8,.9,.95]);ct,cd,_=quality_acquisition_calibration(clear,.05)
    check_test('v12_useful_alarm_available',ct is not None and cd['status']=='EMPIRICAL_THRESHOLD_AVAILABLE' and cd['achieved_injected_detection_fraction']>0)
    cc=attach_instrument_decision(clear,ct,cd);mm=quality_service_summary(cc)
    check_test('v12_available_alarm_sensitivity',mm['decision_availability_fraction']==1 and mm['recall_conditional_available']==1 and mm['false_positive_rate_conditional_available']==0)
    mixed=pd.concat([decisions,cc.assign(group_id=lambda d:'second_'+d.group_id)],ignore_index=True)
    xm=quality_service_summary(mixed)
    check_test('v12_mixed_available_and_unknown_explicit',xm['decision_availability_fraction']==.5 and xm['recall_conditional_available']==1 and xm['service_detection_fraction_all_corrupted']==.5)
    onlynom=clear[clear.fault_type=='nominal'];nt,nd,_=quality_acquisition_calibration(onlynom)
    check_test('v12_missing_fault_support_no_fictitious_threshold',nt is None and nd['status']=='INSUFFICIENT_CALIBRATION_SUPPORT')
    bad=clear.copy();bad['quality_score']=1.1
    expect_failure('v12_invalid_quality_probability_blocked',lambda:quality_acquisition_calibration(bad))
    expect_failure('v12_available_threshold_cannot_exceed_one',lambda:attach_instrument_decision(clear,1.01,{'available':True}))
    qq=clear.copy();qq['split']='test'
    expect_failure('v12_test_cannot_calibrate_aggregation',lambda:quality_acquisition_calibration(qq))
    ct2,cd2,_=quality_acquisition_calibration(pd.concat([clear,clear],ignore_index=True),.05)
    check_test('v12_nominal_replication_not_independent_calibration',ct==ct2 and cd2['nominal_calibration_groups']==3)
    # Real physical-axis aggregation order: one transient high axis may dominate max but not temporal q90.
    n=20;meta=pd.DataFrame({'acquisition_id':['qa']*n,'group_id':['qa']*n,'task':['Route345']*n,
        'split':['cal']*n,'y':[0]*n,'state':['baseline_1']*n,'fault_type':['nominal']*n,'severity':[0.]*n,'excitation':['vehicle']*n,
        'parent_id':[f'qa_{i}' for i in range(n)],'variant_id':[f'qa_{i}::nominal' for i in range(n)],
        'start_sample':np.arange(n)*100,'stop_sample':(np.arange(n)+1)*100,'fs':[100.]*n,'seconds':[1.]*n})
    score=np.full((n,3),.1);score[:,1]=.2;score[0,0]=1.
    class FixtureQuality:
        def predict(self,st):return score
    z=np.zeros((n,3,2),np.float32);st=Store(meta,z,z,z,np.ones((n,3)),np.ones((n,3)),np.zeros((n,3)),physical=['P1','P1','P2'])
    e,phy,_,_=quality_event_frames(st,FixtureQuality(),'max');alt,ap,_,_=quality_event_frames(st,FixtureQuality(),'physical_q90')
    check_test('v12_axes_grouped_as_physical_instruments',e.n_physical_sensors.iloc[0]==2 and len(phy)==2)
    check_test('v12_q90_is_temporal_after_axis_max',e.quality_score.iloc[0]==1 and np.isclose(alt.quality_score.iloc[0],.2))
    check_test('v12_aggregation_preserves_event_identity',set(e.event_id)==set(alt.event_id))
    check_test('v12_window_count_reported',alt.minimum_windows_per_sensor.iloc[0]==20)
    cp=pd.DataFrame({'y':[0,1],'p':[.1,.9],'group_id':['h','d'],'fault_type':['nominal','nominal'],
        'usable_physical_sensors':[2,2],'disagreement':[0.,0.],'u_ensemble':[0.,0.],'qmax':[.9,.9],'split':['cal','cal']})
    c=fit_calibrator(cp,'FaultAware',None,quality_diagnostic=diag);out=decide(cp.assign(split='test'),c)
    check_test('v12_unknown_instrument_does_not_disable_structure',out.accepted.all() and out.sensor_check.isna().all())
    change=decide(cp.assign(split='test',y=[1,0]),c)
    check_test('v12_test_labels_not_used_by_nullable_decision',np.array_equal(out.binary_prediction,change.binary_prediction) and out.sensor_check.equals(change.sensor_check))
    m=metric_dict(out)
    check_test('v12_metrics_explicit_missing_instrument_service',m['instrument_decision_availability']==0 and m['instrument_undetermined_fraction']==1 and np.isnan(m['sensor_check_fraction']))
    check_test('v12_reference_and_output_isolation',VERSION=='1.2.0' and 'runs_v12' in str(BASE_RUN) and BASE_RUN!=REFERENCE_RUN)
    check_test('v12_drift_feature_already_present','normalized_drift' in QBASE_NAMES)
    check_test('v12_default_keeps_reviewed_test_warning',CFG.test_was_previously_inspected)
    print('V1.2 software regression checks total:',len(SOFTWARE_TESTS),flush=True)


## 24 — Exécution : une seule cellule d'orchestration

`full` : contrôles, acquisitions complètes, neuf plis, analyses de qualité/transfert/référence et résultats. `audit` : métadonnées seulement. `smoke` : budget réduit. `selftest` : archives artificielles explicitement étiquetées.

En cas d'arrêt : diagnostics et checkpoints restent dans le dossier de cette V1.1. Reprendre avec le même nom et la même configuration. Une incohérence n'est jamais remplacée par des données synthétiques ou par un retrait silencieux d'acquisition.

In [25]:
def export_manifest(records,name):
    rows=[]
    for r in records:
        z=asdict(r)
        for k in ('channels','sensor_ids','axes','units','metadata'):z[k]=json.dumps(json_safe(z[k]),ensure_ascii=False)
        rows.append(z)
    return table(rows,name)

def discover_all():
    found=[]
    found.extend(discover_z24(CFG.z24_root));found.extend(discover_rt(CFG.route345_file))
    found.extend(discover_openlab(CFG.openlab_2025_root,'openLAB2025'))
    if CFG.run_reference_2024:found.extend(discover_openlab(CFG.openlab_2024_root,'openLAB2024'))
    out=[];seen={}
    for r in found:
        key=(r.task,r.sha256,r.native_key)
        if key in seen:
            previous=seen[key]
            if (previous.state,previous.y,previous.eligible)!=(r.state,r.y,r.eligible):
                raise ValueError('Identical raw object has contradictory state metadata: '+r.path)
            note('duplicate_copy_excluded',r.path,previous.path,task=r.task);continue
        seen[key]=r;out.append(r)
    table(RAW_INVENTORY,'audit/RAW_FILE_HASHES.csv');return out

def header_gates(records):
    all_folds={};summary=[]
    for task in CFG.required_tasks:
        native=[r for r in records if r.task==task];eligible=[r for r in native if r.eligible and r.y in (0,1)]
        errors=[d for d in DIAGNOSTICS if d.get('task')==task and d['kind']=='loader_error']
        gate(task+'_response_records_found',len(native)>0,{'native_records':len(native),'primary_eligible':len(eligible)})
        gate(task+'_native_loader_errors',len(errors)==0,{'error_count':len(errors),'details_file':'DIAGNOSTICS.csv'})
        gate(task+'_two_structural_classes',{r.y for r in eligible}=={0,1},dict(Counter(r.y for r in eligible)))
        try:
            folds=outer_protocols(eligible,task)
            for i,split in enumerate(folds):
                panel,physical,axes=panel_from_train(split)
                gate(task+f'_fold{i+1}_four_disjoint_parts',True,dict(Counter(r.split for r in split)))
                gate(task+f'_fold{i+1}_verified_response_panel',len(set(physical))>=CFG.min_physical_sensors,
                    {'channels':panel,'physical_sensor_count':len(set(physical)),'fs_values':sorted(set(round(r.fs,6) for r in eligible))})
                export_manifest(split,f'audit/{task}_FOLD{i+1}_HEADER_SPLIT.csv')
                table([{'channel':c,'physical_sensor':p,'axis':a} for c,p,a in zip(panel,physical,axes)],f'audit/{task}_FOLD{i+1}_PANEL.csv')
                for s in SPLIT_NAMES:
                    n=Counter(r.y for r in split if r.split==s)
                    summary.append({'task':task,'outer_fold':i+1,'partition':s,'reference_acquisitions':n[0],'altered_acquisitions':n[1]})
                    if s!='train' and min(n.values())<5:
                        note('small_independent_sample_count',task,f'fold {i+1}/{s}: {dict(n)}; no distributional performance guarantee',task=task)
            all_folds[task]=folds
        except Exception as e:gate(task+'_outer_folds',False,str(e));note('split_blocker',task,e,task=task)
    table(summary,'audit/ALL_FOLD_CLASS_COUNTS.csv')
    return all_folds

def fold_signature(records,task,fold):
    return stable_hash({'study':SIGNATURE,'task':task,'fold':fold,'records':[(r.sha256,r.native_key,r.acquisition_id,r.y,r.split,r.channels,r.waveform_hash) for r in records]})

def verified_fold_checkpoint(folder,signature):
    manifest=folder/'FOLD_COMPLETE.json';blob=folder/'FOLD_RESULTS.joblib'
    if not CFG.resume or not manifest.exists() or not blob.exists():return None
    m=json.loads(manifest.read_text())
    if m.get('signature')!=signature:raise ValueError('Data/configuration differ from completed fold; use a new study_name')
    for rel,h in m['output_sha256'].items():
        p=folder/rel
        if not p.is_file() or file_hash(p)!=h:raise ValueError('Completed fold output changed/corrupt: '+str(p))
    result=joblib.load(blob)
    print('  Pli terminé réutilisé après vérification :',folder,flush=True)
    return result

def execute_fold(task,fold_index,records):
    global METRICS,PAIRWISE
    folder=BASE_RUN/'folds'/task/f'fold_{fold_index+1:02d}'
    signature=fold_signature(records,task,fold_index)
    reused=verified_fold_checkpoint(folder,signature)
    if reused is not None:
        PREDICTOR_ORIGINS[str(folder.relative_to(BASE_RUN))]=reused.get('predictor_origin',{'mode':'REUSED_V1_2_FOLD'})
        return reused
    with output_context(folder):
        METRICS=[];PAIRWISE=[]
        export_manifest(records,'audit/LOCKED_SPLIT.csv')
        started=time.perf_counter();run_task(records)
        result={'predictor_origin':PREDICTOR_ORIGINS.get(str(RUN.relative_to(BASE_RUN)),{}),'decisions':ALL_DECISIONS[task],'payload':TRANSFER_PAYLOADS[task],'status':EXPERIMENT_STATUS[task],
            'metrics':METRICS.copy(),'pairwise':PAIRWISE.copy(),'elapsed_seconds':time.perf_counter()-started,
            'signature':signature,'fold_index':fold_index}
        atomic_joblib(result,RUN/'FOLD_RESULTS.joblib')
        hashes={}
        for p in RUN.rglob('*'):
            if p.is_file() and p.name not in {'FOLD_COMPLETE.json'} and not p.name.endswith('.tmp'):
                hashes[str(p.relative_to(RUN))]=file_hash(p)
        write_json({'status':'COMPLETE','signature':signature,'output_sha256':hashes,'elapsed_seconds':result['elapsed_seconds']},RUN/'FOLD_COMPLETE.json')
        return result

def main():
    global CFG,RECORDS,METRICS,PAIRWISE
    try:
        print('\n[1/6] Contrôles logiciels intégrés',flush=True)
        run_regression_tests();run_v10_regression_tests();run_v11_regression_tests();run_v12_regression_tests()
        if CFG.mode=='selftest':
            manifest=BASE_RUN/'audit/SYNTHETIC_PATHS.json'
            if manifest.exists():fixture_paths=json.loads(manifest.read_text())
            else:fixture_paths=make_synthetic_archives();write_json(fixture_paths,manifest)
            CFG=replace(CFG,**fixture_paths)
            write_json({'data_origin':'SYNTHETIC_SOFTWARE_FIXTURES','paths':fixture_paths},BASE_RUN/'audit/FIXTURE_PROVENANCE.json')
        print('\n[2/6] Lecture des métadonnées / empreintes / capteurs',flush=True)
        RECORDS=discover_all();export_manifest(RECORDS,'audit/NATIVE_RECORDS.csv')
        table([{'task':t,'total_recognized':sum(r.task==t for r in RECORDS),'primary_eligible':sum(r.task==t and r.eligible for r in RECORDS)}
            for t in (*CFG.required_tasks,'openLAB2024')],'audit/DATASET_SUMMARY.csv')
        folds=header_gates(RECORDS)
        gate('full_integrity_hashes_enabled',CFG.full_file_hashes,'Full SHA256 input provenance enabled')
        blocked=any(g['mandatory'] and not g['passed'] for g in GATES)
        flush_reports()
        if CFG.mode=='audit':
            write_evidence('AUDIT_BLOCKED' if blocked else 'AUDIT_HEADERS_READY');article_package();return
        if blocked:raise RuntimeError('Mandatory source/label/role/split validation failed. Diagnostic archive is preserved; no training is started.')
        require_compute()
        print('\n[3/6] Vérification des acquisitions complètes, horloges et doublons',flush=True)
        for task in CFG.required_tasks:
            native={r.acquisition_id:r for r in folds[task][0]}
            for i,r in enumerate(native.values()):
                cache_record(r)
                if i%CFG.progress_every==0 or i==len(native)-1:print(f'  {task}: {i+1}/{len(native)} acquisitions complètes vérifiées',flush=True)
            for fold in folds[task]:
                for r in fold:
                    rr=native[r.acquisition_id];r.cache_path=rr.cache_path;r.n_samples=rr.n_samples;r.waveform_hash=rr.waveform_hash
                audit_split(fold)
            gate(task+'_full_clock_and_waveform_integrity',True,{'complete_records':len(native),'test_occurrences_per_acquisition':1})
            table([{'record':r.acquisition_id,'source_native':r.metadata.get('source_preprocessing','source-native'),
                'ours':'raw quality first; short-gap repair only; Hann/Welch demeaning; no automatic second bandpass; train-only references',
                'fs':r.fs,'modality':r.modality,'samples':r.n_samples} for r in native.values()],f'audit/{task}_PREPROCESSING_LEDGER.csv')
        print('\n[4/6] Apprentissage, calibration et tests externes (3 ponts × 3 plis)',flush=True)
        for task in CFG.required_tasks:
            ALL_FOLD_DECISIONS[task]=[];FOLD_PAYLOADS[task]=[];FOLD_SUMMARIES[task]=[]
            for fold,records in enumerate(folds[task]):
                print(f'\n========== {task} — PLI EXTERNE {fold+1}/{CFG.outer_folds} — {DEVICE} ==========',flush=True)
                result=execute_fold(task,fold,records)
                ALL_FOLD_DECISIONS[task].append(result['decisions']);FOLD_PAYLOADS[task].append(result['payload'])
                FOLD_SUMMARIES[task].append({**result['status'],'outer_fold':fold+1,'elapsed_seconds':result['elapsed_seconds']})
                FOLD_METRICS.extend({**v,'outer_fold':fold+1} for v in result['metrics'])
                FOLD_PAIRWISE.extend({**v,'outer_fold':fold+1} for v in result['pairwise'])
                EXPERIMENT_STATUS[task]={'status':'IN_PROGRESS','completed_outer_folds':fold+1}
                flush_reports();write_evidence('RUNNING')
            unique_expected={r.acquisition_id for r in folds[task][0]}
            nominal=pd.concat([x['FaultAware'][float(CFG.primary_coverage)].query("fault_type=='nominal'") for x in ALL_FOLD_DECISIONS[task]])
            gate(task+'_OOF_EACH_ACQUISITION_ONCE',set(nominal.acquisition_id)==unique_expected and not nominal.acquisition_id.duplicated().any(),
                 {'expected':len(unique_expected),'observed':len(nominal)})
            EXPERIMENT_STATUS[task]={'status':'COMPLETE','completed_outer_folds':len(folds[task]),'oof_acquisitions':len(unique_expected),
                'all_folds_calibration_frozen':all(v['calibration_immutable'] for v in FOLD_SUMMARIES[task])}
        print('\n[5/6] Transfert de qualité et référence 2024',flush=True)
        try:run_quality_transfer()
        except Exception as e:
            EXPERIMENT_STATUS['quality_transfer']={'status':'BLOCKED','error':repr(e)};note('transfer_failure','quality_transfer',e,trace=traceback.format_exc())
        if CFG.run_reference_2024:
            try:
                reference=[r for r in RECORDS if r.task=='openLAB2024' and r.eligible]
                for i,r in enumerate(reference):
                    cache_record(r)
                    if i%50==0 or i==len(reference)-1:print(f'  openLAB2024 : {i+1}/{len(reference)}',flush=True)
                run_reference_only(reference)
                if not isinstance(EXPERIMENT_STATUS.get('openLAB2024'),dict):
                    EXPERIMENT_STATUS['openLAB2024']={'status':str(EXPERIMENT_STATUS.get('openLAB2024','NOT_RUN'))}
            except Exception as e:
                EXPERIMENT_STATUS['openLAB2024']={'status':'BLOCKED_REFERENCE_ONLY','error':repr(e)};note('reference_failure','openLAB2024',e,trace=traceback.format_exc())
        else:EXPERIMENT_STATUS['openLAB2024']={'status':'DISABLED_BY_CONFIG'}
        print('\n[6/6] Résultats hors échantillon, statistiques et figures article',flush=True)
        EXPERIMENT_STATUS['quality_crossfit']={'status':'EXECUTED_TRAIN_ONLY' if CFG.variant=='quality_crossfit' else 'DISABLED_BY_VARIANT','calibration_or_test_not_used':True}
        EXPERIMENT_STATUS['state_blocked_generalization']={'status':'NOT_IN_PRIMARY_PROTOCOL','no_claim':True}
        EXPERIMENT_STATUS['cross_excitation_generalization']={'status':'NOT_IN_PRIMARY_PROTOCOL','no_claim':True}
        finalize_plots_tables();flush_reports()
        if any(g['mandatory'] and not g['passed'] for g in GATES):raise RuntimeError('A mandatory post-computation integrity gate failed')
        ev=write_evidence('COMPLETE');article_package()
        if CFG.mode=='selftest':
            check_test('all_three_tasks_all_outer_folds',all(EXPERIMENT_STATUS[t]['completed_outer_folds']==CFG.outer_folds for t in CFG.required_tasks))
            check_test('synthetic_never_paper_ready',not ev['paper_results_available'])
            check_test('three_quality_targets_completed',EXPERIMENT_STATUS.get('quality_transfer',{}).get('status')=='COMPLETE')
            check_test('reference_2024_executed',EXPERIMENT_STATUS.get('openLAB2024',{}).get('status')=='COMPLETE_REFERENCE_ONLY')
            ev=write_evidence('COMPLETE');article_package()
        print('\nSTATUT :',ev['status'],'\nRésultats principaux pour article :',ev['paper_results_available'],
              '\nToutes les expériences demandées :',ev['all_requested_experiments_complete'],'\nDossier :',BASE_RUN,flush=True)
    except BaseException as e:
        note('fatal_exception','pipeline',e,trace=traceback.format_exc());flush_reports()
        status='INTERRUPTED' if isinstance(e,KeyboardInterrupt) else 'BLOCKED'
        write_evidence(status,str(e))
        print('\n'+status+' : diagnostics et checkpoints conservés dans',BASE_RUN,flush=True)
        raise

main()


[1/6] Contrôles logiciels intégrés
Software regression tests (source contracts): 38 passed
Inherited V1.0 regression checks total: 51
V1.1 software regression checks total: 72
V1.2 software regression checks total: 97

[2/6] Lecture des métadonnées / empreintes / capteurs
[PASS] Route345_release_V2: {'md5': '5c3e2aff20e1984fce15bd18f57f62eb', 'expected': '5c3e2aff20e1984fce15bd18f57f62eb', 'synthetic_fixture_not_original_release': False}
[PASS] Z24_response_records_found: {'native_records': 153, 'primary_eligible': 144}
[PASS] Z24_native_loader_errors: {'error_count': 0, 'details_file': 'DIAGNOSTICS.csv'}
[PASS] Z24_two_structural_classes: {0: 18, 1: 126}
[PASS] Z24_fold1_four_disjoint_parts: {'train': 64, 'test': 48, 'dev': 16, 'cal': 16}
[PASS] Z24_fold1_verified_response_panel: {'channels': ['R1V', 'R2L', 'R2T', 'R2V', 'R3V'], 'physical_sensor_count': 3, 'fs_values': [100.0]}
[PASS] Z24_fold2_four_disjoint_parts: {'train': 64, 'test': 48, 'cal': 16, 'dev': 16}
[PASS] Z24_fold2_veri

,task,model,subset,n_acquisitions,coverage,coverage_reference,coverage_altered,selective_risk,false_structural_alarm_rate,false_no_alert_rate,auroc_all
20,Z24,FaultAware,nominal,144,0.784722,0.222222,0.865079,0.000000,0.000000,0.0,0.987213
21,Z24,FaultAware,seen_faults,144,0.769907,0.296296,0.837566,0.001804,0.011111,0.0,0.978019
22,Z24,FaultAware,unseen_fault,144,0.770833,0.277778,0.841270,0.006006,0.037037,0.0,0.973839
23,Z24,FaultAware,all_faults,144,0.770062,0.293210,0.838183,0.002505,0.015432,0.0,0.977159
164,Z24,StandardNN,nominal,144,0.791667,0.277778,0.865079,0.000000,0.000000,0.0,0.984568
165,Z24,StandardNN,seen_faults,144,0.768519,0.329630,0.831217,0.001205,0.007407,0.0,0.975269
166,Z24,StandardNN,unseen_fault,144,0.763889,0.425926,0.812169,0.000000,0.000000,0.0,0.976582
167,Z24,StandardNN,all_faults,144,0.767747,0.345679,0.828042,0.001005,0.006173,0.0,0.975198
212,Route345,FaultAware,nominal,25,0.640000,0.666667,0.631579,0.125000,0.333333,0.0,0.771930
213,Route345,FaultAware,seen_faults,25,0.650667,0.666667,0.645614,0.122951,0.333333,0.0,0.771618



Paquet article : /home/a.riyahi/ICARC2027_MultiBridge_FaultAware_SHM/runs_v12/V1_2_MAIN/ARTICLE_RESULTS.zip
EVIDENCE_SUMMARY.json : /home/a.riyahi/ICARC2027_MultiBridge_FaultAware_SHM/runs_v12/V1_2_MAIN/EVIDENCE_SUMMARY.json
REPORT_ARTICLE.html : /home/a.riyahi/ICARC2027_MultiBridge_FaultAware_SHM/runs_v12/V1_2_MAIN/REPORT_ARTICLE.html
ARTICLE_RESULTS.zip : /home/a.riyahi/ICARC2027_MultiBridge_FaultAware_SHM/runs_v12/V1_2_MAIN/ARTICLE_RESULTS.zip
AUDIT_PACKAGE.zip : /home/a.riyahi/ICARC2027_MultiBridge_FaultAware_SHM/runs_v12/V1_2_MAIN/AUDIT_PACKAGE.zip

STATUT : COMPLETE_WITH_LIMITATIONS 
Résultats principaux pour article : True 
Toutes les expériences demandées : True 
Dossier : /home/a.riyahi/ICARC2027_MultiBridge_FaultAware_SHM/runs_v12/V1_2_MAIN


## Portée scientifique de V1.2
- Pas de nouvelles acquisitions physiques ni de nouveaux états indépendants ; les plis et fenêtres ne multiplient pas les expériences.
- `quality_crossfit` change les scores qualité présentés au réseau structurel à l’entraînement, pas automatiquement le détecteur qualité final.
- Le budget de 5 % est **empirique** ; avec 16, 3 ou 2 acquisitions de calibration, une seule alerte représente respectivement 6,25 %, 33,33 % ou 50 %. Aucun taux populationnel n’est garanti.
- `NO_USEFUL_SENSOR_ALARM_THRESHOLD` signifie une décision instrumentale indisponible, non une absence de défaut. Les fractions non déterminées figurent dans tous les résumés.
- Le quantile temporel 0,90 peut manquer des défauts courts et peut être identique au maximum lorsque peu de fenêtres existent. Les deux méthodes sont évaluées, sans sélection à partir du test.
- Les différences nominal/corrompu ne sont pas des caractéristiques utilisables en exploitation : la copie propre n’y est pas disponible.
- Les résultats V1.2 sont une analyse révisée après examen des versions antérieures. Aucune supériorité, sécurité d’ouvrage, RUL ou acceptation de conférence n’est promise.


## Lire les résultats
Dossier : `project_root/runs_v12/V1_2_MAIN/`.

Ouvrir `REPORT_ARTICLE.html`, puis examiner `EVIDENCE_SUMMARY.json`, `tables/ARTICLE_MAIN_RESULTS.csv`, `tables/CROSSFIT_GROUP_AUDIT.csv`, `tables/QUALITY_CALIBRATION_FEASIBILITY.csv`, `tables/ROUTE345_AGGREGATION_COMPARISON.csv` et `tables/V1_1_V1_2_COMPARISON.csv` si la référence est présente.

Partager `ARTICLE_RESULTS.zip` et `AUDIT_PACKAGE.zip` pour l’analyse. Les archives de résultats n’incluent ni signaux bruts ni poids de modèles. `COMPLETE_WITH_LIMITATIONS` peut signaler une exécution achevée avec un avertisseur non déterminé : lire les statuts par pli avant toute conclusion.
